# ECO waveforms: literature-checked axial models and controlled comparison

Self-contained Kaggle notebook. Run All on CPU; no downloaded data are needed.
Required packages are NumPy, SciPy, Matplotlib, Pandas, h5py and mpmath. Their
versions are printed below. If a package is absent, install it explicitly
(for example `%pip install mpmath`) before Run All; the notebook does not
silently install packages or fetch datasets.
The numerical audit fails if a declared tolerance is exceeded. This release
corrects the OWA global/local phase, computes true complex wormhole poles,
checks the gravastar against its published analytic interior, and compares
models at common source/observer areal radii. All corrections3 source-characteristic
tests remain active. Numerical tolerances and model scope are explicit; a
converged waveform does not establish that it contains a clean echo train.


## 1. Physical models, evidence, and non-equivalence

| Class | Model | Implemented response | Independent benchmark | Limitation |
|---|---|---|---|---|
| Black-hole reference | Schwarzschild | axial Regge–Wheeler evolution | Leaver continued fraction for $(2,2,n)$, $n=0,1,2$ | nonrotating, axial sector |
| Horizonless geometric ECO | thin-shell gravastar | de Sitter interior + Schwarzschild exterior, axial Israel matching | complex-frequency shooting + Pani analytic interior and metric junctions (45 cases) | polar shell dynamics omitted |
| Horizonless geometric ECO | mini boson star | EKG equilibrium + axial metric perturbation | Macedo et al. Tables I and IV: equilibrium + two complex axial modes; explicit tail systematic | **fixed-background axial response**, not the coupled polar scalar–metric waveform |
| Horizonless geometric ECO | Schwarzschild thin-shell wormhole | two Schwarzschild exteriors + axial throat jump | throat flux/jump, complex outgoing poles + Yang et al. Table 6; branch continuation | frozen background shell |
| Effective boundary condition | OWA quantum horizon | complex Boltzmann boundary reflectivity behind the RW barrier | global/local phase identity, independent time evolution, plane invariance and complex poles | not a spacetime solution in the same sense as the three geometric ECOs |

The comparison is therefore between **different mechanisms that modify the black-hole late-time response**, not five equivalent compact-object solutions.

Primary references used by the implementation: [Leaver 1985](https://doi.org/10.1098/rspa.1985.0119), [Pani et al. 2009](https://arxiv.org/abs/0909.0287), [Macedo et al. 2013](https://arxiv.org/abs/1307.4812), [Mark et al. 2017](https://arxiv.org/abs/1706.06155), [Damour & Solodukhin 2007](https://doi.org/10.1103/PhysRevD.76.024016), [Yang, Xia & Miao 2025](https://doi.org/10.1140/epjc/s10052-025-14446-w), [Zimmerman et al. 2023](https://arxiv.org/abs/2306.11166), [Qian et al. 2024](https://doi.org/10.1016/j.physletb.2024.138874), and [Oshita, Wang & Afshordi 2019](https://arxiv.org/abs/1905.00464).


## 2. Architecture and fidelity labels

The single-file Kaggle notebook is split conceptually into the requested modules:

1. `01_parameters`: immutable numerical/physical configuration;
2. `02_backgrounds`: model-specific potential-grid builders;
3. `03_wave_evolution`: one common fourth-order MOL/RK4 solver;
4. `04_analysis`: QNMs, phases, fluxes, overlaps, uncertainties, scans, and validation.

Every direct model exposes the same interface: `model.potential_grid(...)`, `model.boundary_conditions()`, and `model.metadata()`. The quantum horizon instead implements the explicitly labelled frequency-domain boundary interface.

- **[EXACT / ANALYTIC]** equation copied from or algebraically equivalent to a cited linear model;
- **[NUMERICAL]** discretized result with an internal error diagnostic;
- **[LITERATURE MODEL]** named paper prescription;
- **[APPROXIMATION]** controlled reduction stated where used;
- **[ASSUMPTION]** choice not fixed by the field equations;
- **[OPEN PROBLEM]** omitted physics that requires a larger system.

Kerr, polar gravastar shell dynamics, coupled polar boson-star perturbations, and nonlinear mergers remain outside scope and are never relabelled as completed calculations.


In [ ]:
# Environment audit: no network access is required.
import sys, platform, importlib
from pathlib import Path

print("Python:", sys.version)
print("Platform:", platform.platform())
try:
    import torch
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
except Exception as exc:
    print("PyTorch/GPU audit unavailable:", exc)

for pkg in ["numpy", "scipy", "matplotlib", "pandas", "h5py", "sympy", "mpmath"]:
    try:
        mod = importlib.import_module(pkg)
        print(f"{pkg:12s} {getattr(mod, '__version__', 'unknown')}")
    except Exception as exc:
        print(f"MISSING {pkg}: {exc}")

In [ ]:
from __future__ import annotations

import json, math, logging, warnings, time
from dataclasses import dataclass, asdict, replace
from pathlib import Path
from typing import Callable, Any, Protocol

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import h5py
from scipy.integrate import solve_bvp, solve_ivp, cumulative_trapezoid
from scipy.interpolate import PchipInterpolator, CubicSpline
from scipy.optimize import root, least_squares, brentq
from scipy.signal import hilbert, find_peaks, correlate, spectrogram, get_window
from scipy.special import lambertw, loggamma

if not hasattr(np,"trapezoid"):  # NumPy < 2.0 compatibility (Kaggle currently supports either API)
    np.trapezoid=np.trapz

logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")
LOG = logging.getLogger("eco-waveforms")

ARTIFACT_DIR = Path("/kaggle/working/eco_outputs_final") if Path("/kaggle/working").exists() else Path("eco_outputs_final")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)


@dataclass(frozen=True)
class Config:
    M: float = 1.0
    ell: int = 2
    dx: float = 0.10
    high_resolution_dx: float = 0.05
    cfl: float = 0.35
    t_max: float = 520.0
    sample_every: int = 2
    pulse_center: float = 55.0
    pulse_width: float = 4.0
    extraction_x: float = 95.0
    gravastar_compactness: float = 0.47
    wormhole_r0: float = 2.0005
    boson_sigma_c: float = 0.270963318550685  # sqrt(8*pi*G) phi
    detector_total_mass_solar: float = 60.0
    qh_surface_x: float = -40.0  # computational interface, not physical reflection depth
    seed: int = 20260812


CFG = Config()
np.random.seed(CFG.seed)
plt.rcParams.update({"figure.dpi":120,"savefig.dpi":220,"axes.grid":True,"grid.alpha":0.25})
print(CFG)
print("Artifacts:", ARTIFACT_DIR.resolve())


## 3. Common evolution: stability, characteristics, and extraction

We use $G=c=1$, signature $(-,+,+,+)$, and the $e^{-i\omega t}$ convention. Direct models solve

$$\partial_t^2\Psi-\partial_x^2\Psi+V(x)\Psi=0$$

with fourth-order centered spatial differences and RK4. The asymptotic boundaries use outgoing characteristics. At a regular stellar center the code enforces the local parity implied by $\Psi\propto r^{\ell+1}$ in the first fourth-order stencil, rather than using a second-order one-sided point next to a bare Dirichlet boundary.

For $\Pi=\partial_t\Psi$, define $u_\pm=\Pi\pm\partial_x\Psi$. Since increasing $x$ is outward, $u_+$ is the left-moving (ingoing) field and $u_-$ the right-moving (outgoing) field in this notebook. The sign label is stated because interchanging the outward coordinate interchanges the names. The Gaussian choice $\Pi=\Psi_x$ should therefore give $F_{\rm in}\simeq1$.

Every run stores $\Delta x,\Delta t$, CFL, $N_x,N_t$, domain/extraction coordinates, the two characteristic energy fractions, $A_{\max}/A_0$, and early/middle/late log-growth rates. CFL is exactly the requested $0.35$ unless an explicit time-convergence run overrides it. Stability uses an AND criterion: the amplitude ratio must be acceptable and neither the middle nor late segment may show sustained positive exponential growth.


In [ ]:
@dataclass
class Waveform:
    model: str
    t: np.ndarray
    psi: np.ndarray
    x_extract: float
    metadata: dict[str, Any]

    @property
    def analytic(self) -> np.ndarray:
        return hilbert(self.psi)

    @property
    def amplitude(self) -> np.ndarray:
        return np.abs(self.analytic)

    @property
    def phase(self) -> np.ndarray:
        return np.unwrap(np.angle(self.analytic))

    @property
    def instantaneous_frequency(self) -> np.ndarray:
        return np.gradient(self.phase, self.t)/(2*np.pi)

    def masked_instantaneous_frequency(self, epsilon: float=1e-3) -> np.ndarray:
        """Instantaneous frequency only where the analytic amplitude is reliable."""
        out=self.instantaneous_frequency.copy(); amp=self.amplitude
        ok=amp>epsilon*np.max(amp)
        keep=ok.copy(); keep[1:-1] &= ok[:-2]&ok[2:]; keep[[0,-1]]=False
        out[~keep]=np.nan
        return out


class PotentialModel(Protocol):
    name: str
    def potential_grid(self, dx: float, **kwargs): ...
    def boundary_conditions(self) -> dict[str, str]: ...
    def metadata(self) -> dict[str, Any]: ...


@dataclass(frozen=True)
class GridModelAdapter:
    name: str
    builder: Callable[..., tuple]
    left_bc: str
    static_metadata: dict[str, Any]

    def potential_grid(self, dx: float, **kwargs):
        return self.builder(dx=dx,**kwargs)

    def boundary_conditions(self) -> dict[str,str]:
        return {"left":self.left_bc,"right":"outgoing"}

    def metadata(self) -> dict[str,Any]:
        return dict(self.static_metadata)


def schwarzschild_rstar(r: np.ndarray | float, M: float=1.0) -> np.ndarray:
    r=np.asarray(r,dtype=float)
    return r+2*M*np.log(r/(2*M)-1)


def r_from_schwarzschild_rstar(x: np.ndarray | float, M: float=1.0) -> np.ndarray:
    x=np.asarray(x,dtype=float)
    z=np.exp(np.clip(x/(2*M)-1,-700,700))
    return 2*M*(1+np.real(lambertw(z)))


def rw_potential(r: np.ndarray, ell: int=2, M: float=1.0) -> np.ndarray:
    f=1-2*M/r
    return f*(ell*(ell+1)/r**2-6*M/r**3)


_D1_W0=np.array([-137/60,5,-5,10/3,-5/4,1/5]); _D1_W1=np.array([-1/5,-13/12,2,-1,1/3,-1/20])
_D2_W0=np.array([15/4,-77/6,107/6,-13,61/12,-5/6]); _D2_W1=np.array([5/6,-5/4,-1/3,7/6,-1/2,1/12])


def d1_4(y: np.ndarray, dx: float) -> np.ndarray:
    out=np.empty_like(y)
    out[2:-2]=(y[:-4]-8*y[1:-3]+8*y[3:-1]-y[4:])/(12*dx)
    out[0]=_D1_W0@y[:6]/dx; out[1]=_D1_W1@y[:6]/dx
    out[-1]=-(_D1_W0@y[-1:-7:-1])/dx; out[-2]=-(_D1_W1@y[-1:-7:-1])/dx
    return out


def d2_4(y: np.ndarray, dx: float) -> np.ndarray:
    out=np.empty_like(y)
    out[2:-2]=(-y[4:]+16*y[3:-1]-30*y[2:-2]+16*y[1:-3]-y[:-4])/(12*dx**2)
    out[0]=_D2_W0@y[:6]/dx**2; out[1]=_D2_W1@y[:6]/dx**2
    out[-1]=_D2_W0@y[-1:-7:-1]/dx**2; out[-2]=_D2_W1@y[-1:-7:-1]/dx**2
    return out


def d2_4_regular_center(y: np.ndarray, dx: float, ell: int=2) -> np.ndarray:
    """Fourth-order origin stencil using Psi(-rho)=(-1)^(ell+1) Psi(rho)."""
    out=d2_4(y,dx); parity=(-1)**(ell+1); ghost_m1=parity*y[1]
    out[0]=0.0
    out[1]=(-y[3]+16*y[2]-30*y[1]+16*y[0]-ghost_m1)/(12*dx**2)
    return out


def initial_characteristics(x, pulse_center, pulse_width, direction="left", carrier_omega=0.0):
    envelope=np.exp(-0.5*((x-pulse_center)/pulse_width)**2)
    psi=envelope*np.cos(carrier_omega*(x-pulse_center)) if carrier_omega else envelope
    dx=float(np.mean(np.diff(x))); psix=d1_4(psi,dx)
    if direction=="left": pi=psix.copy()
    elif direction=="right": pi=-psix.copy()
    else: raise ValueError("direction must be 'left' or 'right'")
    u_plus=pi+psix; u_minus=pi-psix
    e_plus=np.trapezoid(abs(u_plus)**2,x); e_minus=np.trapezoid(abs(u_minus)**2,x)
    den=max(e_plus+e_minus,1e-300)
    Fin=float(e_plus/den); Fout=float(e_minus/den)
    return psi,pi,{"F_in":Fin,"F_out":Fout,"F_ingoing_left_u_plus":Fin,"F_outgoing_right_u_minus":Fout,
                   "initial_direction":direction,"initial_carrier_omega":float(carrier_omega),
                   "characteristic_sign_convention":"u_plus=Pi+Psi_x is left-moving/ingoing; u_minus=Pi-Psi_x is right-moving/outgoing"}


def evolve_wave(x, V, *, t_max, cfl, pulse_center, pulse_width, extraction_points,
                left_bc="outgoing", sample_every=2, stability_growth_limit=10.0,
                sustained_growth_tolerance=1e-3, ell=2,sponge_width=0.0,sponge_strength=0.0,
                interface_jump=None,initial_direction="left",initial_carrier_omega=0.0,derivative_points=()):
    x=np.asarray(x,float); V=np.asarray(V,float)
    if x.ndim!=1 or V.shape!=x.shape: raise ValueError("x and V must be equal 1-D arrays")
    dx=float(np.mean(np.diff(x)))
    if np.max(abs(np.diff(x)-dx))>1e-9*max(1,abs(dx)): raise ValueError("uniform x grid required")
    dt=float(cfl*dx); n_steps=int(np.ceil(t_max/dt)); actual_tmax=n_steps*dt
    if not (0<cfl<0.55): raise ValueError("CFL outside the stable range")
    if abs(cfl-0.35)<1e-12: assert abs(dt/dx-0.35)<1e-12

    psi,pi,char=initial_characteristics(x,pulse_center,pulse_width,initial_direction,initial_carrier_omega)
    if left_bc=="regular_center": psi[0]=pi[0]=0.0

    # A delta potential is imposed as its exact continuum junction condition,
    # not as Lambda/dx in one grid cell.  The throat value is the fourth-order
    # one-sided derivative constraint, differentiated once for Pi and Pi_t.
    interface_meta=None; interface_i=None; interface_lambda=None
    if interface_jump is not None:
        interface_x=float(interface_jump.get("x",0.0)); interface_lambda=float(interface_jump["Lambda"])
        interface_i=int(np.argmin(abs(x-interface_x)))
        if abs(x[interface_i]-interface_x)>1e-10*max(1,abs(dx)): raise ValueError("interface must be a grid point")
        if interface_i<5 or interface_i>len(x)-6: raise ValueError("interface requires five points on each side")
        jump_den=50.0+12.0*dx*interface_lambda
        def project_interface(a):
            a[interface_i]=(48*(a[interface_i-1]+a[interface_i+1])-36*(a[interface_i-2]+a[interface_i+2])
                            +16*(a[interface_i-3]+a[interface_i+3])-3*(a[interface_i-4]+a[interface_i+4]))/jump_den
            return a
        def jump_residual(a):
            dp=(-25*a[interface_i]+48*a[interface_i+1]-36*a[interface_i+2]+16*a[interface_i+3]-3*a[interface_i+4])/(12*dx)
            dm=(25*a[interface_i]-48*a[interface_i-1]+36*a[interface_i-2]-16*a[interface_i-3]+3*a[interface_i-4])/(12*dx)
            return float(dp-dm-interface_lambda*a[interface_i])
        project_interface(psi); project_interface(pi)
        interface_meta={"x":interface_x,"Lambda":interface_lambda,"method":"fourth-order one-sided jump projection",
                        "condition":"Psi_x(0+)-Psi_x(0-)=Lambda*Psi(0)"}

    # Smooth absorbing layers attenuate any residual Sommerfeld-boundary reflection.
    sigma=np.zeros_like(x)
    if sponge_width>0 and sponge_strength>0:
        qr=x>x[-1]-sponge_width; eta=(x[qr]-(x[-1]-sponge_width))/sponge_width; sigma[qr]=sponge_strength*np.sin(.5*np.pi*eta)**2

    stencils={}
    for name,xe in extraction_points.items():
        i=int(np.searchsorted(x,xe)); start=min(max(i-2,0),len(x)-4)
        inds=np.arange(start,start+4); nodes=x[inds]; weights=np.ones(4)
        for j in range(4):
            for k in range(4):
                if j!=k: weights[j]*=(xe-nodes[k])/(nodes[j]-nodes[k])
        stencils[name]=(inds,weights)

    # A derivative record is requested explicitly. Compose the centered
    # fourth-order field derivative with the same cubic observer extraction;
    # this equals extracting d1_4(psi, dx), without forming that whole array.
    derivative_stencils={}
    for name in derivative_points:
        if name not in stencils: raise ValueError("Derivative recorder is not an extraction point: "+str(name))
        key=name+"_dx"
        if key in extraction_points: raise ValueError("Derivative record name collision: "+key)
        inds,weights=stencils[name]
        if inds[0]<2 or inds[-1]>len(x)-3:
            raise ValueError("Derivative recorder requires centered interior support")
        support=np.arange(inds[0]-2,inds[-1]+3)
        if interface_i is not None and support[0]<=interface_i<=support[-1]:
            raise ValueError("Derivative recorder must not span a derivative-discontinuous interface")
        dw=np.zeros(len(support))
        for node,weight in zip(inds,weights):
            for shift,coefficient in ((-2,1.),(-1,-8.),(1,8.),(2,-1.)):
                dw[node+shift-support[0]]+=weight*coefficient/(12*dx)
        derivative_stencils[key]=(support,dw)

    nrec=n_steps//sample_every+1
    times=np.empty(nrec); records={k:np.empty(nrec) for k in (*stencils,*derivative_stencils)}; ir=0
    monitor_stride=max(1,n_steps//1000); monitor_t=[]; monitor_a=[]

    def rhs(q,p):
        if interface_i is not None:
            q=q.copy(); p=p.copy(); project_interface(q); project_interface(p)
        qt=p.copy(); qxx=d2_4_regular_center(q,dx,ell) if left_bc=="regular_center" else d2_4(q,dx)
        if interface_i is not None:
            # Six same-side points give a fourth-order second derivative at
            # the nodes adjacent to the derivative-discontinuous interface.
            wi=np.array([5/6,-5/4,-1/3,7/6,-1/2,1/12],float)/dx**2
            qxx[interface_i+1]=wi@q[interface_i:interface_i+6]
            qxx[interface_i-1]=wi@q[interface_i:interface_i-6:-1]
        pt=qxx-V*q-2*sigma*p; dq=d1_4(q,dx); dp=d1_4(p,dx)
        if interface_i is not None:
            project_interface(pt)
        if left_bc=="outgoing": qt[0]=dq[0]; pt[0]=dp[0]
        elif left_bc=="regular_center": qt[0]=pt[0]=0.0
        else: raise ValueError(left_bc)
        qt[-1]=-dq[-1]; pt[-1]=-dp[-1]
        return qt,pt

    initial_max=float(np.max(abs(psi))); t0=time.perf_counter()
    for n in range(n_steps+1):
        if n%sample_every==0:
            times[ir]=n*dt
            for name,(inds,weights) in stencils.items(): records[name][ir]=float(weights@psi[inds])
            for name,(inds,weights) in derivative_stencils.items(): records[name][ir]=float(weights@psi[inds])
            ir+=1
        if n%monitor_stride==0 or n==n_steps:
            monitor_t.append(n*dt); monitor_a.append(float(np.max(abs(psi))))
        if n==n_steps: break
        k1q,k1p=rhs(psi,pi); k2q,k2p=rhs(psi+.5*dt*k1q,pi+.5*dt*k1p)
        k3q,k3p=rhs(psi+.5*dt*k2q,pi+.5*dt*k2p); k4q,k4p=rhs(psi+dt*k3q,pi+dt*k3p)
        psi+=dt*(k1q+2*k2q+2*k3q+k4q)/6; pi+=dt*(k1p+2*k2p+2*k3p+k4p)/6
        if left_bc=="regular_center": psi[0]=pi[0]=0.0
        if interface_i is not None: project_interface(psi); project_interface(pi)

    mt=np.asarray(monitor_t); ma=np.asarray(monitor_a); loga=np.log(np.maximum(ma,1e-300))
    edges=np.linspace(mt[0],mt[-1],4); growth_rates={}; growth_limits={}
    for label,lo,hi in zip(("early","middle","late"),edges[:-1],edges[1:]):
        q=(mt>=lo)&(mt<hi if label!="late" else mt<=hi)
        growth_rates[label]=float(np.polyfit(mt[q],loga[q],1)[0]) if np.count_nonzero(q)>=3 else np.nan
        growth_limits[label]=max(sustained_growth_tolerance,float(np.log(2)/max(hi-lo,1e-300)))
    max_amp_ratio=float(np.max(ma)/max(initial_max,1e-300))
    amp_ok=bool(max_amp_ratio<=stability_growth_limit)
    sustained_ok=bool(all(np.isfinite(growth_rates[k]) and growth_rates[k]<=growth_limits[k] for k in ("middle","late")))
    stable=bool(amp_ok and sustained_ok)
    if not stable:
        warnings.warn(f"Numerical stability failure: Amax/A0={max_amp_ratio:.3e}, segmented rates={growth_rates}")
    meta={"dx":dx,"dt":dt,"cfl":dt/dx,"Nx":len(x),"Nt":n_steps+1,"steps":n_steps,
          "x_left":float(x[0]),"x_right":float(x[-1]),"x_extract":float(next(iter(extraction_points.values()))),
          "extraction_points":{k:float(v) for k,v in extraction_points.items()},
          "requested_t_max":t_max,"actual_t_max":actual_tmax,"t_max":actual_tmax,"runtime_s":time.perf_counter()-t0,
          "max_abs_psi":float(np.max(ma)),"max_abs_initial":initial_max,"max_abs_peak":float(np.max(ma)),
          "max_amp_ratio":max_amp_ratio,"late_growth_rates":growth_rates,"growth_rate_limits":growth_limits,"amplitude_growth_pass":amp_ok,
          "sustained_growth_pass":sustained_ok,"stability_status":"STABLE" if stable else "UNSTABLE","stability_pass":stable,
          "absorbing_layer":{"side":"right","width":sponge_width,"strength":sponge_strength,"profile":"disabled" if sponge_strength==0 else "sin^2"},**char}
    if interface_i is not None:
        interface_meta["final_Psi_jump_residual"]=jump_residual(psi); interface_meta["final_Pi_jump_residual"]=jump_residual(pi)
        meta["internal_interface"]=interface_meta
    if derivative_stencils:
        meta["spatial_derivative_recording"]={"method":"centered fourth-order D1 composed with cubic observer interpolation",
            "records":list(derivative_stencils),"coordinate":"outward-increasing native exterior coordinate",
            "dx":dx,"changes_evolution":False}
    return times[:ir],{k:v[:ir] for k,v in records.items()},meta


def normalize(y):
    s=np.max(abs(y)); return y/s if s else y.copy()


def relative_l2(t_ref,y_ref,t,y,t1,t2):
    m=(t_ref>=t1)&(t_ref<=t2); yi=CubicSpline(t,y,extrapolate=False)(t_ref[m]); ok=np.isfinite(yi)
    return float(np.linalg.norm(y_ref[m][ok]-yi[ok])/max(np.linalg.norm(y_ref[m][ok]),1e-300))


def cpm_radiation_flux(wf,ell=2):
    # CPM input only. The evolved RW Q is half the time derivative of CPM, not a constant rescaling.
    # This helper is not used for the notebook RW waveforms; see rw_radiation_flux below.
    coeff=math.factorial(ell+2)/math.factorial(ell-2)/(64*np.pi)
    return coeff*np.gradient(wf.psi,wf.t)**2


def evolve_model(model: PotentialModel, *, cfg=CFG, dx=None, t_max=None, extraction_points=None, grid_kwargs=None):
    """Common model-facing solver requested by the architecture audit."""
    dx=cfg.dx if dx is None else dx; grid_kwargs={} if grid_kwargs is None else dict(grid_kwargs)
    x,V,r,grid_meta=model.potential_grid(dx=dx,**grid_kwargs)
    extraction_points={"infinity":cfg.extraction_x} if extraction_points is None else dict(extraction_points)
    internal_jump={"x":0.0,"Lambda":grid_meta["throat_Lambda"]} if "throat_Lambda" in grid_meta else None
    t,rec,num=evolve_wave(x,V,t_max=cfg.t_max if t_max is None else t_max,cfl=cfg.cfl,pulse_center=cfg.pulse_center,
        pulse_width=cfg.pulse_width,extraction_points=extraction_points,left_bc=model.boundary_conditions()["left"],sample_every=cfg.sample_every,
        interface_jump=internal_jump)
    primary=next(iter(extraction_points)); wf=Waveform(model.name,t,rec[primary],extraction_points[primary],{**model.metadata(),**grid_meta,**num})
    return wf,x,V,r,rec


### Analysis utilities

An **autoregressive, Prony-type linear-prediction estimator** is used as a free time-domain mode finder. It is not called matrix pencil or ESPRIT. Candidate modes are clustered across windows and orders without consulting literature frequencies; only afterward are stable clusters identified against Leaver's modes. A separate variable-projection fit and a fixed-Leaver-spectrum amplitude decomposition prevent a single nonlinear fit from masquerading as an independent QNM calculation. Detector-weighted overlaps use an explicit analytic aLIGO design-sensitivity approximation and a stated $60M_\odot$ total-mass mapping; they are a sensitivity study, not detector-data inference.


In [ ]:
def linear_prediction_modes(t,y,t1,t2,order=12):
    m=(t>=t1)&(t<=t2); tt=t[m]; yy=y[m]
    if yy.size<5*order: raise ValueError("window too short")
    z=hilbert(yy); z-=np.mean(z)
    X=np.column_stack([z[order-k-1:len(z)-k-1] for k in range(order)])
    a,*_=np.linalg.lstsq(X,z[order:],rcond=1e-11); roots=np.roots(np.r_[1,-a])
    s=np.log(roots)/float(np.mean(np.diff(tt))); rows=[]
    for sj,rj in zip(s,roots):
        alpha=-sj.real; omega=abs(sj.imag)
        if alpha>0 and .02<omega<2 and abs(rj)<1.001 and np.isfinite(alpha+omega):
            rows.append({"omega_R":omega,"omega_I":-alpha,"tau":1/alpha,"Q":omega/(2*alpha),"root_abs":abs(rj)})
    return pd.DataFrame(rows).sort_values(["omega_I","omega_R"],ascending=[False,True]).reset_index(drop=True)


def cluster_linear_prediction_modes(table, rel_tol=.055, min_trials=3):
    """Cluster stable complex candidates before any literature-based identification."""
    if len(table)==0: return pd.DataFrame(columns=["omega_R","omega_I","stable_trials","relative_scatter","resolved_candidate"])
    z=(table.omega_R+1j*table.omega_I).to_numpy(); parent=np.arange(len(z))
    def find(i):
        while parent[i]!=i:
            parent[i]=parent[parent[i]]; i=parent[i]
        return i
    def union(i,j):
        a,b=find(i),find(j)
        if a!=b: parent[b]=a
    for i in range(len(z)):
        for j in range(i):
            if table.trial.iloc[i]==table.trial.iloc[j]: continue
            if abs(z[i]-z[j])/max(abs(z[i]),abs(z[j]),1e-12)<rel_tol: union(i,j)
    groups={}
    for i in range(len(z)): groups.setdefault(find(i),[]).append(i)
    rows=[]
    for ids in groups.values():
        trials=table.trial.iloc[ids].nunique(); zz=z[ids]; center=np.median(zz.real)+1j*np.median(zz.imag)
        scatter=float(np.median(abs(zz-center))/max(abs(center),1e-12))
        rows.append({"omega_R":center.real,"omega_I":center.imag,"stable_trials":int(trials),
                     "relative_scatter":scatter,"resolved_candidate":bool(trials>=min_trials and scatter<rel_tol)})
    return pd.DataFrame(rows).sort_values(["resolved_candidate","stable_trials","relative_scatter"],ascending=[False,False,True]).reset_index(drop=True)


def damped_sinusoid_fit(t,y,t1,t2,guess=(.374,.089)):
    m=(t>=t1)&(t<=t2); tt=t[m]; yy=y[m]; tau=tt-tt[0]; yy=yy/max(np.max(abs(yy)),1e-300)
    def design(w,a):
        e=np.exp(-a*tau); return np.column_stack((e*np.cos(w*tau),e*np.sin(w*tau),np.ones_like(tau),tau/np.ptp(tau)))
    def fun(q):
        X=design(*q); beta,*_=np.linalg.lstsq(X,yy,rcond=None); return X@beta-yy
    sol=least_squares(fun,guess,bounds=([.15,.005],[.8,.8]),xtol=1e-12,ftol=1e-12,gtol=1e-12)
    return {"omega_R":sol.x[0],"omega_I":-sol.x[1],"rms":float(np.sqrt(np.mean(fun(sol.x)**2))),"success":sol.success}


def fixed_mode_decomposition(t,y,t1,t2,modes):
    m=(t>=t1)&(t<=t2); tt=t[m]; z=hilbert(y)[m]; tau=tt-tt[0]; z=z/max(np.max(abs(z)),1e-300)
    X=np.column_stack([np.exp((w.imag+1j*w.real)*tau) for w in modes]+[np.ones_like(tau),tau/np.ptp(tau)])
    beta,*_=np.linalg.lstsq(X,z,rcond=1e-12); rms=float(np.sqrt(np.mean(abs(X@beta-z)**2)))
    return beta[:len(modes)],rms


def spectrum(t,y,window="tukey"):
    from scipy.signal.windows import tukey
    dt=float(np.mean(np.diff(t))); win=tukey(len(y),.15) if window=="tukey" else get_window(window,len(y))
    Y=np.fft.rfft((y-np.mean(y))*win); return np.fft.rfftfreq(len(y),dt),abs(Y),np.unwrap(np.angle(Y))


def optimized_overlap(a,b):
    za=hilbert(a); zb=hilbert(b); n=len(za)+len(zb)-1
    corr=np.fft.ifft(np.fft.fft(za,n)*np.conj(np.fft.fft(zb,n)))
    return float(np.max(abs(corr))/(np.linalg.norm(za)*np.linalg.norm(zb)))


def windowed_signal(wf,t_common,t1,t2):
    from scipy.signal.windows import tukey
    m=(t_common>=t1)&(t_common<=t2); y=CubicSpline(wf.t,wf.psi,extrapolate=False)(t_common[m])
    return t_common[m],(y-np.mean(y))*tukey(len(y),.15)


def recurrence_delay(t,y,t_start,delay_hint,search_fraction=.30):
    m=t>=t_start; z=hilbert(y[m]); z-=np.mean(z); dt=float(np.mean(np.diff(t[m])))
    ac=correlate(z,z,mode="full",method="fft")[len(z)-1:]; ac=abs(ac)/np.maximum(np.arange(len(z),0,-1),1)
    lag=np.arange(len(ac))*dt; q=(lag>(1-search_fraction)*delay_hint)&(lag<(1+search_fraction)*delay_hint)
    if not np.any(q): return np.nan,np.nan
    i=np.flatnonzero(q)[np.argmax(ac[q])]; return float(lag[i]),float(ac[i]/ac[0])


def matched_packet_observables(t,y,template_window,min_separation,t_start=None):
    m=(t>=template_window[0])&(t<=template_window[1]); template=y[m]-np.mean(y[m])
    cc=abs(correlate(y-np.mean(y),template,mode="same",method="fft")); dt=float(np.mean(np.diff(t)))
    peaks,_=find_peaks(cc,distance=max(1,int(min_separation/dt)),prominence=.02*np.max(cc))
    peaks=peaks[t[peaks]>=(template_window[0] if t_start is None else t_start)]
    env=abs(hilbert(y)); phase=np.unwrap(np.angle(hilbert(y))); freq=np.gradient(phase,t)/(2*np.pi)
    good=env>1e-3*np.max(env); safe=good.copy(); safe[1:-1] &= good[2:]&good[:-2]
    safe[[0,-1]]=False; freq[~safe]=np.nan
    out=pd.DataFrame({"time":t[peaks],"matched_filter":cc[peaks]/np.max(cc),"amplitude":env[peaks],"phase":phase[peaks],"central_frequency":freq[peaks]})
    if len(out): out["delta_t"]=out.time.diff()
    return out


def envelope_packet_delay(t,y,t_start,delay_hint):
    env=abs(hilbert(y)); dt=float(np.mean(np.diff(t)))
    peaks,_=find_peaks(env,distance=max(1,int(.60*delay_hint/dt)),prominence=.015*np.max(env))
    peaks=peaks[t[peaks]>=t_start]
    return float(np.median(np.diff(t[peaks]))) if len(peaks)>=2 else np.nan


def aligo_design_psd(f):
    f=np.asarray(f,float); x=np.maximum(f,1e-3)/215.0
    shape=x**(-4.14)-5*x**(-2)+111*(1-x*x+.5*x**4)/(1+.5*x*x)
    return 1e-49*np.maximum(shape,1e-6)


def detector_overlap(a,b,dt_M,total_mass_solar=60.0,fmin=20.0,fmax=1024.0):
    from scipy.signal.windows import tukey
    n=len(a); nfft=1<<(2*n-1).bit_length(); Msec=4.925490947e-6*total_mass_solar; dt=dt_M*Msec
    win=tukey(n,.1); A=np.fft.rfft((a-np.mean(a))*win,nfft)*dt; B=np.fft.rfft((b-np.mean(b))*win,nfft)*dt
    f=np.fft.rfftfreq(nfft,dt); band=(f>=fmin)&(f<=min(fmax,f[-1])); Sn=aligo_design_psd(f)
    df=f[1]-f[0]; aa=4*df*np.sum(abs(A[band])**2/Sn[band]); bb=4*df*np.sum(abs(B[band])**2/Sn[band])
    q=np.zeros(nfft,complex); q[np.flatnonzero(band)]=A[band]*np.conj(B[band])/Sn[band]
    corr=4*df*nfft*np.fft.ifft(q)
    return float(np.clip(np.max(abs(corr))/np.sqrt(max(aa*bb,1e-300)),0,1))


## 4. Schwarzschild benchmark, overtone audit, and domain tests

The black-hole reference solves the axial Regge–Wheeler equation. The independent frequency-domain benchmark is Leaver's continued fraction. We calculate $(2,2,n)$ for $n=0,1,2$; the time-domain side uses an unsupervised autoregressive linear-prediction estimator and a nonlinear damped-sinusoid fit. Stable time-domain clusters are established before literature identification. Because broad Gaussian data do not necessarily resolve short-lived overtones, a fixed-Leaver-spectrum decomposition reports their excitation amplitudes without claiming that those frequencies were independently measured.

The same section performs the requested boundary tests for $x_R=(250,300,400)M$, $x_L=(-120,-140,-180)M$ and a simultaneous retarded-time extraction comparison at $x=(75,95,120)M$.


In [ ]:
_BH_CACHE={}


def black_hole_grid(cfg=CFG,dx=None,x_left=-140.0,x_right=400.0):
    dx=cfg.dx if dx is None else float(dx); x=np.arange(x_left,x_right+.5*dx,dx); r=r_from_schwarzschild_rstar(x,cfg.M); V=rw_potential(r,cfg.ell,cfg.M)
    return x,V,r,{"M":cfg.M,"ell":cfg.ell,"x_left":x_left,"x_right":x_right}


def run_black_hole(cfg=CFG,dx=None,t_max=None,x_left=-140.0,x_right=400.0,extraction_xs=(75.0,95.0,120.0),cfl=None):
    dx=cfg.dx if dx is None else float(dx); t_max=cfg.t_max if t_max is None else float(t_max); cfl=cfg.cfl if cfl is None else float(cfl)
    extraction_xs=tuple(float(q) for q in extraction_xs)
    key=(cfg,dx,t_max,float(x_left),float(x_right),extraction_xs,cfl,cfg.sample_every)
    if key in _BH_CACHE: return _BH_CACHE[key]
    x,V,r,grid_meta=black_hole_grid(cfg,dx,x_left,x_right)
    points={f"x{q:g}":q for q in extraction_xs}; points["horizon"]=cfg.qh_surface_x
    t,rec,meta=evolve_wave(x,V,t_max=t_max,cfl=cfl,pulse_center=cfg.pulse_center,pulse_width=cfg.pulse_width,
        extraction_points=points,left_bc="outgoing",sample_every=cfg.sample_every)
    primary=f"x{cfg.extraction_x:g}"
    wf=Waveform("black_hole",t,rec[primary],cfg.extraction_x,{**meta,**grid_meta})
    out=(wf,rec["horizon"],x,V,rec); _BH_CACHE[key]=out; return out


bh,bh_horizon,x_bh,V_bh,bh_records=run_black_hole()
display(pd.DataFrame([bh.metadata]))


In [ ]:
def leaver_rw_residual(z,ell=2,depth=400):
    w=z[0]+1j*z[1]; L=ell*(ell+1)
    alpha=lambda n:(n+1)*(n+1-2j*w)
    beta=lambda n:-2*n*n+(-2+8j*w)*n+8*w*w+4j*w+3-L
    gamma=lambda n:(n-2j*w)**2-4
    tail=beta(depth)
    for n in range(depth-1,0,-1): tail=beta(n)-alpha(n)*gamma(n+1)/tail
    val=beta(0)-alpha(0)*gamma(1)/tail
    return np.array([val.real,val.imag])


def schwarzschild_qnm_leaver(ell=2,guess=(.747,-.178),depth=480,M=1.0):
    sol=root(lambda z:leaver_rw_residual(z,ell,depth),guess,tol=1e-12)
    if not sol.success or np.linalg.norm(sol.fun)>1e-9: raise RuntimeError(sol.message)
    return (sol.x[0]+1j*sol.x[1])/(2*M),sol


published_qnms=np.array([.37367168-.08896232j,.34671100-.27391488j,.30105345-.47827698j])
guesses=[(.747,-.178),(.693,-.548),(.602,-.957)]
leaver_modes=[]; qnm_rows=[]
for n,(guess,lit) in enumerate(zip(guesses,published_qnms)):
    vals=[]
    for depth in [240,480,800]: vals.append(schwarzschild_qnm_leaver(CFG.ell,guess,depth,CFG.M)[0])
    w=vals[-1]; leaver_modes.append(w)
    qnm_rows.append({"mode":f"(2,2,{n})","Leaver":w,"literature":lit,"relative_error":abs(w-lit)/abs(lit),"depth_spread":max(abs(v-w) for v in vals)})
leaver_modes=np.asarray(leaver_modes); bh_qnm_df=pd.DataFrame(qnm_rows)

lp_windows=[(162,195),(164,198),(166,202),(168,205)]; lp_orders=[16,20,24]; lp_tables=[]
for order in lp_orders:
    for win in lp_windows:
        tab=linear_prediction_modes(bh.t,bh.psi,*win,order=order); tab["window"]=str(win); tab["order"]=order; tab["trial"]=f"{order}:{win}"; lp_tables.append(tab)
lp_all=pd.concat(lp_tables,ignore_index=True)
lp_clusters=cluster_linear_prediction_modes(lp_all,rel_tol=.055,min_trials=3)
resolved_clusters=lp_clusters[lp_clusters.resolved_candidate].copy()
td_rows=[]
for n,w in enumerate(leaver_modes):
    if len(resolved_clusters):
        zz=(resolved_clusters.omega_R+1j*resolved_clusters.omega_I).to_numpy(); j=int(np.argmin(abs(zz-w))); candidate=zz[j]
        err=float(abs(candidate-w)/abs(w)); threshold=.08 if n==0 else .15
        # Mode identity is assigned only here, after the literature-independent stability clustering.
        resolved=bool(err<threshold); trials=int(resolved_clusters.iloc[j].stable_trials); scatter=float(resolved_clusters.iloc[j].relative_scatter)
    else: candidate=np.nan; err=np.inf; resolved=False; trials=0; scatter=np.nan
    td_rows.append({"mode":f"(2,2,{n})","Leaver":w,"AR_linear_prediction":candidate if resolved else np.nan,
                    "TD_relative_error":err if resolved else np.nan,"stable_trials":trials,"cluster_scatter":scatter,
                    "resolution_status":"RESOLVED" if resolved else "UNRESOLVED","resolved_free_TD":resolved})
td_qnm_df=pd.DataFrame(td_rows)

fit_rows=[]
for window in [(160,190),(164,198),(168,205)]:
    q=damped_sinusoid_fit(bh.t,bh.psi,*window); q["window"]=str(window); q["relative_error"]=abs((q["omega_R"]+1j*q["omega_I"])-leaver_modes[0])/abs(leaver_modes[0]); fit_rows.append(q)
bh_fit_df=pd.DataFrame(fit_rows)
fixed_amplitudes,fixed_fit_rms=fixed_mode_decomposition(bh.t,bh.psi,158,190,leaver_modes)
td_qnm_df["fixed_spectrum_amplitude"]=abs(fixed_amplitudes)
display(bh_qnm_df); display(lp_clusters.head(12)); display(td_qnm_df); display(bh_fit_df)
print("Fixed three-mode Leaver-basis RMS:",fixed_fit_rms)

# Simultaneous extraction-radius convergence in retarded time u=t-x.
u_grid=np.arange(40,300,0.08); ext_shifted={}
for xe in [75.,95.,120.]:
    key=f"x{xe:g}"; ext_shifted[xe]=CubicSpline(bh.t-xe,bh_records[key],extrapolate=False)(u_grid)
extraction_rows=[]
for xe in [75.,120.]:
    ok=np.isfinite(ext_shifted[xe])&np.isfinite(ext_shifted[95.])
    for region,lo,hi in [("prompt",55,130),("late",130,240),("full",55,240)]:
        m=ok&(u_grid>=lo)&(u_grid<=hi); y=ext_shifted[xe][m]; yref=ext_shifted[95.][m]
        amp=float(np.dot(y,yref)/max(np.dot(y,y),1e-300)); raw=float(np.linalg.norm(y-yref)/max(np.linalg.norm(yref),1e-300)); shape=float(np.linalg.norm(amp*y-yref)/max(np.linalg.norm(yref),1e-300))
        extraction_rows.append({"x_extract":xe,"reference":95.,"region":region,"u_min":lo,"u_max":hi,"retarded_raw_L2":raw,"best_amplitude_factor":amp,"retarded_shape_L2":shape})
extraction_df=pd.DataFrame(extraction_rows)
extraction_raw_error=float(extraction_df.loc[extraction_df.region=="full","retarded_raw_L2"].max())
extraction_error=float(extraction_df.loc[extraction_df.region=="full","retarded_shape_L2"].max())
extraction_prompt_error=float(extraction_df.loc[extraction_df.region=="prompt","retarded_shape_L2"].max())
extraction_late_error=float(extraction_df.loc[extraction_df.region=="late","retarded_shape_L2"].max())

# Mandatory boundary-reflection suite. The full stated analysis interval 100M--340M includes late response
# but ends 10M before the earliest causal return from xR=250M (350M for this pulse/extractor geometry).
right_family={xr:run_black_hole(x_left=-180,x_right=xr,t_max=520,extraction_xs=(95.,))[0] for xr in [250.,300.,400.]}
left_family={xl:run_black_hole(x_left=xl,x_right=400,t_max=520,extraction_xs=(95.,))[0] for xl in [-120.,-140.,-180.]}
boundary_rows=[]
for xr,wf in right_family.items():
    if xr!=400.: boundary_rows.append({"family":"right","variation":f"xR={xr:g}","reference":"xR=400","t_min":100.,"t_max":340.,"earliest_causal_return":350.,"epsilon_boundary":relative_l2(right_family[400.].t,right_family[400.].psi,wf.t,wf.psi,100,340)})
for xl,wf in left_family.items():
    if xl!=-180.: boundary_rows.append({"family":"left","variation":f"xL={xl:g}","reference":"xL=-180","t_min":100.,"t_max":340.,"earliest_causal_return":2*abs(xl)+CFG.pulse_center+CFG.extraction_x,"epsilon_boundary":relative_l2(left_family[-180.].t,left_family[-180.].psi,wf.t,wf.psi,100,340)})
boundary_df=pd.DataFrame(boundary_rows); boundary_error=float(boundary_df.epsilon_boundary.max())
display(extraction_df); display(boundary_df)

fig,ax=plt.subplots(1,3,figsize=(16,4))
ax[0].plot(x_bh,V_bh); ax[0].set(xlim=(-20,50),xlabel=r"$r_*/M$",ylabel=r"$M^2V$",title="Regge–Wheeler potential")
ax[1].plot(bh.t,bh.psi,lw=.8); ax[1].axvspan(164,198,color="tab:orange",alpha=.15); ax[1].set(xlim=(100,300),xlabel=r"$t/M$",title="BH response and fit window")
for xe in ext_shifted: ax[2].plot(u_grid,ext_shifted[xe],lw=.8,label=f"x={xe:g}M")
ax[2].set(xlim=(50,180),xlabel=r"$u/M=t/M-x/M$",title="Retarded-time extraction test"); ax[2].legend()
fig.tight_layout(); fig.savefig(ARTIFACT_DIR/"bh_qnm_boundary_extraction.png")


## 5. Thin-shell gravastar: time domain and independent frequency domain

The de Sitter interior and Schwarzschild exterior are joined at $r=a=M/\mu$. Axial Israel matching gives continuity of $\Psi$ and $\partial_r\Psi$. The time-domain grid is constructed so both the center and shell are exact nodes; the shell potential is represented by a control-volume average and the center stencil uses the regular-series parity $\Psi\sim r^{\ell+1}$.

The independent complex-frequency solver integrates the regular solution $\Psi\sim r^{\ell+1}$ from the center and an outgoing asymptotic series

$$\Psi_{\rm out}=e^{i\omega r_*}\sum_{k=0}^{N}a_k r^{-k}$$

inward from large radius. A zero of their scale-free Wronskian is the QNM. Results are checked against changes in the matching radius and asymptotic order, ten independent complex initial guesses with post-solve root clustering, and a stable autoregressive linear-prediction cluster. This is a genuine boundary-value eigenfrequency calculation, not a cavity-time proxy.


In [ ]:
def gravastar_grid(compactness,cfg=CFG,dx=None,x_right=400.0):
    requested_dx=cfg.dx if dx is None else float(dx); M=cfg.M; a=M/compactness
    if not 0<compactness<.5: raise ValueError("gravastar compactness must lie in (0,1/2)")
    H=np.sqrt(2*M/a**3); xa=float(schwarzschild_rstar(a,M)); rin_star_a=np.arctanh(H*a)/H; x_center=xa-rin_star_a
    n_in=max(4,int(np.rint((xa-x_center)/requested_dx))); dx=(xa-x_center)/n_in
    n_out=int(np.ceil((x_right-xa)/dx)); x=xa+np.arange(-n_in,n_out+1)*dx; shell_index=n_in
    inside=np.arange(len(x))<=shell_index; r=np.empty_like(x)
    u=(x[inside]-xa)+rin_star_a; r[inside]=np.tanh(H*u)/H; r[~inside]=r_from_schwarzschild_rstar(x[~inside],M)
    V=np.empty_like(x); f_in=1-2*M*r[inside]**2/a**3
    V[inside]=f_in*cfg.ell*(cfg.ell+1)/np.maximum(r[inside],1e-12)**2; V[~inside]=rw_potential(r[~inside],cfg.ell,M); V[0]=V[1]
    Vin_shell=(1-2*M/a)*cfg.ell*(cfg.ell+1)/a**2; Vout_shell=float(rw_potential(np.array([a]),cfg.ell,M)[0])
    V[shell_index]=.5*(Vin_shell+Vout_shell)
    return x,V,r,{"a":a,"compactness":compactness,"shell_x":xa,"center_x":x_center,"shell_index":shell_index,
                  "requested_dx":requested_dx,"shell_aligned_error":float(abs(x[shell_index]-xa)),
                  "interface_treatment":"shell-aligned control-volume potential average",
                  "center_treatment":f"fourth-order parity stencil for Psi~r^{cfg.ell+1}"}


_GS_CACHE={}


def run_gravastar(compactness=CFG.gravastar_compactness,cfg=CFG,dx=None,t_max=None,extraction_xs=(75.,95.,120.)):
    key=(cfg,float(compactness),cfg.dx if dx is None else float(dx),cfg.t_max if t_max is None else float(t_max),tuple(float(q) for q in extraction_xs))
    if key in _GS_CACHE: return _GS_CACHE[key]
    x,V,r,meta=gravastar_grid(compactness,cfg,dx); points={f"x{q:g}":q for q in extraction_xs}
    t,rec,num=evolve_wave(x,V,t_max=cfg.t_max if t_max is None else t_max,cfl=cfg.cfl,pulse_center=cfg.pulse_center,
        pulse_width=cfg.pulse_width,extraction_points=points,left_bc="regular_center",sample_every=cfg.sample_every)
    wf=Waveform("thin_shell_gravastar",t,rec[f"x{cfg.extraction_x:g}"],cfg.extraction_x,{**meta,**num,"M":cfg.M,"ell":cfg.ell})
    out=(wf,x,V,r,rec); _GS_CACHE[key]=out; return out


def gravastar_regular_data(a,w,ell=2,M=1.0):
    eps=2e-5*a; H2=2*M/a**3
    def rhs(r,y):
        f=1-H2*r*r; fp=-2*H2*r; V=f*ell*(ell+1)/r**2
        return [y[1],-(fp/f)*y[1]-(w*w-V)/(f*f)*y[0]]
    y0=np.array([eps**(ell+1),(ell+1)*eps**ell],complex)
    sol=solve_ivp(rhs,(eps,a),y0,rtol=2e-10,atol=1e-13,method="DOP853")
    if not sol.success: raise RuntimeError(sol.message)
    return sol.y[:,-1]


def outgoing_rw_series(w,r,ell=2,M=1.0,order=12):
    r=float(r); L=ell*(ell+1); a=np.zeros(order+1,complex); a[0]=1
    for m in range(1,order+1):
        q=((m-1)*m-L)*a[m-1]
        if m>=2: q+=M*(-2*(m-2)*m+6)*a[m-2]
        a[m]=q/(2j*w*m)
    n=np.arange(order+1); S=np.sum(a*r**(-n)); Sp=np.sum(-n*a*r**(-n-1))
    return S,Sp


def gravastar_qnm_residual(w,compactness=.47,ell=2,M=1.0,rmax=60.0,order=12):
    a=M/compactness; yin=gravastar_regular_data(a,w,ell,M); S,Sp=outgoing_rw_series(w,rmax,ell,M,order)
    fmax=1-2*M/rmax; y0=np.array([S,1j*w/fmax*S+Sp],complex)
    def rhs(r,y):
        f=1-2*M/r; fp=2*M/r**2; V=f*(ell*(ell+1)/r**2-6*M/r**3)
        return [y[1],-(fp/f)*y[1]-(w*w-V)/(f*f)*y[0]]
    sol=solve_ivp(rhs,(rmax,a),y0,rtol=2e-10,atol=2e-12,method="DOP853")
    if not sol.success: raise RuntimeError(sol.message)
    yout=sol.y[:,-1]
    return (yin[0]*yout[1]-yin[1]*yout[0])/(np.linalg.norm(yin)*np.linalg.norm(yout))


def gravastar_qnm(compactness=.47,guess=(.405,-.025),rmax=60.,order=12):
    def fun(z):
        q=gravastar_qnm_residual(z[0]+1j*z[1],compactness,CFG.ell,CFG.M,rmax,order); return [q.real,q.imag]
    sol=root(fun,guess,tol=1e-11)
    w=sol.x[0]+1j*sol.x[1]
    if not sol.success or abs(gravastar_qnm_residual(w,compactness,CFG.ell,CFG.M,rmax,order))>1e-8: raise RuntimeError(sol.message)
    return w


gs,x_gs,V_gs,r_gs,gs_records=run_gravastar()
gravastar_fd=[]; g0=(.405,-.025)
for rmax in [30.,40.,60.,80.]:
    for order in [10,12,14]:
        w=gravastar_qnm(CFG.gravastar_compactness,g0,rmax,order); g0=(w.real,w.imag)
        gravastar_fd.append({"rmax":rmax,"series_order":order,"omega":w,"residual":abs(gravastar_qnm_residual(w,CFG.gravastar_compactness,CFG.ell,CFG.M,rmax,order))})
gravastar_fd_df=pd.DataFrame(gravastar_fd); gs_mode_fd=gravastar_fd_df.iloc[-1].omega
gravastar_fd_spread=max(abs(w-gs_mode_fd) for w in gravastar_fd_df.omega)

# Independent multi-start robustness audit: guesses are not chained and the
# converged roots are clustered only after every solve has finished.
gravastar_multistart_guesses=[(.38,-.06),(.40,-.02),(.40,-.04),(.42,-.04),(.45,-.08),
                             (.35,-.03),(.50,-.06),(.30,-.012),(.43,-.015),(.405,-.025)]
gravastar_multistart_rows=[]
for guess in gravastar_multistart_guesses:
    try:
        wr=gravastar_qnm(CFG.gravastar_compactness,guess,60.,14)
        if not (0<wr.real<1.5 and -1<wr.imag<0):
            raise RuntimeError(f"non-decaying/out-of-band root rejected: {wr}")
        gravastar_multistart_rows.append({"guess":complex(*guess),"converged":True,"omega":wr,
            "residual":abs(gravastar_qnm_residual(wr,CFG.gravastar_compactness,CFG.ell,CFG.M,60.,14)),"message":"converged"})
    except Exception as exc:
        gravastar_multistart_rows.append({"guess":complex(*guess),"converged":False,"omega":np.nan+1j*np.nan,"residual":np.nan,"message":str(exc)})
gravastar_multistart_df=pd.DataFrame(gravastar_multistart_rows)
converged_roots=gravastar_multistart_df.loc[gravastar_multistart_df.converged,"omega"].to_numpy(complex)
clusters=[]
for w in converged_roots:
    placed=False
    for c in clusters:
        if abs(w-c["center"])/max(abs(c["center"]),1e-12)<2e-6:
            c["members"].append(w); c["center"]=sum(c["members"])/len(c["members"]); placed=True; break
    if not placed: clusters.append({"center":w,"members":[w]})
gs_lp=[]
for win in [(175,235),(180,240),(200,300),(250,450)]:
    q=linear_prediction_modes(gs.t,gs.psi,*win,order=12); q["window"]=str(win); q["trial"]=str(win); gs_lp.append(q)
gs_lp=pd.concat(gs_lp,ignore_index=True); gs_clusters=cluster_linear_prediction_modes(gs_lp,rel_tol=.06,min_trials=3)
gs_resolved=gs_clusters[gs_clusters.resolved_candidate]
# Order independently of the continuation seed. Claim only the least-damped
# recovered, TD-supported axial candidate, not completeness of the spectrum.
trial_modes=(gs_resolved.omega_R+1j*gs_resolved.omega_I).to_numpy()
cluster_info=[]; stability_trials=[]
for cl in clusters:
    z=cl["center"]
    err=np.min(abs(trial_modes-z))/abs(z) if len(trial_modes) else np.inf
    experiments=[]
    for rm,order in [(40.,14),(80.,14),(60.,10),(60.,12)]:
        try:
            value=gravastar_qnm(CFG.gravastar_compactness,(z.real,z.imag),rm,order)
            frac=abs(value-z)/abs(z)
            resid=abs(gravastar_qnm_residual(value,CFG.gravastar_compactness,CFG.ell,CFG.M,rm,order))
            message="converged"
        except (RuntimeError,ValueError,FloatingPointError,OverflowError) as exc:
            value=complex(np.nan,np.nan); frac=np.inf; resid=np.nan; message=str(exc)
        experiments.append(dict(cluster=z,rmax=rm,series_order=order,omega=value,relative_shift=frac,residual=resid,message=message))
    stability_trials.extend(experiments)
    spread=max(r["relative_shift"] for r in experiments)
    cluster_info.append(dict(omega=z,n_starts=len(cl["members"]),
        max_cluster_spread=max(abs(q-z) for q in cl["members"]),
        residual=abs(gravastar_qnm_residual(z,CFG.gravastar_compactness,CFG.ell,CFG.M,60.,14)),
        damping=-z.imag,TD_relative_distance=err,TD_supported=err<.08,FD_relative_spread=spread))
gravastar_root_clusters_df=pd.DataFrame(cluster_info,columns=["omega","n_starts","max_cluster_spread",
    "residual","damping","TD_relative_distance","TD_supported","FD_relative_spread"]).sort_values("damping").reset_index(drop=True)
verified=gravastar_root_clusters_df.loc[gravastar_root_clusters_df.TD_supported & (gravastar_root_clusters_df.FD_relative_spread<1e-6)]
if verified.empty: raise RuntimeError("No independently clustered, TD-supported gravastar root.")
gravastar_root_stability_df=pd.DataFrame(stability_trials); display(gravastar_root_stability_df)
gravastar_root_stability_df.to_csv(ARTIFACT_DIR/"gravastar_root_stability.csv",index=False)
candidate=verified.iloc[0]; gs_mode_multistart=complex(candidate.omega)
gs_mode_td=trial_modes[np.argmin(abs(trial_modes-gs_mode_multistart))]
gravastar_multistart_attempt_count=len(gravastar_multistart_df)
gravastar_fundamental_support=int(candidate.n_starts)
gravastar_multistart_fund_error=abs(gs_mode_multistart-gs_mode_fd)/abs(gs_mode_fd)
print("Distinct decaying roots in sampled band:",len(clusters),"relative cluster tolerance: 2e-6;",
      "candidate is least damped among TD-supported roots, not a global mode count.")
gs_td_fd_error=float(abs(gs_mode_td-gs_mode_fd)/abs(gs_mode_fd)) if np.isfinite(gs_mode_td) else np.inf
display(gravastar_fd_df); display(gravastar_multistart_df); display(gravastar_root_clusters_df); display(gs_clusters)
display(pd.DataFrame([{"omega_FD":gs_mode_fd,"omega_multistart":gs_mode_multistart,"omega_TD":gs_mode_td,
    "multistart_relative_error":gravastar_multistart_fund_error,"TD_relative_error":gs_td_fd_error,"FD_rmax_order_spread":gravastar_fd_spread}]))

a=gs.metadata["a"]; f_in=1-2*CFG.M*a*a/a**3; f_out=1-2*CFG.M/a; gravastar_shell_residual=abs(f_in-f_out)
gravastar_shell_alignment_error=float(gs.metadata["shell_aligned_error"])
fig,ax=plt.subplots(1,2,figsize=(12,4)); ax[0].plot(x_gs,V_gs); ax[0].axvline(gs.metadata["shell_x"],c="k",ls="--")
ax[0].set(xlim=(gs.metadata["center_x"],40),ylim=(0,1),xlabel=r"$x/M$",title="Gravastar axial potential")
ax[1].plot(gs.t,gs.psi,lw=.8); ax[1].set(xlim=(100,450),xlabel=r"$t/M$",title=f"TD response; FD $M\\omega={gs_mode_fd.real:.4f}{gs_mode_fd.imag:+.4f}i$")
fig.tight_layout(); fig.savefig(ARTIFACT_DIR/"gravastar_fd_td.png")


## 6. Mini boson star: equilibrium, fixed-background axial response, photon sphere

Macedo et al. use $\tilde\phi=\sqrt{4\pi G}\phi$; the code uses $\sigma=\sqrt{8\pi G}\phi=\sqrt2\tilde\phi$. The continuation BVP targets mini-BS I in their Table I.

The perturbation evolved here is the **decoupled axial metric perturbation on the equilibrium background**. It is not the full boson-star gravitational waveform; the coupled scalar–metric polar problem is an explicit open extension.

For $g_{tt}=-\alpha^2(r)$, a null circular orbit satisfies

$$\frac{d}{dr}\left(\frac{\alpha^2}{r^2}\right)=0\quad\Longleftrightarrow\quad r\frac{\alpha'}{\alpha}-1=0.$$

The code searches this condition and reports either all roots or a quantitative negative margin establishing absence.


In [ ]:
@dataclass
class BosonBackground:
    r: np.ndarray; sigma: np.ndarray; dsigma: np.ndarray; a: np.ndarray; alpha: np.ndarray
    omega_s: float; mass: float; radius_99: float; residual: float; sigma_c: float; phi_tilde_c: float


def solve_mini_boson_star(sigma_c=CFG.boson_sigma_c,rmax=50,n=1200,tol=2e-7):
    r=np.linspace(1e-5,rmax,n)
    def ode(r,y,p):
        s,u,a,al=y; w=p[0]; rr=np.maximum(r,1e-10)
        ap_a=(1-a*a)/(2*rr)+.5*rr*((w*w/al**2+1)*a*a*s*s+u*u)
        alp_al=(a*a-1)/(2*rr)+.5*rr*((w*w/al**2-1)*a*a*s*s+u*u)
        return np.vstack((u,-(2/rr+alp_al-ap_a)*u-a*a*(w*w/al**2-1)*s,a*ap_a,al*alp_al))
    start=min(.18,sigma_c); s=start*np.exp(-r/5); y=np.vstack((s,-s/5,1+.02*r*r*np.exp(-r/4),.90+.10*r/rmax)); p=np.array([.90]); sol=None
    for sc in np.linspace(start,sigma_c,max(2,int(np.ceil(abs(sigma_c-start)/.02))+1)):
        def bc(ya,yb,p,sc=sc): return np.array([ya[0]-sc,ya[1],ya[2]-1,yb[0],yb[2]*yb[3]-1])
        sol=solve_bvp(ode,bc,r,y,p=p,tol=tol,max_nodes=50000,verbose=0)
        if sol.status!=0: raise RuntimeError(f"BVP continuation failed at sigma_c={sc}: {sol.message}")
        y=sol.sol(r); p=sol.p
    rr=np.linspace(1e-5,rmax,5000); s,u,a,al=sol.sol(rr); mass=.5*rr*(1-1/a**2); M=float(mass[-1])
    idx=np.searchsorted(mass,.99*M); R99=float(rr[min(idx,len(rr)-1)])
    bc_res=max(abs(s[0]-sigma_c),abs(u[0]),abs(a[0]-1),abs(s[-1]),abs(a[-1]*al[-1]-1)); residual=float(max(bc_res,np.max(sol.rms_residuals)))
    return BosonBackground(rr,s,u,a,al,float(sol.p[0]),M,R99,residual,float(sigma_c),float(sigma_c/np.sqrt(2)))


def boson_photon_spheres(bg):
    dalpha=CubicSpline(bg.r,bg.alpha).derivative()(bg.r); condition=bg.r*dalpha/bg.alpha-1
    roots=[]; valid=(bg.r>.02)&(bg.r<bg.r[-1]-1)
    ids=np.flatnonzero(valid[:-1]&valid[1:]&(condition[:-1]*condition[1:]<0))
    spline=CubicSpline(bg.r,condition)
    for i in ids: roots.append(brentq(spline,bg.r[i],bg.r[i+1]))
    return np.asarray(roots),float(np.max(condition[valid])),condition


bs_bg=solve_mini_boson_star(); photon_spheres,photon_sphere_margin,photon_condition=boson_photon_spheres(bs_bg)
lit_bs={"phi_tilde_c":.1916,"omega_s":.853087,"mass":.63300,"radius_99":7.86149}
bs_benchmark=pd.DataFrame([{"quantity":k,"published":v,"numerical":getattr(bs_bg,k),"relative_error":abs(getattr(bs_bg,k)-v)/abs(v)} for k,v in lit_bs.items()])
print("Photon sphere:","absent" if len(photon_spheres)==0 else photon_spheres,"; max[r alpha'/alpha - 1] =",photon_sphere_margin)
display(bs_benchmark)

fig,ax=plt.subplots(1,4,figsize=(17,4)); ax[0].plot(bs_bg.r,bs_bg.sigma/np.sqrt(2)); ax[0].set(xlim=(0,25),xlabel=r"$\mu r$",ylabel=r"$\tilde\phi$",title="Scalar")
ax[1].plot(bs_bg.r,.5*bs_bg.r*(1-1/bs_bg.a**2)); ax[1].set(xlim=(0,25),title="Mass function")
ax[2].plot(bs_bg.r,bs_bg.alpha**2,label=r"$\alpha^2$"); ax[2].plot(bs_bg.r,bs_bg.a**2,label=r"$a^2$"); ax[2].set(xlim=(0,25),title="Metric"); ax[2].legend()
ax[3].plot(bs_bg.r,photon_condition); ax[3].axhline(0,c="k",ls="--"); ax[3].set(xlim=(0,25),xlabel=r"$\mu r$",title="Photon-sphere condition")
fig.tight_layout(); fig.savefig(ARTIFACT_DIR/"boson_background_photon_sphere.png")


In [ ]:
def boson_axial_grid(bg,cfg=CFG,dx=None,x_right_M=400):
    ell=cfg.ell; dx=cfg.dx if dx is None else dx; m=.5*bg.r*(1-1/bg.a**2)
    x=cumulative_trapezoid(bg.a/bg.alpha,bg.r,initial=0); V=bg.alpha**2*(ell*(ell+1)/bg.r**2-6*m/bg.r**3+bg.sigma**2)
    xM=x/bg.mass; VM=V*bg.mass**2; rM=bg.r/bg.mass; xu=np.arange(0,x_right_M+.5*dx,dx); Vu=np.empty_like(xu); ru=np.empty_like(xu)
    inside=xu<=xM[-1]; Vu[inside]=PchipInterpolator(xM,VM)(xu[inside]); ru[inside]=PchipInterpolator(xM,rM)(xu[inside])
    r_end=rM[-1]; x_end=xM[-1]; rs_end=schwarzschild_rstar(r_end,1.0)
    ru[~inside]=r_from_schwarzschild_rstar(rs_end+(xu[~inside]-x_end),1.0); Vu[~inside]=rw_potential(ru[~inside],ell,1.0); Vu[0]=Vu[1]
    return xu,Vu,ru


def run_boson_star(bg=bs_bg,cfg=CFG,dx=None,t_max=None):
    x,V,r=boson_axial_grid(bg,cfg,dx); t,rec,num=evolve_wave(x,V,t_max=cfg.t_max if t_max is None else t_max,cfl=cfg.cfl,
        pulse_center=cfg.pulse_center,pulse_width=cfg.pulse_width,extraction_points={"infinity":cfg.extraction_x},left_bc="regular_center",sample_every=cfg.sample_every)
    local_photon_spheres,local_photon_margin,_=boson_photon_spheres(bg)
    meta={**num,"ADM_mass_scaled":1.,"raw_mass_mu_units":bg.mass,"omega_s_over_mu":bg.omega_s,"R99_over_M":bg.radius_99/bg.mass,
          "sigma_c":bg.sigma_c,"EKG_residual":bg.residual,"ell":cfg.ell,"perturbation_scope":"fixed-background axial metric sector",
          "photon_sphere":bool(len(local_photon_spheres)),"r_ph":float(local_photon_spheres[0]/bg.mass) if len(local_photon_spheres) else np.nan,
          "photon_spheres_mu_r":local_photon_spheres.tolist(),"photon_sphere_margin":local_photon_margin,
          "photon_sphere_metadata_scope":"recomputed for this background"}
    return Waveform("mini_boson_star_axial_fixed_background",t,rec["infinity"],cfg.extraction_x,meta),x,V,r


bs,x_bs,V_bs,r_bs=run_boson_star(); display(pd.DataFrame([bs.metadata]))
fig,ax=plt.subplots(1,2,figsize=(12,4)); ax[0].plot(x_bs,V_bs); ax[0].set(xlim=(0,80),ylim=(0,1),xlabel=r"$x/M$",title="Axial fixed-background potential")
ax[1].plot(bs.t,bs.psi,lw=.8); ax[1].set(xlim=(100,400),xlabel=r"$t/M$",title="Axial fixed-background response")
fig.tight_layout(); fig.savefig(ARTIFACT_DIR/"boson_axial_fixed_background.png")


## 7. Schwarzschild thin-shell wormhole with axial gravitational throat matching

Two Schwarzschild exteriors $r\ge a>2M$ are joined at $x=0$.  The axial potential is not merely a smooth mirrored Regge--Wheeler barrier.  The distributional throat matter found by Yang, Xia, and Miao (EPJC **85**, 742, 2025) gives

$$V_{\rm WH}(x)=V_{\rm RW}[r(|x|)]+\Lambda_{\rm th}\delta(x),
\qquad \Lambda_{\rm th}=\frac{2(a-M)}{a^2}>0.$$

With $\Psi_{tt}-\Psi_{xx}+V\Psi=0$, the exact matching conditions are

$$[\Psi]_0=0,\qquad \Psi_x(0^+)-\Psi_x(0^-)=\Lambda_{\rm th}\Psi(0).$$

The solver imposes this junction directly with fourth-order one-sided derivatives; it never replaces the delta distribution by a narrow cell.  For a plane wave the isolated throat has

$$r_\delta=\frac{\Lambda_{\rm th}}{2i\omega-\Lambda_{\rm th}},\qquad
t_\delta=\frac{2i\omega}{2i\omega-\Lambda_{\rm th}}.$$

Later cells compose these amplitudes with both exterior barriers.  The packet delay and QNM condition therefore include barrier propagation, throat reflection/transmission, and the full cavity denominator.


In [ ]:
def rw_peak_radius(ell=2,M=1.0):
    L=ell*(ell+1); roots=np.roots([2*L,-(6*L+18)*M,48*M*M]); return float(np.max(np.real(roots[np.isreal(roots)])))


def wormhole_grid(r0=CFG.wormhole_r0,cfg=CFG,dx=None,x_left=-260,x_right=400):
    if r0<=2*cfg.M: raise ValueError("r0 must exceed 2M")
    dx=cfg.dx if dx is None else dx
    # Construct outward from the throat so x=0 is exact at every resolution.
    xl=-np.arange(dx,abs(x_left)+.5*dx,dx)[::-1]; xr=np.arange(0,x_right+.5*dx,dx); x=np.r_[xl,xr]
    rs0=float(schwarzschild_rstar(r0,cfg.M))
    r=r_from_schwarzschild_rstar(abs(x)+rs0,cfg.M); V=rw_potential(r,cfg.ell,cfg.M); rpk=rw_peak_radius(cfg.ell,cfg.M)
    xpk=float(schwarzschild_rstar(rpk,cfg.M)-rs0); Lambda=2*(r0-cfg.M)/r0**2
    return x,V,r,{"r0":r0,"rstar0":rs0,"r_peak":rpk,"x_peak":xpk,"geometric_roundtrip":4*xpk,
        "throat_Lambda":Lambda,"throat_term":"Lambda_th delta(x)","throat_matching":"Psi_x(0+)-Psi_x(0-)=Lambda_th Psi(0)",
        "throat_reference":"Yang, Xia & Miao, Eur. Phys. J. C 85, 742 (2025), doi:10.1140/epjc/s10052-025-14446-w"}


def throat_delta_coefficients(w,Lambda):
    w=np.asarray(w,float); den=2j*w-Lambda
    return Lambda/den,2j*w/den


def run_wormhole(r0=CFG.wormhole_r0,cfg=CFG,dx=None,t_max=None,extraction_xs=(75.,95.,120.),*,x_left=-400.):
    x,V,r,meta=wormhole_grid(r0,cfg,dx,x_left=x_left); points={f"right{q:g}":q for q in extraction_xs}; points["left"]=-cfg.extraction_x
    t,rec,num=evolve_wave(x,V,t_max=cfg.t_max if t_max is None else t_max,cfl=cfg.cfl,pulse_center=cfg.pulse_center,
        pulse_width=cfg.pulse_width,extraction_points=points,left_bc="outgoing",sample_every=cfg.sample_every,
        interface_jump={"x":0.0,"Lambda":meta["throat_Lambda"]})
    wf=Waveform("schwarzschild_thin_shell_wormhole_axial_matching",t,rec[f"right{cfg.extraction_x:g}"],cfg.extraction_x,{**meta,**num,"M":cfg.M,"ell":cfg.ell})
    return wf,rec["left"],x,V,rec


wh,wh_left,x_wh,V_wh,wh_records=run_wormhole(); overlap=abs(x_wh)<=min(abs(x_wh[0]),abs(x_wh[-1]))
symmetry_residual=float(np.max(abs(V_wh[overlap]-np.interp(-x_wh[overlap],x_wh,V_wh))))
w_delta=np.geomspace(1e-4,2,1000); rd,td=throat_delta_coefficients(w_delta,wh.metadata["throat_Lambda"])
throat_flux_residual=float(np.max(abs(abs(rd)**2+abs(td)**2-1)))
throat_scattering_jump_residual=float(np.max(abs(2j*w_delta*rd-wh.metadata["throat_Lambda"]*(1+rd))))
print(wh.metadata); print("Potential symmetry residual:",symmetry_residual,"throat flux/jump:",throat_flux_residual,throat_scattering_jump_residual)

# Executable common-interface registry. Each entry can be passed to evolve_model(...).
MODEL_REGISTRY={
    "schwarzschild":GridModelAdapter("black_hole",lambda dx,**kw:black_hole_grid(CFG,dx,**kw),"outgoing",{"class":"black-hole reference"}),
    "gravastar":GridModelAdapter("thin_shell_gravastar",lambda dx,**kw:gravastar_grid(kw.pop("compactness",CFG.gravastar_compactness),CFG,dx,**kw),"regular_center",{"class":"horizonless geometric ECO"}),
    "boson_star":GridModelAdapter("mini_boson_star_axial_fixed_background",lambda dx,**kw:(*boson_axial_grid(kw.pop("background",bs_bg),CFG,dx,**kw),{"scope":"fixed-background axial"}),"regular_center",{"class":"horizonless geometric ECO"}),
    "wormhole":GridModelAdapter("schwarzschild_thin_shell_wormhole_axial_matching",lambda dx,**kw:wormhole_grid(kw.pop("r0",CFG.wormhole_r0),CFG,dx,**kw),"outgoing",{"class":"horizonless geometric ECO","internal_interface":"axial throat jump"}),
}
architecture_df=pd.DataFrame([{"key":k,"model":v.name,"boundary_conditions":v.boundary_conditions(),**v.metadata()} for k,v in MODEL_REGISTRY.items()])
display(architecture_df)

# Independent source-only BH controls with local field AND spatial derivative.
# All positions below are global Schwarzschild tortoise coordinates in M units.
_CHAR_SOURCE_CACHE={}
def matched_source_control(offset,dx=None,monitors=(-55.,-65.,-75.),sample_t=None,*,x_left=-400.,t_max=650.):
    dx=CFG.dx if dx is None else float(dx)
    sample_t=bh.t if sample_t is None else np.asarray(sample_t,float)
    monitors=tuple(float(q) for q in monitors)
    if len(monitors)!=len(set(monitors)) or not monitors: raise ValueError("Distinct monitors required")
    if sample_t.ndim!=1 or not np.all(np.isfinite(sample_t)) or np.any(np.diff(sample_t)<=0) or sample_t[0]<0:
        raise ValueError("Common time grid must be finite, increasing, and nonnegative")
    end=max(float(t_max),float(sample_t[-1])+.2)
    key=(float(offset),dx,monitors,float(x_left),end,CFG)
    if key not in _CHAR_SOURCE_CACHE:
        x=np.arange(x_left,400.+.5*dx,dx)
        v=rw_potential(r_from_schwarzschild_rstar(x+offset,CFG.M),CFG.ell,CFG.M)
        points={"det":CFG.extraction_x}
        names=[f"monitor{j}" for j in range(len(monitors))]
        points.update({name:m-offset for name,m in zip(names,monitors)})
        if any(not x[4]<q<x[-5] for q in points.values()): raise ValueError("Observer outside safe extraction interior")
        t,rec,meta=evolve_wave(x,v,t_max=end,cfl=CFG.cfl,pulse_center=CFG.pulse_center,
            pulse_width=CFG.pulse_width,extraction_points=points,left_bc="outgoing",
            sample_every=CFG.sample_every,derivative_points=tuple(names))
        wf=Waveform("matched_source_only_BH",t,rec["det"],CFG.extraction_x,
            {**meta,"global_monitors":monitors,"global_source":CFG.pulse_center+offset,
             "global_detector":CFG.extraction_x+offset,"native_coordinate_offset":offset})
        records={m:{"native_t":t,"native_total":rec[name],"native_dpsi":rec[name+"_dx"],
                    "monitor_global_rstar":m,"monitor_native":m-offset,"dx":meta["dx"],
                    "potential_at_monitor":float(rw_potential(r_from_schwarzschild_rstar(m,CFG.M),CFG.ell,CFG.M)),
                    "earliest_8sigma_left_return":CFG.pulse_center-8*CFG.pulse_width+m-offset-2*x[0]}
                 for name,m in zip(names,monitors)}
        _CHAR_SOURCE_CACHE[key]=(wf,records)
    wf,native=_CHAR_SOURCE_CACHE[key]
    records={}
    for m,rec in native.items():
        if sample_t[-1]>rec["native_t"][-1]: raise ValueError("Derivative source record too short")
        records[m]={**rec,"t":sample_t.copy(),
            "total":CubicSpline(rec["native_t"],rec["native_total"],extrapolate=False)(sample_t),
            "dpsi":CubicSpline(rec["native_t"],rec["native_dpsi"],extrapolate=False)(sample_t)}
    return wf,records

# Recording continues beyond the analysis window, preventing a truncated
# late monitor field from biasing the characteristic decomposition near DC.
source_dt=float(bh.t[1]-bh.t[0])
source_t=source_dt*np.arange(int(np.floor(650./source_dt))+1)
gs_bh_native,gs_monitor_records=matched_source_control(0.,sample_t=source_t)
wh_bh_native,wh_monitor_records=matched_source_control(wh.metadata["rstar0"],sample_t=source_t)
reference_native={"GS":gs_bh_native,"WH":wh_bh_native,"BS":bh,"QH":bh,"BH":bh}
# Also match the native BS experiment to its own physical BH control.
# Equal numeric center-based x and Schwarzschild r* are not equal radii.
# Use the *evolved* radius map, including its analytic vacuum continuation
# beyond the end of the stellar equilibrium grid. Both target nodes here
# are represented on that grid; the short equilibrium alone cannot cover 95M.
bs_native_radii=PchipInterpolator(x_bs,r_bs,extrapolate=False)([CFG.pulse_center,CFG.extraction_x])
assert np.all(np.isfinite(bs_native_radii)) and np.all(bs_native_radii>2*CFG.M)
bs_reference_positions=schwarzschild_rstar(bs_native_radii,CFG.M)
bs_reference_cfg=replace(CFG,pulse_center=float(bs_reference_positions[0]),
    extraction_x=float(bs_reference_positions[1]))
bs_native_bh,*_=run_black_hole(cfg=bs_reference_cfg,t_max=650.,x_left=-400.,
    extraction_xs=(bs_reference_cfg.extraction_x,))
reference_native['BS']=bs_native_bh
reference_common={k:CubicSpline(v.t,v.psi,extrapolate=False)(bh.t) for k,v in reference_native.items()}
# Legacy total-record aliases are diagnostic only; physical reconstruction
# explicitly decomposes both longer records frequency by frequency.
gs_incident=gs_monitor_records[-65.]["total"][:len(bh.t)]
wh_incident=wh_monitor_records[-65.]["total"][:len(bh.t)]


# A finite left exterior can return radiation to the *far* observer before it
# reaches the near observer. Compare three domains, holding all physics fixed.
wh_boundary_rows=[]
central_left=float(wh.metadata["x_left"])
for end in (-260.,-480.):
    other,other_left,*_=run_wormhole(x_left=end)
    for side,main_data,test_data,observer in (("near",wh.psi,other.psi,CFG.extraction_x),
                                              ("far",wh_left,other_left,-CFG.extraction_x)):
        interp=CubicSpline(other.t,test_data,extrapolate=False)(wh.t)
        for lo,hi in ((190.,450.),(190.,500.),(470.,500.)):
            ok=(wh.t>=lo)&(wh.t<=hi)
            val=np.linalg.norm((interp-main_data)[ok])/np.linalg.norm(main_data[ok])
            # Approximate causal bound for a Gaussian, truncating its negligible
            # tail at 8 widths; this is not a statement of strict compact support.
            earliest=CFG.pulse_center-8*CFG.pulse_width+observer-2*max(end,central_left)
            wh_boundary_rows.append(dict(channel=side,primary_left=central_left,trial_left=end,
                 window_start=lo,window_end=hi,earliest_8sigma_left_boundary_return_M=earliest,
                 relative_domain_change=val))
wh_domain_df=pd.DataFrame(wh_boundary_rows)
wh_domain_df.to_csv(ARTIFACT_DIR/"wh_observer_domain_convergence.csv",index=False)
wh_domain_fine=wh_domain_df[wh_domain_df.trial_left==-480.]
assert (wh_domain_fine.earliest_8sigma_left_boundary_return_M>wh_domain_fine.window_end).all()
assert (wh_domain_fine.relative_domain_change<1e-7).all(),wh_domain_df
display(wh_domain_df.query("window_start==190 and window_end==500"))

def monitor_characteristics(t,psi,dpsi,nfft,eta):
    """Measured characteristic amplitudes in NUMPY's convention.

    Physical transform integrates exp(+izt), z=w+i*eta. NumPy F,D are its
    complex conjugates for real records, so conjugating 1/(iz) changes BOTH
    the sign and the damping. In NumPy coordinates I=(F+D/(eta+iw))/2.
    The w=0 bin lies at z=i*eta, NOT at physical z=0. It remains finite.
    A static real-axis field has no unique directional split: eta=0 raises.
    """
    t,psi,dpsi=map(np.asarray,(t,psi,dpsi))
    if eta<=0 or not np.isfinite(eta): raise ValueError('positive Laplace damping required; real-axis DC is ambiguous')
    if t.ndim!=1 or len(t)<3 or psi.shape!=t.shape or dpsi.shape!=t.shape: raise ValueError('same one-dimensional grid required')
    if not np.isrealobj(psi) or not np.isrealobj(dpsi) or not np.isfinite([t,psi,dpsi]).all(): raise ValueError('finite real records required')
    dt=t[1]-t[0]
    if abs(t[0])>1e-12 or dt<=0 or not np.allclose(np.diff(t),dt,rtol=1e-9,atol=1e-12) or nfft<len(t): raise ValueError('uniform t=0 record and adequate padding required')
    omega=2*np.pi*np.fft.rfftfreq(nfft,dt); damp=np.exp(-eta*t)
    F=np.fft.rfft(psi*damp,nfft); D=np.fft.rfft(dpsi*damp,nfft)
    q=D/(eta+1j*omega)
    return dict(omega=omega+1j*eta,field_fft=F,derivative_fft=D,
                ingoing_fft=.5*(F+q),outgoing_fft=.5*(F-q))

def characteristic_diagnostics(parts,cutoff):
    z=parts['omega']; f=parts['field_fft']; d=parts['derivative_fft']
    inc=parts['ingoing_fft']; out=parts['outgoing_fft']; band=z.real<=cutoff
    active=band&(abs(inc)>1e-6*np.max(abs(inc[band])))
    return dict(outgoing_to_ingoing_l2=float(np.linalg.norm(out[band])/np.linalg.norm(inc[band])),
        masked_max_abs_O_over_I=float(np.max(abs(out[active]/inc[active]))),
        input_amplitude_floor_fraction=1e-6,
        identity_relative_L2=float(np.linalg.norm(inc+out-f)/max(np.linalg.norm(f),1e-300)),
        derivative_identity_relative_L2=float(np.linalg.norm((z.imag+1j*z.real)*(inc-out)-d)/max(np.linalg.norm(d),1e-300)),
        excluded_ingoing_spectral_power=float(np.sum(abs(inc[~band])**2)/np.sum(abs(inc)**2)))


# The legacy packet weighting uses the measured ingoing component too.
n_packet=1<<(4*len(source_t)-1).bit_length()
wh_packet_parts=monitor_characteristics(source_t,wh_monitor_records[-65.]['total'],wh_monitor_records[-65.]['dpsi'],n_packet,.008)
wh_incident=np.exp(.008*bh.t)*np.fft.irfft(wh_packet_parts['ingoing_fft'],n_packet)[:len(bh.t)]


## 8. Quantum horizon: published global phase and computational interface

The OWA coefficient in Eqs. (14)-(15), or Appendix A (31)-(33), is the **global**
ratio $R_g=A_{out}/A_{in}$ in $A_{in}e^{-i\omega x}+A_{out}e^{i\omega x}$.
Its phase already includes the reflection depth. With the standard Schwarzschild
$x=r+2M\log(r/2M-1)$, the near-horizon lapse is $C e^{\kappa x}$,
$C=e^{-1/2}$. We include that factor in the published expression; the hierarchy
parameter is $E_{Pl}M/\gamma$, not an extra adjustable time shift.

At an arbitrary computational plane $x_H$ the local ratio is
$R_H=R_g e^{2i\omega x_H}$. Consequently
$$K_H=\frac{T_E R_H e^{i\omega(x_E-x_H)}}{1-R_{BH}R_H e^{-2i\omega x_H}}
=\frac{T_E R_g e^{i\omega(x_E+x_H)}}{1-R_{BH}R_g}.$$
The measured ingoing input changes by $e^{-i\omega\Delta x_H}$ when the plane
moves, cancelling the corresponding change in $K_H$. No physical reflecting
surface is moved in this test. Constant local reflectors are different models
and retain their physical surface-distance tests.

The computational plane is $x_H=-40M$, in the near-horizon overlap region, not
the frequency-dependent reflection depth. Its small residual potential and the
discarded negative-time part of the asymptotic boundary kernel are measured;
the independent time evolution must reproduce the *untruncated* spectral law
within the predeclared 1% raw-error bound. A second plane and grid refinement
test this boundary approximation. The source is measured as field and spatial
derivative in a separate, longer-domain absorbing evolution. The zero-frequency
ingoing split is not defined; its transfer-weighted contribution vanishes and
is explicitly set to zero. This does not alter the nonzero-frequency response.

References: [OWA, Eqs.14-16 and Appendix A](https://arxiv.org/html/1905.00464),
[Mark et al., transfer formalism](https://arxiv.org/abs/1706.06155).


In [ ]:
def quantum_reflection(w,kappa=.25,gamma=1.,EplM=1e8):
    # Published global A_out/A_in. Appendix A: C=exp(-1/2) in standard r*.
    EplM=EplM*np.exp(-.5)
    w=np.asarray(w,float); out=np.ones(w.shape,complex); pos=abs(w)>1e-14; q=abs(w[pos])
    logA=1j*q/kappa*np.log(gamma*q/(4*kappa*EplM))+loggamma(-2j*q/kappa)-loggamma(-1j*q/kappa)-loggamma(1-1j*q/kappa)
    Rp=np.exp(-np.pi*q/kappa)*np.exp(-2j*np.imag(logA)); out[pos]=np.where(w[pos]>=0,Rp,np.conj(Rp)); return out


def quantum_local_reflection(w,surface,**kwargs):
    return quantum_reflection(w,**kwargs)*np.exp(2j*np.asarray(w)*surface)


def barrier_coefficients_vectorized(wgrid,ell=2,M=1.,xL=-80.,xR=200.,step=.02,extract=95.):
    w=np.asarray(wgrid,float)
    if np.any(w<=0): raise ValueError("scattering integration requires strictly positive omega")
    w2=w*w; n=int(np.ceil((xR-xL)/step)); xs=np.linspace(xR,xL,n+1); mids=.5*(xs[:-1]+xs[1:])
    idet=int(np.argmin(abs(xs-extract)))
    if abs(xs[idet]-extract)>1e-9: raise ValueError("Jost extraction plane must be a grid node")
    Vn=rw_potential(r_from_schwarzschild_rstar(xs,M),ell,M); Vm=rw_potential(r_from_schwarzschild_rstar(mids,M),ell,M)
    psi=np.exp(1j*w*xR); p=1j*w*psi
    for j in range(n):
        h=xs[j+1]-xs[j]; k1q=p; k1p=(Vn[j]-w2)*psi
        q2=psi+.5*h*k1q; p2=p+.5*h*k1p; k2q=p2; k2p=(Vm[j]-w2)*q2
        q3=psi+.5*h*k2q; p3=p+.5*h*k2p; k3q=p3; k3p=(Vm[j]-w2)*q3
        q4=psi+h*k3q; p4=p+h*k3p; k4q=p4; k4p=(Vn[j+1]-w2)*q4
        psi+=h*(k1q+2*k2q+2*k3q+k4q)/6; p+=h*(k1p+2*k2p+2*k3p+k4p)/6
        if j+1==idet: detector_factor=psi.copy()*np.exp(-1j*w*extract)
    A=.5*(psi+p/(1j*w))*np.exp(-1j*w*xL); B=.5*(psi-p/(1j*w))*np.exp(+1j*w*xL)
    return B/A,1/A,detector_factor/A


def half_barrier_coefficients_vectorized(wgrid,r0,ell=2,M=1.,xR=180.,step=.02):
    """Exterior Jost solution decomposed at the throat-centered x=0 basis."""
    w=np.asarray(wgrid,float); w2=w*w; rs0=float(schwarzschild_rstar(r0,M))
    n=int(np.ceil(xR/step)); xs=np.linspace(xR,0.,n+1); mids=.5*(xs[:-1]+xs[1:])
    Vn=rw_potential(r_from_schwarzschild_rstar(xs+rs0,M),ell,M)
    Vm=rw_potential(r_from_schwarzschild_rstar(mids+rs0,M),ell,M)
    psi=np.exp(1j*w*xR); p=1j*w*psi
    for j in range(n):
        h=xs[j+1]-xs[j]; k1q=p; k1p=(Vn[j]-w2)*psi
        q2=psi+.5*h*k1q; p2=p+.5*h*k1p; k2q=p2; k2p=(Vm[j]-w2)*q2
        q3=psi+.5*h*k2q; p3=p+.5*h*k2p; k3q=p3; k3p=(Vm[j]-w2)*q3
        q4=psi+h*k3q; p4=p+h*k3p; k4q=p4; k4p=(Vn[j+1]-w2)*q4
        psi+=h*(k1q+2*k2q+2*k3q+k4q)/6; p+=h*(k1p+2*k2p+2*k3p+k4p)/6
    A=.5*(psi+p/(1j*w)); B=.5*(psi-p/(1j*w))
    return B/A,1/A,psi,p


def interp_scattering(xnew,x,z,low_kind,ell=2,method="pchip"):
    """Magnitude/unwrapped-phase interpolation plus the RW low-frequency limit; never freezes a transfer function."""
    xnew=np.asarray(xnew,float); x=np.asarray(x,float); z=np.asarray(z,complex); out=np.zeros(xnew.shape,complex)
    mag=abs(z); phase=np.unwrap(np.angle(z)); slope=float(np.polyfit(x[:min(8,len(x))],phase[:min(8,len(x))],1)[0])
    low=xnew<x[0]; middle=(xnew>=x[0])&(xnew<=x[-1])
    if low_kind=="R": low_mag=1-(1-mag[0])*(np.maximum(xnew[low],0)/x[0])**(2*ell+2)
    elif low_kind=="T": low_mag=mag[0]*(np.maximum(xnew[low],0)/x[0])**(ell+1)
    elif low_kind=="T_E": low_mag=mag[0]*np.maximum(xnew[low],0)/x[0]  # finite detector, fixed outer truncation
    else: raise ValueError(low_kind)
    low_phase=phase[0]+slope*(xnew[low]-x[0]); out[low]=low_mag*np.exp(1j*low_phase)
    if method=="pchip":
        mm=PchipInterpolator(x,mag)(xnew[middle]); pp=PchipInterpolator(x,phase)(xnew[middle])
    elif method=="linear":
        mm=np.interp(xnew[middle],x,mag); pp=np.interp(xnew[middle],x,phase)
    else: raise ValueError(method)
    out[middle]=mm*np.exp(1j*pp); return out


# Logarithmic sampling resolves omega -> 0, where T~omega^(ell+1) and |R|->1.
wgrid=np.unique(np.r_[np.geomspace(1e-4,1e-2,36,endpoint=False),np.linspace(.01,1.5,420)])
xH=float(CFG.qh_surface_x); xE=float(bh.x_extract)
Rbh,Tbh,Tdet=barrier_coefficients_vectorized(wgrid,CFG.ell,CFG.M,xL=xH,extract=xE)
Rhalf,Thalf,half_jost_psi0,half_jost_dpsi0=half_barrier_coefficients_vectorized(wgrid,CFG.wormhole_r0,CFG.ell,CFG.M)
flux_curve=abs(Rbh)**2+abs(Tbh)**2-1; flux_residual=float(np.max(abs(flux_curve))); flux_residual_low=float(np.max(abs(flux_curve[wgrid<.01])))
N=len(bh.t); dt=float(np.mean(np.diff(bh.t))); nfft=1<<(int(4*N-1).bit_length())
L_surface_to_extraction=xE-xH; L_cavity_roundtrip=-2*xH


def surface_K(w,Rbar,Tout,Rwall,xH,xE):
    if xH>=0 or xE<=xH: raise ValueError("surface must lie left of reference plane and extraction")
    return np.exp(1j*w*(xE-xH))*Tout*Rwall/(1-Rbar*Rwall*np.exp(-2j*w*xH))


# Measure the QH input independently, including its spatial derivative.
qh_source_bh,qh_source_records=matched_source_control(0.,monitors=(xH,xH-10.),sample_t=source_t)
def qh_ingoing_spectrum(record,nf):
    omega=2*np.pi*np.fft.rfftfreq(nf,dt)
    field=np.fft.rfft(record['total'],nf); deriv=np.fft.rfft(record['dpsi'],nf)
    inc=np.zeros_like(field);inc[1:]=.5*(field[1:]+deriv[1:]/(1j*omega[1:]))
    return inc


def quantum_echo_from_coefficients(hierarchy,Rtab,Ttab,grid=wgrid,nfft_use=nfft,method="pchip",constant_R=None,*,incident=None,surface=None,detector=None):
    wfq=2*np.pi*np.fft.rfftfreq(nfft_use,dt); Rbi=interp_scattering(wfq,grid,Rtab,"R",CFG.ell,method); Tbi=interp_scattering(wfq,grid,Ttab,"T_E",CFG.ell,method)
    surface=xH if surface is None else float(surface)
    Rq=quantum_local_reflection(wfq,surface,kappa=1/(4*CFG.M),gamma=1.,EplM=hierarchy)
    if constant_R is not None: Rq=np.full_like(wfq,constant_R,dtype=complex)
    if incident is None:
        incident_fft=qh_ingoing_spectrum(qh_source_records[surface],nfft_use)
    else:
        incident=np.asarray(incident)
        if incident.ndim!=1 or len(incident)<N: raise ValueError("input record too short")
        incident_fft=np.fft.rfft(incident,nfft_use)
    surface=xH if surface is None else float(surface); detector=xE if detector is None else float(detector)
    # Input sampling dt is the common source/observer sampling interval.
    K=surface_K(wfq,Rbi,Tbi,Rq,surface,detector); K[wfq>grid[-1]]=0
    # NumPy's positive-frequency basis is exp(+i omega t), the conjugate of
    # the stated exp(-i omega t) physical convention.
    echo_spec=np.conj(K)*incident_fft
    return np.fft.irfft(echo_spec,nfft_use)[:N],Rq,K,wfq


def build_quantum_horizon(hierarchy=1e8):
    echo,Rq,K,wfq=quantum_echo_from_coefficients(hierarchy,Rbh,Tdet)
    wf=Waveform("owa_boltzmann_quantum_horizon",bh.t,bh.psi+echo,bh.x_extract,{"M":CFG.M,"ell":CFG.ell,"EplM":hierarchy,
        "surface_x":xH,"barrier_plane":0.,"extraction_x":xE,"surface_to_extraction":L_surface_to_extraction,"geometric_loop_length":L_cavity_roundtrip,"R_Q_basis":"published global OWA coefficient rephased locally; C=exp(-1/2)","barrier_flux_residual":flux_residual,"omega_min_integrated":float(wgrid[0]),
        "low_frequency_treatment":"|R|->1; linear continuation of finite-detector transmission below the first Jost sample (zero at DC); tested by cutoff sweep",
        "fft_convention":"echo_spec=conj(K_physical)*FFT(signal) because NumPy positive frequencies use exp(+i omega t)"})
    return wf,echo,Rq,K


qh,qh_echo,Rq,Kphys=build_quantum_horizon(1e8); wfft=2*np.pi*np.fft.rfftfreq(nfft,dt); TH=1/(8*np.pi*CFG.M)
quantum_identity=float(np.max(abs(abs(Rq[1:])**2-np.exp(-wfft[1:]/TH))))

# Phase and total-delay decomposition on the integrated scattering grid.
phase_R=np.unwrap(np.angle(Rbh)); phase_T=np.unwrap(np.angle(Tbh)); Rq_grid=quantum_local_reflection(wgrid,xH,EplM=1e8)
den=1-Rbh*Rq_grid*np.exp(-2j*wgrid*xH); Kfull=surface_K(wgrid,Rbh,Tdet,Rq_grid,xH,xE)
tau_prop=np.full_like(wgrid,L_surface_to_extraction); tau_T=np.gradient(np.unwrap(np.angle(Tdet)),wgrid,edge_order=2); tau_Q=np.gradient(np.unwrap(np.angle(Rq_grid)),wgrid,edge_order=2)
tau_den=-np.gradient(np.unwrap(np.angle(den)),wgrid,edge_order=2); tau_sum=tau_prop+tau_T+tau_Q+tau_den; tau_total=np.gradient(np.unwrap(np.angle(Kfull)),wgrid,edge_order=2)
reliable=(wgrid>=.003)&(wgrid<=1.45); phase_decomposition_residual=float(np.max(abs(tau_total[reliable]-tau_sum[reliable])))

# Controlled analytic causality and phase-sign test for F(t-x).
tc=np.arange(8192)*.05; t0c=80.; delay_c=35.; width_c=5.; omega_c=.37; phi_c=.41
Fc=np.exp(-.5*((tc-t0c)/width_c)**2)*np.exp(1j*omega_c*(tc-t0c)); wc_fft=2*np.pi*np.fft.fftfreq(len(tc),.05)
Fc_shifted=np.fft.ifft(np.fft.fft(Fc)*np.exp(-1j*wc_fft*delay_c)*np.exp(1j*phi_c))
Fc_expected=np.exp(-.5*((tc-(t0c+delay_c))/width_c)**2)*np.exp(1j*omega_c*(tc-(t0c+delay_c)))*np.exp(1j*phi_c)
causality_l2=float(np.linalg.norm(Fc_shifted-Fc_expected)/np.linalg.norm(Fc_expected)); causality_arrival_error=float(abs(tc[np.argmax(abs(Fc_shifted))]-(t0c+delay_c)))
phase_recovered=float(np.angle(np.vdot(Fc_expected*np.exp(-1j*phi_c),Fc_shifted))); causality_phase_error=float(abs(np.angle(np.exp(1j*(phase_recovered-phi_c)))))

# Independent full transfer-chain convention test with controlled complex
# barrier/throat coefficients.  This tests amplitude, phase and group delay,
# including the cavity denominator, rather than testing a pure delay only.
tchain=np.arange(16384)*.05; tchain0=250.; chain_width=28.; chain_wc=.37
chain_in=np.exp(-.5*((tchain-tchain0)/chain_width)**2)*np.cos(chain_wc*(tchain-tchain0))
wchain=2*np.pi*np.fft.rfftfreq(len(tchain),.05)
Rb_c=.23*np.exp(1j*(.17+2.3*wchain)); Tb_c=.71*np.exp(1j*(-.11+1.7*wchain)); Rq_c=.41*np.exp(1j*(.29+4.1*wchain)); Lc=13.
Kc=np.exp(1j*wchain*Lc)*Tb_c*Rq_c/(1-Rb_c*Rq_c)
chain_out=np.fft.irfft(np.conj(Kc)*np.fft.rfft(chain_in),len(tchain))
chain_phase=np.unwrap(np.angle(Kc)); chain_tau=float(np.interp(chain_wc,wchain,np.gradient(chain_phase,wchain,edge_order=2)))
chain_Kc=complex(np.interp(chain_wc,wchain,Kc.real)+1j*np.interp(chain_wc,wchain,Kc.imag)); zin=hilbert(chain_in); zout=hilbert(chain_out)
ichain=int(np.argmax(abs(zout))); chain_delay_measured=float(tchain[ichain]-tchain0); chain_amp_measured=float(abs(zout[ichain])/np.max(abs(zin)))
chain_phase_measured=float(np.angle(zout[ichain])); chain_phase_expected=float(np.angle(np.exp(1j*(-np.angle(chain_Kc)+chain_wc*chain_tau))))
chain_amplitude_error=float(abs(chain_amp_measured-abs(chain_Kc))/abs(chain_Kc)); chain_delay_error=float(abs(chain_delay_measured-chain_tau))
chain_phase_error=float(abs(np.angle(np.exp(1j*(chain_phase_measured-chain_phase_expected)))))
qh_transfer_chain_test=pd.DataFrame([{"expected_abs_K":abs(chain_Kc),"measured_amplitude_ratio":chain_amp_measured,"relative_amplitude_error":chain_amplitude_error,
    "expected_group_delay":chain_tau,"measured_delay":chain_delay_measured,"absolute_delay_error":chain_delay_error,
    "expected_output_phase":chain_phase_expected,"measured_output_phase":chain_phase_measured,"wrapped_phase_error":chain_phase_error,
    "applied_numpy_transfer":"conj(K_physical)"}])

# omega_min, phase interpolation, and inverse-FFT convergence of the isolated correction.
qh_low_rows=[]; late_q=(bh.t>=190)&(bh.t<=500); ref_norm=max(np.linalg.norm(qh_echo[late_q]),1e-300)
zref_low=hilbert(qh_echo)
for cutoff in [1e-2,5e-3,2.5e-3,1e-3]:
    keep=wgrid>=cutoff; e,*_=quantum_echo_from_coefficients(1e8,Rbh[keep],Tdet[keep],wgrid[keep])
    ze=hilbert(e); phase_mask=late_q&(abs(zref_low)>.02*np.max(abs(zref_low[late_q])))&(abs(ze)>.02*np.max(abs(ze[late_q])))
    phase_delta=np.angle(ze[phase_mask]*np.conj(zref_low[phase_mask])); phase_weight=abs(ze[phase_mask])*abs(zref_low[phase_mask])
    phase_rms=float(np.sqrt(np.sum(phase_weight*phase_delta**2)/np.sum(phase_weight))) if np.sum(phase_weight) else np.nan
    qh_low_rows.append({"omega_min":cutoff,"n_integrated_samples":int(np.count_nonzero(keep)),"n_samples_below_0.01":int(np.count_nonzero(keep&(wgrid<.01))),
                        "late_correction_relative_L2":float(np.linalg.norm(e[late_q]-qh_echo[late_q])/ref_norm),"late_weighted_phase_rms":phase_rms})
qh_lowfreq_df=pd.DataFrame(qh_low_rows)
qh_echo_linear,*_=quantum_echo_from_coefficients(1e8,Rbh,Tdet,method="linear")
phase_interpolation_error=float(np.linalg.norm(qh_echo_linear[late_q]-qh_echo[late_q])/ref_norm)
qh_echo_2n,*_=quantum_echo_from_coefficients(1e8,Rbh,Tdet,nfft_use=2*nfft)
time_reconstruction_error=float(np.linalg.norm(qh_echo_2n[late_q]-qh_echo[late_q])/ref_norm)
print("Barrier flux residual:",flux_residual,"low-frequency:",flux_residual_low,"Boltzmann identity:",quantum_identity)
print("Phase decomposition:",phase_decomposition_residual,"causality L2:",causality_l2,"phase sign:",causality_phase_error)
display(qh_transfer_chain_test); display(qh_lowfreq_df)

quantum_scan_rows=[]; quantum_scan_waveforms={}; prompt_q=(bh.t>=100)&(bh.t<190)
prompt_peak=max(np.max(abs(bh.psi[prompt_q])),1e-300); prompt_rms=max(np.sqrt(np.mean(bh.psi[prompt_q]**2)),1e-300)
for hierarchy in [1e6,1e8,1e10,1e12]:
    wf,echo,rq,k=build_quantum_horizon(hierarchy); quantum_scan_waveforms[hierarchy]=(wf,echo)
    rqg=quantum_local_reflection(wgrid,xH,EplM=hierarchy); kg=surface_K(wgrid,Rbh,Tdet,rqg,xH,xE)
    tau=np.gradient(np.unwrap(np.angle(kg)),wgrid,edge_order=2); tauq=np.gradient(np.unwrap(np.angle(rqg)),wgrid,edge_order=2)
    env_echo=abs(hilbert(echo)); pk,_=find_peaks(env_echo[late_q],prominence=.03*np.max(env_echo[late_q]),distance=max(1,int(20/dt))); nlate=int(len(pk))
    quantum_scan_rows.append({"EplM_over_gamma":hierarchy,"abs_R_at_0.37":float(np.interp(.37,wgrid,abs(rqg))),"arg_R_at_0.37":float(np.interp(.37,wgrid,np.unwrap(np.angle(rqg)))),
                              "tau_Q_at_0.37":float(np.interp(.37,wgrid,tauq)),"total_group_delay_at_0.37":float(np.interp(.37,wgrid,tau)),
                              "windows_M":"BH prompt [100,190); late [190,500]; peak ratios use BH prompt peak; RMS ratio uses BH prompt RMS",
                              "A_late_residual_over_BH_prompt":float(np.max(abs((wf.psi-bh.psi)[late_q]))/prompt_peak),
                              "A_late_QH_correction_over_BH_prompt":float(np.max(abs(echo[late_q]))/prompt_peak),
                              "A_late_total_over_BH_prompt":float(np.max(abs(wf.psi[late_q]))/prompt_peak),
                              "isolated_QH_correction_rms_over_prompt":float(np.sqrt(np.mean(echo[late_q]**2))/prompt_rms),
                              "late_correction_peak_count":nlate,"morphology":"recurrence candidate" if nlate>=3 else "diffuse/single correction"})
quantum_scan_df=pd.DataFrame(quantum_scan_rows); display(quantum_scan_df)

fig,ax=plt.subplots(2,4,figsize=(20,8))
ax[0,0].plot(wgrid,abs(Rbh)**2,label=r"$|R_{BH}|^2$"); ax[0,0].plot(wgrid,abs(Tbh)**2,label=r"$|T_{BH}|^2$"); ax[0,0].legend(); ax[0,0].set(xscale="log",xlim=(1e-4,1.5),title="Barrier flux",xlabel=r"$M\omega$")
for hierarchy in [1e6,1e8,1e10,1e12]:
    rqg=quantum_local_reflection(wgrid,xH,EplM=hierarchy); tq=np.gradient(np.unwrap(np.angle(rqg)),wgrid,edge_order=2)
    ax[0,1].plot(wgrid,abs(rqg),label=f"{hierarchy:.0e}"); ax[0,2].plot(wgrid,np.unwrap(np.angle(rqg)),label=f"{hierarchy:.0e}"); ax[0,3].plot(wgrid,tq,label=f"{hierarchy:.0e}")
ax[0,1].set(xlim=(0,1),title=r"$|R_Q|$",xlabel=r"$M\omega$"); ax[0,2].set(xlim=(0,1),title=r"$\arg R_Q$",xlabel=r"$M\omega$"); ax[0,2].legend(fontsize=7); ax[0,3].set(xlim=(.1,.8),title=r"$\tau_Q$",xlabel=r"$M\omega$")
ax[1,0].plot(wgrid,tau_total,label="direct total"); ax[1,0].plot(wgrid,tau_sum,"--",label="component sum"); ax[1,0].set(xlim=(.1,.8),title="Total phase delay",xlabel=r"$M\omega$"); ax[1,0].legend()
ax[1,1].plot(wgrid,tau_prop,label="propagation"); ax[1,1].plot(wgrid,tau_T,label="transmission"); ax[1,1].plot(wgrid,tau_Q,label="quantum reflection"); ax[1,1].plot(wgrid,tau_den,label="cavity denominator"); ax[1,1].set(xlim=(.1,.8),title="Delay decomposition",xlabel=r"$M\omega$"); ax[1,1].legend(fontsize=7)
ax[1,2].plot(qh.t,qh.psi,label="total"); ax[1,2].plot(qh.t,1e3*qh_echo,label=r"$10^3\times$ correction"); ax[1,2].set(xlim=(100,500),title="Quantum-boundary response",xlabel=r"$t/M$"); ax[1,2].legend()
ax[1,3].loglog(qh_lowfreq_df.omega_min,qh_lowfreq_df.late_correction_relative_L2,"o-"); ax[1,3].invert_xaxis(); ax[1,3].set(title=r"$\omega_{min}$ convergence",xlabel=r"$M\omega_{min}$",ylabel="late relative L2")
fig.tight_layout(); fig.savefig(ARTIFACT_DIR/"quantum_phase_hierarchy.png")

# These checks do not use the production waveform to set their expected delays.
def plane_reflector_tests():
    h=.05; t=np.arange(32768)*h; w=2*np.pi*np.fft.rfftfreq(len(t),h)
    src=np.exp(-.5*((t-28.)/1.8)**2); spec=np.fft.rfft(src)
    rows=[]
    for surface in (-80.,-60.):
        for r in (0.,.04,.08):
            k=surface_K(w,.5,.8,r,surface,95.)
            y=np.fft.irfft(np.conj(k)*spec,len(t))
            centers=28.+(95.-surface)+np.arange(3)*(-2*surface)
            expected=sum(.8*r*(.5*r)**j*np.exp(-.5*((t-time)/1.8)**2)
                         for j,time in enumerate(centers))
            first3=t<centers[-1]+12
            peaks=[np.flatnonzero(abs(t-v)<6)[np.argmax(abs(y[abs(t-v)<6]))] for v in centers]
            # Now keep the incident source at the barrier, NOT at the test surface.
            # Its arrival at the surface itself is delayed by -surface.
            src_at_wall=np.exp(-.5*((t-(28.-surface))/1.8)**2)
            y_fixed=np.fft.irfft(np.conj(k)*np.fft.rfft(src_at_wall),len(t))
            j0=np.argmax(abs(y_fixed)); fixed_delay=t[j0]-28. if r else np.nan
            rows.append(dict(xH=surface,R=r,first_delay=t[peaks[0]]-28. if r else np.nan,
                        first_delay_from_fixed_barrier_source=fixed_delay,
                        spacing=t[peaks[1]]-t[peaks[0]] if r else np.nan,first_height=y[peaks[0]],
                        max_error=float(np.max(abs(y[first3]-expected[first3])))))
    df=pd.DataFrame(rows)
    assert np.all(df.loc[df.R==0,'max_error']==0)
    nz=df[df.R>0]
    assert np.max(abs(nz.first_delay-(95-nz.xH)))<h and np.max(abs(nz.spacing+2*nz.xH))<h
    assert np.max(abs(nz.first_delay_from_fixed_barrier_source-(95.-2*nz.xH)))<h
    assert np.max(abs(nz.first_height/nz.R-.8))<1e-10 and nz.max_error.max()<1e-10
    return df


def independent_rw_td(h, surface, detector, t_end, reflection):
    """Centered diamond update (dt=dx); separate incoming and outgoing wall characteristics.

    q1[n] = in[n+1]+out[n-1] at a locally free left boundary. Update the returning
    wave with an actual causal history convolution, without K, R_BH or T_BH.
    The distant right boundary cannot return to the analysis window.
    """
    x=surface+h*np.arange(int(round((400.-surface)/h))+1)
    idx=int(round((detector-surface)/h)); assert abs(x[idx]-detector)<1e-9
    v=rw_potential(r_from_schwarzschild_rstar(x,CFG.M),CFG.ell,CFG.M)
    assert v[0]<1e-8  # explicit near-horizon overlap-region bound
    nt=int(np.ceil(t_end/h))+1; t=h*np.arange(nt)
    nker=1<<(max(32*nt,32768)-1).bit_length()
    wg=2*np.pi*np.fft.rfftfreq(nker,h)
    if reflection=='OWA':
        coeff=quantum_local_reflection(wg,surface,kappa=1/(4*CFG.M),EplM=1e8)
        kernel=np.fft.irfft(np.conj(coeff),nker)
        negative_fraction=float(np.linalg.norm(kernel[nker//2+1:])/np.linalg.norm(kernel))
        ir=kernel[:nt].copy()
    else:
        negative_fraction=0.; ir=np.array([float(reflection)])
    # Pair the reflecting and absorbing PDEs on identical grids; subtract only at readout.
    a=CFG.pulse_width; u=x-CFG.pulse_center; q0=np.exp(-.5*(u/a)**2)
    velocity=-u*q0/a**2; accel=((u/a)**2-1)*q0/a**2-v*q0
    q=np.stack([q0,q0]); prev=q-h*velocity+.5*h*h*accel
    nxt=np.empty_like(q); fac=1/(1+.5*h*h*v[1:-1])
    incoming=np.zeros(nt); outgoing=np.zeros(nt); record=np.empty(nt); record[0]=0.
    for j in range(1,nt):
        nxt[:,1:-1]=(q[:,:-2]+q[:,2:])*fac-prev[:,1:-1]
        nxt[:,-1]=q[:,-2]; nxt[0,0]=q[0,1]
        incoming[j]=q[1,1]-(outgoing[j-2] if j>=2 else 0.)
        count=min(j+1,len(ir)); outgoing[j]=np.dot(ir[:count],incoming[j-count+1:j+1][::-1])
        nxt[1,0]=incoming[j]+outgoing[j]
        record[j]=nxt[1,idx]-nxt[0,idx]
        prev,q,nxt=q,nxt,prev
    return t,record,negative_fraction


plane_tests=plane_reflector_tests(); display(plane_tests)
td_trials=[]; td_owa_fine=None
for law in (.08,'OWA'):
    if law=='OWA': freq=qh_echo
    else:
        freq,*_=quantum_echo_from_coefficients(1e8,Rbh,Tdet,constant_R=law)
    for h in (.10,.05):
        tt,delta_td,neg=independent_rw_td(h,xH,xE,bh.t[-1]+.1,law)
        yy=CubicSpline(tt,delta_td,extrapolate=False)(bh.t)
        test=(bh.t>=190)&(bh.t<=500)
        err=float(np.linalg.norm((freq-yy)[test])/np.linalg.norm(yy[test]))
        td_trials.append(dict(law=str(law),dx=h,xH=xH,xE=xE,window='190 <= t/M <= 500',
                              raw_relative_L2=err,negative_lag_kernel_L2_fraction=neg))
        if law=='OWA' and h==.05: td_owa_fine=yy
qh_td_tests=pd.DataFrame(td_trials); display(qh_td_tests)
plane_tests.to_csv(ARTIFACT_DIR/'qh_coordinate_tests.csv',index=False)
qh_td_tests.to_csv(ARTIFACT_DIR/'qh_independent_td_checks.csv',index=False)

# Re-record the ingoing field at the moved surface. The source and detector stay fixed.
surface2=xH-10.; alt_bh,alt_incident,*_=run_black_hole(cfg=replace(CFG,qh_surface_x=surface2))
assert np.array_equal(alt_bh.t,bh.t)
r2,_,t2det=barrier_coefficients_vectorized(wgrid,CFG.ell,CFG.M,xL=surface2,extract=xE)
f2,*_=quantum_echo_from_coefficients(1e8,r2,t2det,constant_R=.08,incident=alt_incident,surface=surface2)
tc2,y2,_=independent_rw_td(.05,surface2,xE,bh.t[-1]+.1,.08)
yy2=CubicSpline(tc2,y2)(bh.t); test2=(bh.t>=190)&(bh.t<=500)
alt_error=float(np.linalg.norm((yy2-f2)[test2])/np.linalg.norm(yy2[test2]))
delta0,*_=quantum_echo_from_coefficients(1e8,r2,t2det,constant_R=0.,incident=alt_incident,surface=surface2)
tz,ztd,_=independent_rw_td(.10,surface2,xE,bh.t[-1]+.1,0.)
zero_error=max(np.max(abs(delta0)),np.max(abs(ztd)))
plane_tests["source_record"]="unit Gaussian peak; free analytic check; delays measured from packet center t=28"
plane_tests.to_csv(ARTIFACT_DIR/"qh_coordinate_tests.csv",index=False)
pd.DataFrame([dict(surface=surface2,detector=xE,R=.08,dx=.05,raw_relative_L2=alt_error,
    zero_wall_max_abs=zero_error,normalization="190<=t/M<=500; direct correction L2; no alignment")]).to_csv(ARTIFACT_DIR/"qh_shifted_surface_pde.csv",index=False)


# Hold the physical OWA object fixed; only change computational plane.
qh_plane_rows=[]
qh_analytic_grid=wgrid[wgrid>=.01]
for plane in (xH,xH-10.):
    rb,_,te=barrier_coefficients_vectorized(wgrid,CFG.ell,CFG.M,xL=plane,extract=xE)
    other,*_=quantum_echo_from_coefficients(1e8,rb,te,surface=plane)
    mask=(bh.t>=190)&(bh.t<=500)
    change=float(np.linalg.norm((other-qh_echo)[mask])/np.linalg.norm(qh_echo[mask]))
    if plane==xH:
        direct=td_owa_fine;neg=float(qh_td_tests.query("law=='OWA' and dx==.05").negative_lag_kernel_L2_fraction.iloc[0])
    else:
        tcheck,ycheck,neg=independent_rw_td(.05,plane,xE,bh.t[-1]+.1,'OWA')
        direct=CubicSpline(tcheck,ycheck)(bh.t)
    error=float(np.linalg.norm((other-direct)[mask])/np.linalg.norm(direct[mask]))
    # Independent algebraic test at each frequency, using a globally fixed
    # ingoing amplitude A_in=1: I_H=exp(-i*w*xH).
    rg=quantum_reflection(wgrid,EplM=1e8)
    local=quantum_local_reflection(wgrid,plane,EplM=1e8)
    via_local=surface_K(wgrid,rb,te,local,plane,xE)*np.exp(-1j*wgrid*plane)
    global_result=te*rg*np.exp(1j*wgrid*xE)/(1-rb*rg)
    algebra=float(np.max(abs(via_local-global_result)/(1+abs(global_result))))
    qh_plane_rows.append(dict(computational_plane=plane,physical_hierarchy=1e8,
        raw_waveform_change=change,raw_TD_FD_error=error,global_local_identity_error=algebra,
        negative_lag_kernel_fraction=neg,
        potential_at_interface=float(rw_potential(r_from_schwarzschild_rstar(plane,CFG.M),CFG.ell,CFG.M))))
qh_plane_df=pd.DataFrame(qh_plane_rows)
assert qh_plane_df.raw_waveform_change.max()<1e-4,qh_plane_df
assert qh_plane_df.raw_TD_FD_error.max()<.01,qh_plane_df
assert qh_plane_df.global_local_identity_error.max()<1e-12,qh_plane_df
assert qh_plane_df.negative_lag_kernel_fraction.max()<.001,qh_plane_df
qh_plane_df.to_csv(ARTIFACT_DIR/'qh_fixed_physics_interface_invariance.csv',index=False)
np.savez_compressed(ARTIFACT_DIR/'qh_measured_source.npz',source_t=source_t,fft_n=nfft,
    monitor_field=qh_source_records[xH]['total'],monitor_derivative=qh_source_records[xH]['dpsi'],
    ingoing_fft=qh_ingoing_spectrum(qh_source_records[xH],nfft),omega=wfft,
    global_R=quantum_reflection(wfft,EplM=1e8),local_R=Rq,K=Kphys,
    surface=xH,detector=xE,prediction=qh_echo,time=bh.t)
display(qh_plane_df)


## 9. Waveform, phase-delay, and echo-time uncertainty diagnostics

The wormhole delay is evaluated across the actual finite packet bandwidth rather than at a nominal ray frequency. Echo times are estimated independently from matched-packet peaks, envelope peaks, autocorrelation, and the power-weighted Wigner delay. The quoted uncertainty contains extraction-radius, resolution, window, bandwidth, and method components. It is a numerical/operational uncertainty for this initial-data experiment, not a statistical detector posterior.


The wormhole spectrum below is found by complex outgoing-Jost matching, **not**
by interpreting real-axis phase crossings as exact modes. For reflection-even
axial modes $\Psi_x(0^+)=\Lambda\Psi(0)/2$; reflection-odd axial modes have
$\Psi(0)=0$. These are spatial reflection families, not axial/polar gravitational
parity. The selected $n=0,1$ branches in each family are continued in throat
radius, retaining actual modes with $0<M\Re\omega<0.55$; this is not a
completeness claim. All continued branches and exclusions are saved. Exterior
complex-contour, boundary-distance, expansion-order and tolerance
tests accompany the roots. [Yang, Xia & Miao, Table 6](https://doi.org/10.1140/epjc/s10052-025-14446-w)
provides independent published frequencies in units $2M\omega$.


In [ ]:
"""True complex-frequency axial WH poles; not real-axis phase crossings.

Insert definitions before section 9; refine phase seeds with
complex_modes_from_estimates(), then call run_wh_pole_audit() on the refined
production map. Reflection parity is x -> -x, not axial/polar
gravitational parity. Both families here are axial gravitational perturbations.
"""
import numpy as np
import pandas as pd
from scipy.integrate import solve_ivp
from scipy.optimize import root

def wh_outgoing_cauchy(w,a,ell=2,M=1.,length=90.,angle=.65,order=16,rtol=2e-11):
    """Outgoing Jost data at the throat by analytic exterior contour shooting.

    Psi=e^(i*w*rstar)*u and r=a+s exp(i*angle). Factoring the outgoing
    exponential removes QNM growth along the real radius. The ray stays in
    Re(r)>2M and Im(w exp(i angle))>0; the outgoing asymptotic series then
    specifies the decaying branch without oscillatory far-boundary fitting.
    Returns data up to a common, irrelevant e^(i*w*rstar(a)) multiplier.
    """
    if a<=2*M or w.real<=0 or w.imag>=0: raise ValueError('decaying positive-frequency WH pole required')
    direction=np.exp(1j*angle)
    if (w*direction).imag<=0: raise ValueError('rotate contour farther into outgoing decay sector')
    R=a+length*direction; L=ell*(ell+1)
    c=np.zeros(order+1,complex);c[0]=1.
    for n in range(1,order+1):
        v=((n-1)*n-L)*c[n-1]
        if n>=2:v+=M*(-2*(n-2)*n+6)*c[n-2]
        c[n]=v/(2j*w*n)
    n=np.arange(order+1); u=np.sum(c*R**(-n));du=np.sum(-n*c*R**(-n-1))
    def rhs(s,y):
        r=a+s*direction;f=1-2*M/r;fp=2*M/r**2;V=f*(L/r**2-6*M/r**3)
        return direction*np.array([y[1],(V*y[0]-(f*fp+2j*w*f)*y[1])/f**2])
    sol=solve_ivp(rhs,(length,0.),[u,du],method='DOP853',rtol=rtol,atol=rtol*.01)
    if not sol.success:raise RuntimeError(sol.message)
    u,du=sol.y[:,-1];f=1-2*M/a
    return u,f*du+1j*w*u

def wh_pole_residual(w,a,parity,ell=2,M=1.,**numeric):
    u,p=wh_outgoing_cauchy(w,a,ell,M,**numeric)
    # Integrating the actual positive delta jump gives 2 Psi'(0+)=Lambda Psi(0)
    # for even reflection parity. Odd reflection parity gives Psi(0)=0.
    lam=2*(a-M)/a**2
    value=(p-.5*lam*u) if parity=='even' else u
    if parity not in ('even','odd'):raise ValueError('reflection parity must be even or odd')
    return value/(abs(u)+abs(p)+1e-300)

def wh_complex_pole(a,guess,parity,ell=2,M=1.,**numeric):
    def fun(v):
        w=v[0]+1j*v[1]
        try:r=wh_pole_residual(w,a,parity,ell,M,**numeric)
        except ValueError:return [10+abs(v[0]),10+abs(v[1])]
        return [r.real,r.imag]
    sol=root(fun,[complex(guess).real,complex(guess).imag],tol=2e-10)
    w=complex(*sol.x);res=abs(wh_pole_residual(w,a,parity,ell,M,**numeric))
    # MINPACK can report stagnation after reaching the ODE accuracy floor.
    # Accept only by a finite, independently reevaluated residual, not status.
    if not np.isfinite(w.real+w.imag+res) or res>2e-8:
        raise RuntimeError(f'WH pole not converged: {sol.message}; residual={res:g}')
    return w,res

_WH_POLE_TRACK_CACHE={}
_WH_POLE_TRACK_LOG=[]

def wh_continued_pole(r0,parity,overtone,ell=2,M=1.):
    """Track an identified near-horizon pole in log(r0/M-2), without relabeling.

    The four numerical anchor guesses only initialize genuine complex roots at
    r0/M=2.0005; they are not tabulated answers. All roots, including anchors,
    solve the outgoing ODE and exact delta matching. At each subsequent step
    a small displacement and the same stringent residual are required. A
    rejected step is logged and retried at half size, never silently dropped.
    """
    if ell!=2 or parity not in ('even','odd') or overtone not in (0,1):
        raise ValueError('validated continuation map is ell=2, reflection even/odd, overtone n=0,1')
    if r0<=2*M:raise ValueError('wormhole throat must exceed 2M')
    seeds={'even':(.187-.000007j,.347-.00325j),'odd':(.228-.000048j,.389-.01054j)}
    key=(float(M),int(ell),parity,int(overtone));target=float(np.log(r0/M-2.))
    cache=_WH_POLE_TRACK_CACHE.setdefault(key,{})
    if not cache:
        anchor=np.log(.0005);w,res=wh_complex_pole(2.0005*M,seeds[parity][overtone]/M,parity,ell,M)
        cache[anchor]=(w,res)
        _WH_POLE_TRACK_LOG.append(dict(r0_over_M=2.0005,parity=parity,overtone=overtone,
            log_throat_step=0.,omega_R=w.real,omega_I=w.imag,residual=res,accepted=True,reason='independent outgoing anchor'))
    start=min(cache,key=lambda t:abs(t-target));w,res=cache[start]
    step=min(.2,abs(target-start));direction=np.sign(target-start)
    while abs(target-start)>1e-11:
        trial=start+direction*min(step,abs(target-start));a=M*(2+np.exp(trial))
        try:
            candidate,rr=wh_complex_pole(a,w,parity,ell,M,angle=.8)
            if abs(candidate-w)>.15*abs(w):
                raise RuntimeError('root displacement too large to identify the continued branch')
        except (ValueError,RuntimeError) as exc:
            _WH_POLE_TRACK_LOG.append(dict(r0_over_M=a/M,parity=parity,overtone=overtone,
                log_throat_step=trial-start,omega_R=np.nan,omega_I=np.nan,residual=np.nan,
                accepted=False,reason=str(exc)))
            step*=.5
            if step<1e-5:raise RuntimeError(f'WH continuation failed for {parity} n={overtone}, r0/M={a/M:g}') from exc
            continue
        _WH_POLE_TRACK_LOG.append(dict(r0_over_M=a/M,parity=parity,overtone=overtone,
            log_throat_step=trial-start,omega_R=candidate.real,omega_I=candidate.imag,residual=rr,
            accepted=True,reason='outgoing zero; continuously tracked branch'))
        start=trial;w,res=candidate,rr;cache[start]=(w,res);step=min(.2,step*1.5)
    return w,res

def complex_modes_from_estimates(phase_estimates,r0,ell=2,M=1.,max_seed_Momega=.55,artifact_dir=None):
    """True n=0,1 poles in each reflection family, followed continuously in r0.

    Legacy argument max_seed_Momega now bounds the *actual* M Re(omega), not
    approximate seeds. The map selects n=0,1 and M Re(omega)<0.55 and does not
    claim completeness. Phase estimates are retained only as diagnostics;
    their poor damping approximations must not choose or relabel a branch.
    All four branches are computed; excluded poles remain in the audit CSV.
    A family with fewer than two in-band poles has no same-family spacing.
    """
    from pathlib import Path
    rows=[]
    for family in ('even','odd'):
        seeds=phase_estimates[phase_estimates.parity==family].sort_values('omega_R')
        for n in (0,1):
            w,res=wh_continued_pole(r0,family,n,ell,M)
            guess=complex(seeds.iloc[n].omega_R,seeds.iloc[n].omega_I) if len(seeds)>n else complex(np.nan,np.nan)
            if any(r['parity']==family and abs(w-complex(r['omega_R'],r['omega_I']))*M<1e-6 for r in rows):
                raise RuntimeError(f'duplicate WH branch for {family} at r0={r0:g}')
            rows.append(dict(r0_over_M=r0/M,parity=family,overtone=n,omega_R=w.real,omega_I=w.imag,residual=res,
                phase_estimate_omega_R=guess.real,phase_estimate_omega_I=guess.imag,
                phase_estimate_relative_error=abs(guess-w)/abs(w),
                method='complex outgoing Jost zero; log-throat branch continuation',
                selected_for_spacing=bool(M*w.real<max_seed_Momega),max_actual_Momega=max_seed_Momega,
                map_scope='selected n=0,1 branches of each reflection parity; not exhaustive spectrum',
                qnm_condition="outgoing at both infinities; even Psi'=Lambda Psi/2, odd Psi=0"))
    all_modes=pd.DataFrame(rows).sort_values('omega_R').reset_index(drop=True)
    modes=all_modes[all_modes.selected_for_spacing].copy().reset_index(drop=True)
    modes['Delta_omega']=modes.groupby('parity').omega_R.diff()
    out=artifact_dir if artifact_dir is not None else globals().get('ARTIFACT_DIR')
    if out is not None:
        out=Path(out);out.mkdir(parents=True,exist_ok=True)
        all_modes.to_csv(out/f'wormhole_complex_branches_r0_{r0/M:.7f}.csv',index=False)
        pd.DataFrame(_WH_POLE_TRACK_LOG).to_csv(out/'wormhole_complex_pole_continuation.csv',index=False)
    return modes

def run_wh_pole_audit(modes,artifact_dir,r0=2.0005,M=1.,ell=2):
    """Published Table 6 comparison and complex poles at the production throat.

    Table 6: Yang, Xia & Miao, EPJC85 742 (2025), primary published Prony
    estimates, units abar=a/(2M), 2M*omega. A numerical comparison tolerance
    of 1% is set before running; published Prony digits are not treated as
    exact analytic constants. Complex-pole numerics converge much more tightly.
    """
    from pathlib import Path
    out=Path(artifact_dir);out.mkdir(parents=True,exist_ok=True)
    published=[(1.06,.782781-.039926j),(1.1,.825691-.072068j),(1.2,.862155-.137343j)]
    literature=[];convergence=[]
    for abar,twomw in published:
        a=2*M*abar;w,res=wh_complex_pole(a,twomw/(2*M),'even',ell,M)
        variants=[dict(length=70.),dict(length=120.),dict(angle=.8),dict(order=20),dict(rtol=4e-12)]
        for setting in variants:
            ww,rr=wh_complex_pole(a,w,'even',ell,M,**setting)
            convergence.append(dict(a_over_2M=abar,parity='even',setting=str(setting),omega_R=ww.real,
                omega_I=ww.imag,relative_change=abs(ww-w)/abs(w),residual=rr))
        error=abs(2*M*w-twomw)/abs(twomw)
        real_error=abs(2*M*w.real-twomw.real)/abs(twomw.real)
        damping_error=abs(2*M*w.imag-twomw.imag)/abs(twomw.imag)
        literature.append(dict(model='Schwarzschild thin-shell axial wormhole',a_over_2M=abar,
            reflection_parity='even',published_2Momega_R=twomw.real,published_2Momega_I=twomw.imag,
            numerical_2Momega_R=2*M*w.real,numerical_2Momega_I=2*M*w.imag,
            relative_complex_error=error,relative_real_error=real_error,relative_damping_error=damping_error,
            required_less_than=.01,passed=max(error,real_error,damping_error)<.01,
            source='Yang Xia Miao 2025 Table 6 (Prony estimate)',source_url='https://link.springer.com/article/10.1140/epjc/s10052-025-14446-w/tables/6'))
    ldf=pd.DataFrame(literature);cdf=pd.DataFrame(convergence)
    ldf.to_csv(out/'wormhole_published_QNM_benchmark.csv',index=False)
    if 'method' not in modes or not modes.method.str.startswith('complex outgoing Jost zero').all():
        raise ValueError('run_wh_pole_audit requires true complex modes, not phase estimates')
    for row in modes.itertuples():
        w=complex(row.omega_R,row.omega_I);family=row.parity
        if abs(wh_pole_residual(w,r0,family,ell,M))>=2e-8:
            raise RuntimeError('production frequency is not a converged outgoing WH pole')
        for setting in (dict(length=120.),dict(angle=.8),dict(order=20),dict(rtol=4e-12)):
            ww,rr=wh_complex_pole(r0,w,family,ell,M,**setting)
            cdf.loc[len(cdf)]=dict(a_over_2M=r0/(2*M),parity=family,setting=str(setting),omega_R=ww.real,
                omega_I=ww.imag,relative_change=abs(ww-w)/abs(w),residual=rr)
    modes.to_csv(out/'wormhole_complex_poles.csv',index=False);cdf.to_csv(out/'wormhole_complex_pole_convergence.csv',index=False)
    assert ldf.passed.all(),ldf
    assert cdf.relative_change.max()<2e-6,cdf
    assert all((modes.parity==family).sum()>=2 for family in ('even','odd')),modes
    return modes,ldf,cdf



waveforms={"BH":bh,"GS":gs,"BS":bs,"WH":wh,"QH":qh}; colors=dict(BH="black",GS="tab:blue",BS="tab:green",WH="tab:orange",QH="tab:red")

# The incident packet spectrum is measured from the independent BH-horizon
# record, not from a hand-picked wormhole echo.
m0=(bh.t>=55)&(bh.t<=280); dt0=float(np.mean(np.diff(bh.t[m0])))
from scipy.signal.windows import tukey
wpacket=2*np.pi*np.fft.rfftfreq(np.sum(m0),dt0); Ppacket=abs(np.fft.rfft((wh_incident[m0]-np.mean(wh_incident[m0]))*tukey(np.sum(m0),.2)))**2
bandp=(wpacket>=.15)&(wpacket<=.9); wc=float(np.sum(wpacket[bandp]*Ppacket[bandp])/np.sum(Ppacket[bandp]));
packet_bandwidth=float(np.sqrt(np.sum((wpacket[bandp]-wc)**2*Ppacket[bandp])/np.sum(Ppacket[bandp])))
Pw=np.interp(wgrid,wpacket[bandp],Ppacket[bandp],left=0,right=0)


def wormhole_transfer_chain(r0,Rc,Tc):
    Lambda=2*(r0-CFG.M)/r0**2; rd,td=throat_delta_coefficients(wgrid,Lambda)
    # Effective reflection seen from one exterior: direct throat reflection
    # plus transmission to, and all returns from, the opposite exterior.
    Rth=rd+td**2*Rc/(1-rd*Rc); loop=Rc*Rth; den=1-loop
    return {"Lambda":Lambda,"r_delta":rd,"t_delta":td,"R_throat_effective":Rth,"loop":loop,"denominator":den,
            "even_loop":Rc*(rd+td),"odd_loop":Rc*(rd-td)}


def roots_from_phase(loop,label):
    phase=np.unwrap(np.angle(loop)); tau=np.gradient(phase,wgrid,edge_order=2); band=(wgrid>=.06)&(wgrid<=1.05)&(tau>0)
    roots=[]
    for i in np.flatnonzero(band[:-1]&band[1:]):
        lo,hi=phase[i]/(2*np.pi),phase[i+1]/(2*np.pi)
        for n in range(int(np.ceil(min(lo,hi))),int(np.floor(max(lo,hi)))+1):
            level=2*np.pi*n
            if (phase[i]-level)*(phase[i+1]-level)<=0 and phase[i+1]!=phase[i]:
                root=wgrid[i]+(wgrid[i+1]-wgrid[i])*(level-phase[i])/(phase[i+1]-phase[i])
                tr=float(np.interp(root,wgrid,tau)); mag=float(np.interp(root,wgrid,abs(loop)))
                if tr>0: roots.append({"parity":label,"omega_R":root,"omega_I":np.log(max(mag,1e-300))/tr,"phase_level":level,"group_time":tr})
    if not roots: return pd.DataFrame(columns=["parity","omega_R","omega_I","phase_level","group_time"])
    return pd.DataFrame(roots).drop_duplicates(subset=["parity","phase_level"]).sort_values("omega_R")


def wormhole_qnm_map_for_r0(r0,Rc=None,Tc=None):
    if Rc is None or Tc is None: Rc,Tc,_,_=half_barrier_coefficients_vectorized(wgrid,r0,CFG.ell,CFG.M)
    chain=wormhole_transfer_chain(r0,Rc,Tc)
    table=pd.concat([roots_from_phase(chain["even_loop"],"even"),roots_from_phase(chain["odd_loop"],"odd")],ignore_index=True).sort_values("omega_R").reset_index(drop=True)
    table=complex_modes_from_estimates(table,r0,CFG.ell,CFG.M,artifact_dir=ARTIFACT_DIR)
    return table,chain


wormhole_qnm_map,wh_chain=wormhole_qnm_map_for_r0(CFG.wormhole_r0,Rhalf,Thalf)
wormhole_qnm_map,wh_literature_df,wh_pole_convergence_df=run_wh_pole_audit(
    wormhole_qnm_map,ARTIFACT_DIR,r0=CFG.wormhole_r0,M=CFG.M,ell=CFG.ell)
display(wh_literature_df)
phi_wh=np.unwrap(np.angle(wh_chain["loop"])); tau_w=np.gradient(phi_wh,wgrid,edge_order=2)
relevant=(wgrid>=.15)&(wgrid<=.9)&(Pw>0)&np.isfinite(tau_w)
phase_weight=Pw*abs(Thalf**2*wh_chain["R_throat_effective"])**2
tau_weighted=float(np.sum(tau_w[relevant]*phase_weight[relevant])/np.sum(phase_weight[relevant])); Tweighted=tau_weighted
Tcarrier=float(np.interp(wc,wgrid,tau_w)); Tgeom=float(wh.metadata["geometric_roundtrip"])
delay_distribution=tau_w[relevant]; bandwidth_delay_sigma=float(np.sqrt(np.sum(phase_weight[relevant]*(delay_distribution-Tweighted)**2)/np.sum(phase_weight[relevant])))
tau_first=np.gradient(np.unwrap(np.angle(Thalf**2*wh_chain["r_delta"])),wgrid,edge_order=2)
Tfirstweighted=float(np.sum(tau_first[relevant]*phase_weight[relevant])/np.sum(phase_weight[relevant]))


def pulse_train_quality(t,y,packet_table):
    """Matched-packet recurrence plus envelope trough isolation; envelope peaks alone never decide."""
    if len(packet_table)<3 or packet_table.delta_t.dropna().size<2:
        return {"separated_pulses":False,"stable_spacing":False,"spacing_cv":np.nan,"median_trough_ratio":np.nan,"n_packets":len(packet_table)}
    times=packet_table.time.to_numpy(); d=np.diff(times); env=abs(hilbert(y)); trough=[]
    for a,b in zip(times[:-1],times[1:]):
        m=(t>a)&(t<b)
        if np.any(m):
            ea=float(np.interp(a,t,env)); eb=float(np.interp(b,t,env)); trough.append(float(np.min(env[m])/max(min(ea,eb),1e-300)))
    cv=float(np.std(d,ddof=1)/np.mean(d)) if len(d)>1 else np.inf; tr=float(np.median(trough)) if trough else np.inf
    return {"separated_pulses":bool(tr<.45),"stable_spacing":bool(cv<.15),"spacing_cv":cv,"median_trough_ratio":tr,"n_packets":len(packet_table)}


prompt_est=abs(CFG.pulse_center-wh.metadata["x_peak"])+abs(CFG.extraction_x-wh.metadata["x_peak"])
first_throat_est=prompt_est+2*wh.metadata["x_peak"]
template_window=(first_throat_est-12,first_throat_est+22); min_packet_sep=max(12.,.42*abs(Tweighted))
Tacf,ac_strength=recurrence_delay(wh.t,wh.psi,template_window[0],abs(Tweighted),.38)
wh_packets=matched_packet_observables(wh.t,wh.psi,template_window,min_packet_sep,template_window[0])
Tmatched=float(np.median(wh_packets.delta_t.dropna())) if wh_packets.delta_t.notna().any() else np.nan
Tenvelope=envelope_packet_delay(wh.t,wh.psi,template_window[0],abs(Tweighted))

# Extraction-radius contribution: packet spacing is invariant under the retarded-time shift.
extract_delays=[]
for xe in [75.,95.,120.]:
    shift=xe-95.; win=(template_window[0]+shift,template_window[1]+shift)
    q=matched_packet_observables(wh.t,wh_records[f"right{xe:g}"],win,min_packet_sep,win[0])
    if q.delta_t.notna().any(): extract_delays.append(float(np.median(q.delta_t.dropna())))
sigma_extract=.5*(max(extract_delays)-min(extract_delays)) if len(extract_delays)>1 else 0.

# Required three-level resolution check, plus all three extraction radii.
wh_coarse,_,_,_,wh_coarse_records=run_wormhole(dx=.20,t_max=520,extraction_xs=(75.,95.,120.))
wh_fine,_,_,_,wh_fine_records=run_wormhole(dx=.05,t_max=520,extraction_xs=(75.,95.,120.))
# Every resolution is interrogated on exactly the same sampling support.
# Native waveforms are interpolated without fitted shifts.
# Use the coarsest recorded sampling interval (.14 M at the default settings).
timing_sample_dt=max(float(np.median(np.diff(w.t))) for w in (wh_coarse,wh,wh_fine))
timing_common_end=min(520.,*(float(w.t[-1]) for w in (wh_coarse,wh,wh_fine)))
timing_common_t=timing_sample_dt*np.arange(int(np.floor(timing_common_end/timing_sample_dt))+1)
resolution_rows=[]
for dxx,wfx,recs in [(.20,wh_coarse,wh_coarse_records),(.10,wh,wh_records),(.05,wh_fine,wh_fine_records)]:
    timing_signal=CubicSpline(wfx.t,wfx.psi,extrapolate=False)(timing_common_t)
    assert np.isfinite(timing_signal).all()
    q=matched_packet_observables(timing_common_t,timing_signal,template_window,min_packet_sep,template_window[0])
    quality=pulse_train_quality(timing_common_t,timing_signal,q)
    spacing=float(np.median(q.delta_t.dropna())) if q.delta_t.notna().any() else np.nan
    ex=[]
    for xe in [75.,95.,120.]:
        shift=xe-95.; win=(template_window[0]+shift,template_window[1]+shift)
        record=CubicSpline(wfx.t,recs[f"right{xe:g}"],extrapolate=False)(timing_common_t)
        qq=matched_packet_observables(timing_common_t,record,win,min_packet_sep,win[0])
        if qq.delta_t.notna().any(): ex.append(float(np.median(qq.delta_t.dropna())))
    resolution_rows.append({"dx":dxx,"record_requested_end":520.,"analysis_end":float(timing_common_t[-1]),
                            "common_sample_dt":timing_sample_dt,"T_recurrence":spacing,"n_packets":quality["n_packets"],"spacing_cv":quality["spacing_cv"],
                            "separated_pulses":quality["separated_pulses"],"stable_spacing":quality["stable_spacing"],
                            "extraction_spacing_spread":max(ex)-min(ex) if len(ex)>1 else np.nan})
wormhole_pulse_convergence_df=pd.DataFrame(resolution_rows)
finite_resolution=wormhole_pulse_convergence_df.T_recurrence.dropna().to_numpy()
wormhole_spacing_resolution_fraction=float(np.ptp(finite_resolution)/np.median(finite_resolution)) if len(finite_resolution)>=2 else np.inf
wormhole_extraction_spacing_fraction=float(np.nanmax(wormhole_pulse_convergence_df.extraction_spacing_spread)/np.nanmedian(finite_resolution)) if len(finite_resolution) else np.inf
# Discrete packet-index maxima do not inherit the PDE stencil's fourth
# order. Quote the full observed grid spread with a one-sample timing floor:
# two peak locations, each quantized by at most dt/2, give a spacing error dt.
sigma_resolution=max(float(np.ptp(finite_resolution)),timing_sample_dt) if len(finite_resolution)>=2 else np.nan
window_delays=[]
for shift in [-6.,0.,6.]:
    win=(template_window[0]+shift,template_window[1]+shift); q=matched_packet_observables(wh.t,wh.psi,win,min_packet_sep,win[0])
    if q.delta_t.notna().any(): window_delays.append(float(np.median(q.delta_t.dropna())))
sigma_window=float(np.std(window_delays,ddof=1)) if len(window_delays)>1 else 0.
method_values=np.asarray([Tmatched,Tenvelope,Tacf,Tweighted],float); method_values=method_values[np.isfinite(method_values)]
sigma_method=float(np.std(method_values,ddof=1)) if len(method_values)>1 else 0.
echo_uncertainty_components=pd.DataFrame([
    {"source":"extraction radius","delta_T":sigma_extract},{"source":"resolution (grid spread, sampling floor)","delta_T":sigma_resolution},
    {"source":"template window","delta_T":sigma_window},{"source":"finite packet bandwidth","delta_T":bandwidth_delay_sigma},
    {"source":"peak/delay method","delta_T":sigma_method}])
T_echo_uncertainty=float(np.sqrt(np.sum(echo_uncertainty_components.delta_T**2))); T_echo_report=Tmatched; wormhole_delay_error=abs(Tmatched-Tweighted)/abs(Tweighted)
display(wh_packets.head(8)); display(wormhole_pulse_convergence_df); display(echo_uncertainty_components)
print(f"Wormhole carrier={wc:.4f}, bandwidth={packet_bandwidth:.4f}, smooth geometric={Tgeom:.3f}, full-chain carrier={Tcarrier:.3f}, weighted={Tweighted:.3f}")
print(f"First throat-reflection group delay={Tfirstweighted:.3f}; recurrence={T_echo_report:.2f} +/- {T_echo_uncertainty:.2f} M; autocorrelation={Tacf:.2f}, envelope={Tenvelope:.2f}")

# Quantum correction timing uses the correction, never the prompt-dominated total.
Kgrid=surface_K(wgrid,Rbh,Tdet,Rq_grid,xH,xE)
qh_delay_group=float(np.interp(wc,wgrid,tau_total))
def qh_packet_audit(wf,echo,rq):
    loop_phase=np.unwrap(np.angle(Rbh*rq))+wgrid*L_cavity_roundtrip
    loop_hint=float(np.interp(wc,wgrid,np.gradient(loop_phase,wgrid)))
    pk=int(np.argmax(abs(hilbert(echo)))); first=float(wf.t[pk]); length=wf.t[-1]-first
    empty=pd.DataFrame(columns=["time","delta_t"])
    if loop_hint<=0 or length<1.8*loop_hint: return np.nan,empty,loop_hint
    win=(first-18,first+18)
    packets=matched_packet_observables(wf.t,echo,win,.65*loop_hint,win[0])
    quality=pulse_train_quality(wf.t,echo,packets)
    return (float(packets.delta_t.median()) if quality["stable_spacing"] and quality["separated_pulses"] else np.nan),packets,loop_hint
qh_delay_obs,qh_packets,qh_loop_hint=qh_packet_audit(qh,qh_echo,Rq_grid)

# Corrected parity-resolved map from det[I-R_half S_delta]=0.
def family_periods(modes):
    result={}
    for family in ("even","odd"):
        roots=np.sort(modes.loc[modes.parity==family,"omega_R"].to_numpy())
        d=np.diff(roots); mid=.5*(roots[1:]+roots[:-1])
        wt=np.interp(mid,wpacket,Ppacket,left=0,right=0)
        use=(abs(mid-wc)<=max(2.5*packet_bandwidth,.08))&(wt>0)&(d>0)
        result[family]=float(np.average(2*np.pi/d[use],weights=wt[use])) if np.any(use) else np.nan
    return result
family_times=family_periods(wormhole_qnm_map)
# A mixed right-side recurrence is not evidence of a single resolved family.
# A common period would require agreement with BOTH populated families.
wormhole_qnm_spacing_error=float(max(abs(Tmatched-v)/v for v in family_times.values())) if all(np.isfinite(list(family_times.values()))) else np.inf


def classify_echo_report(*,model,photon_sphere,cavity,reflective,trapped,separated,stable_spacing,
                         group_agreement,qnm_agreement,boundary_stable,resolution_stable,dominant_structure):
    clean=bool(cavity and reflective and separated and stable_spacing and boundary_stable and resolution_stable and (group_agreement or qnm_agreement))
    classification="CLEAN_ECHO" if clean else ("TRAPPED_RESONANCE" if trapped else "NO_CLEAN_ECHO")
    return {"model":model,"Photon sphere present?":bool(photon_sphere),"Cavity identified?":bool(cavity),"Reflective boundary?":bool(reflective),
            "Trapped modes present?":bool(trapped),"Separated pulses identified?":bool(separated),"Stable echo spacing?":bool(stable_spacing),
            "Group-delay agreement?":bool(group_agreement),"QNM-spacing agreement?":bool(qnm_agreement),"Boundary stable?":bool(boundary_stable),
            "Resolution stable?":bool(resolution_stable),"dominant_late_structure":dominant_structure,"Echo classification":classification}


wh_quality=pulse_train_quality(wh.t,wh.psi,wh_packets)
qh_quality=pulse_train_quality(qh.t,qh_echo,qh_packets) if len(qh_packets) else {"separated_pulses":False,"stable_spacing":False,"spacing_cv":np.nan,"median_trough_ratio":np.nan}
display(wormhole_qnm_map.head(12)); print("Same-family spacing times:",family_times,"worst discrepancy vs mixed recurrence:",wormhole_qnm_spacing_error)

figq,aq=plt.subplots(1,2,figsize=(11,4));
for parity,g in wormhole_qnm_map.groupby("parity"): aq[0].plot(g.omega_R,g.omega_I,"o-",label=parity)
aq[0].axhline(0,c="k",lw=.5); aq[0].set(xlabel=r"$M\omega_R$",ylabel=r"$M\omega_I$",title="Throat-matched cavity-QNM map"); aq[0].legend()
for parity,g in wormhole_qnm_map.groupby("parity"): aq[1].plot(g.omega_R,g.Delta_omega,"o-",label=parity)
aq[1].axhline(2*np.pi/Tmatched,c="k",ls="--",label="measured recurrence"); aq[1].set(xlabel=r"$M\omega_R$",ylabel=r"$M\Delta\omega$",title="Spectral/temporal spacing"); aq[1].legend(); figq.tight_layout(); figq.savefig(ARTIFACT_DIR/"wormhole_qnm_spacing.png")

fig,ax=plt.subplots(2,3,figsize=(17,8))
wlo=max(.10,wc-2.5*packet_bandwidth); whi=min(.90,wc+2.5*packet_bandwidth); wdisplay=(wgrid>=wlo)&(wgrid<=whi)
ax[0,0].plot(x_wh,V_wh); ax[0,0].axvline(0,c="tab:red",ls="--",label=fr"jump $\Lambda={wh.metadata['throat_Lambda']:.3f}$"); ax[0,0].set(xlim=(-45,45),title="Regular potential + exact throat interface",xlabel=r"$x/M$"); ax[0,0].legend(fontsize=8)
ax[0,1].plot(wgrid,phi_wh); ax[0,1].axvline(wc,c="k",ls="--",label=fr"$\omega_c={wc:.3f}$"); plo,phi=np.nanpercentile(phi_wh[wdisplay],[1,99]); ppad=.15*(phi-plo); ax[0,1].set(xlim=(wlo,whi),ylim=(plo-ppad,phi+ppad),title=r"$\arg(R_{1/2}R_{\rm th,eff})$",xlabel=r"$M\omega$"); ax[0,1].legend()
ax[0,2].plot(wgrid,tau_w,label="full-chain group delay"); ax[0,2].fill_between(wgrid,0,Pw/max(Pw.max(),1e-300)*np.nanmax(abs(tau_w[wdisplay])),alpha=.2,label="incident packet power")
ax[0,2].axvline(wc,c="k",ls="--"); lo,hi=np.nanpercentile(tau_w[wdisplay],[1,99]); pad=.15*(hi-lo); ax[0,2].set(xlim=(wlo,whi),ylim=(lo-pad,hi+pad),title="Wigner delay across packet",xlabel=r"$M\omega$"); ax[0,2].legend(fontsize=8)
ax[1,0].plot(wh.t,wh.psi,label="right"); ax[1,0].plot(wh.t,wh_left,label="left",alpha=.7); ax[1,0].set(xlim=(100,500),title="Two-sided response",xlabel=r"$t/M$"); ax[1,0].legend()
ax[1,1].errorbar([0],[T_echo_report],yerr=[T_echo_uncertainty],fmt="o",label="reported"); ax[1,1].scatter(range(1,5),[Tmatched,Tenvelope,Tacf,Tweighted],label="methods")
ax[1,1].set(xticks=range(5),xticklabels=["combined","matched","envelope","ACF","group"],title="Echo-time estimators",ylabel=r"$T/M$"); ax[1,1].legend(fontsize=8)
for name,wf in waveforms.items(): ax[1,2].semilogy(wf.t,np.maximum(wf.amplitude/np.max(wf.amplitude),1e-9),lw=.8,label=name,color=colors[name])
ax[1,2].set(xlim=(100,500),ylim=(1e-8,2),title="Late-time morphology",xlabel=r"$t/M$"); ax[1,2].legend(ncol=2,fontsize=8)
fig.tight_layout(); fig.savefig(ARTIFACT_DIR/"wormhole_wigner_uncertainty.png")

fig,axs=plt.subplots(len(waveforms),1,figsize=(12,12),sharex=True,sharey=True)
for ax,(name,wf) in zip(axs,waveforms.items()):
    fs=1/np.mean(np.diff(wf.t)); f,tt,S=spectrogram(wf.psi,fs=fs,window=("tukey",.25),nperseg=min(512,len(wf.t)//4),noverlap=min(448,len(wf.t)//4-1),mode="magnitude")
    db=20*np.log10(np.maximum(S/S.max(),1e-8)); ax.pcolormesh(tt,2*np.pi*f,db,shading="auto",cmap="magma",vmin=-70,vmax=0); ax.set(ylim=(0,1.2),ylabel=name+"\n"+r"$M\omega$")
axs[-1].set_xlabel(r"$t/M$"); fig.suptitle("Time-frequency response (dB)"); fig.tight_layout(); fig.savefig(ARTIFACT_DIR/"spectrograms.png")


## Additional complex modes: boson star and the specified quantum horizon

These calculations solve homogeneous boundary-value problems, independently of
waveform peaks. All use $e^{-i\omega t}$ and stable modes have $\Im\omega<0$.
The reported frequencies are $M\omega$, $Mf=\Re(M\omega)/(2\pi)$,
$\tau/M=-1/\Im(M\omega)$ and $Q=\Re\omega/(2|\Im\omega|)$.
The selected lists are not proofs of spectral completeness or ordering of the
entire spectrum.

**Mini-boson star I [LITERATURE MODEL + CONTROLLED APPROXIMATION].**
The axial metric perturbation decouples from the scalar perturbation. With
$c=\alpha/a$, $\Psi=r^{\ell+1}y$ and the notebook's $\sigma=\sqrt{8\pi G}\phi$,
the regular interior equation is
$$y''+\left[\frac{2(\ell+1)}r+F\right]y'+Q_r y=0,$$
$$F=\frac{a^2-1}{r}-ra^2\sigma^2,\quad
Q_r=\frac{(4-\ell^2)(a^2-1)}{r^2}-(\ell+2)a^2\sigma^2+\frac{\omega^2}{c^2}.$$
Regular center data are matched to an outgoing Schwarzschild exterior at
$R_2=1.4R_{99}$ using continuity of $\Psi$ and $d\Psi/dr$, as in
[Macedo et al., Eq.68 and Table IV](https://arxiv.org/html/1307.4812).
The paper's finite scalar-tail truncation is reproduced explicitly, not
mistaken for an exact material surface. Varying $R_2/R_{99}=1.3,1.4,1.5$ measures
its systematic effect separately from numerical tolerance, background-grid,
center-radius and complex-exterior tests. The waveform solver retains the
full solved background; this truncated-exterior mode benchmark therefore does
not justify machine-precision mode claims for an untruncated boson star.

**OWA quantum horizon [DERIVED EFFECTIVE-MODEL POLES].**
For $\Re z>0$, let $s=iz/\kappa$ and $C=e^{-1/2}$. The analytic continuation
of the published positive-frequency ratio is
$$R_g(z)=e^{-\pi z/\kappa-2s\log[\gamma zM/(C E_{Pl}M)]}
\frac{\Gamma(2s)\Gamma(-s)\Gamma(1-s)}
{\Gamma(-2s)\Gamma(s)\Gamma(1+s)}.$$
This follows from [OWA Eqs.14-16 and 27-33](https://arxiv.org/html/1905.00464);
one must not insert $|z|$ or conjugate $A(z)$ after moving off the real axis.
The actual outgoing RW solution at a computational plane has Cauchy data
$(q,p)$. The pole equation is
$$q+p/(iz)-R_g(z)e^{2izx_H}[q-p/(iz)]=0.$$
Complex-contour outgoing data, interface changes, radial tolerances, expansion
orders and hierarchy continuation test these zeros. The real-axis gamma ratio
is checked against the coefficient used in the waveform. These are numerical
predictions of this specified boundary model, not tabulated frequencies from
the paper and not a microscopic quantum-gravity calculation.


In [ ]:
"""Actual axial mini-boson-star poles, not Fourier-response peaks.

Macedo et al. (2013), arXiv:1307.4812v3 Sec. V.A and Table IV. The
exponentially decaying scalar tail is retained to the matching radius and
replaced there by Schwarzschild, exactly the approximation whose few-percent
systematic error the paper discusses. Numerical convergence at fixed radius
does not remove this matter-tail systematic. No photon sphere is required.

Insert after solve_mini_boson_star/bs_bg and wh_outgoing_cauchy definitions.
Dependencies: numpy, scipy, pandas. Raw background lengths are in mu^-1;
reported Momega uses the independently solved ADM mass, not an assumed M=1.
"""
import numpy as np
import pandas as pd
from scipy.interpolate import CubicSpline
from scipy.integrate import solve_ivp
from scipy.optimize import root


class BosonAxialModeProblem:
    def __init__(self,bg,ell=2,match_radius=None,rtol=2e-11,center_radius=2e-4,
                 length=100.,angle=1.20,order=18):
        self.bg=bg; self.ell=ell; self.rtol=rtol; self.r0=center_radius
        self.R=1.4*bg.radius_99 if match_radius is None else float(match_radius)
        if not bg.r[0]<self.r0<self.R<bg.r[-1]:
            raise ValueError('Center and matching radii must lie inside the solved background')
        self.s=CubicSpline(bg.r,bg.sigma)
        self.a=CubicSpline(bg.r,bg.a)
        self.alpha=CubicSpline(bg.r,bg.alpha)
        self.exterior=dict(length=length,angle=angle,order=order,rtol=rtol)

    def regular_cauchy(self,w):
        """Factor Psi=r^(ell+1)y; analytically cancel the central 1/r² terms.

        c=alpha/a; F=c'/c=(a²-1)/r-r*a²*sigma² follows from EKG.
        y''+[2(ell+1)/r+F]y'+Q*y=0, with
        Q=(4-ell²)(a²-1)/r²-(ell+2)a² sigma²+w²/c².
        This avoids unstable subtraction of two centrifugal singularities.
        """
        l=self.ell; k=l+1; r0=self.r0
        al0=float(self.alpha(0.)); sc=self.bg.sigma_c
        A2=((self.bg.omega_s/al0)**2+1)*sc**2/3
        Q0=(4-l*l)*A2-(l+2)*sc**2+w*w/al0**2
        b=-Q0/(2*(2*l+3))
        def rhs(r,y):
            a=float(self.a(r)); al=float(self.alpha(r)); s=float(self.s(r))
            c=al/a; F=(a*a-1)/r-r*a*a*s*s
            Q=(4-l*l)*(a*a-1)/(r*r)-(l+2)*a*a*s*s+w*w/(c*c)
            return np.array([y[1],-(2*k/r+F)*y[1]-Q*y[0]],complex)
        sol=solve_ivp(rhs,(r0,self.R),np.array([1+b*r0*r0,2*b*r0],complex),
                      method='DOP853',rtol=self.rtol,atol=self.rtol*.01)
        if not sol.success:raise RuntimeError(sol.message)
        y,yp=sol.y[:,-1];c=float(self.alpha(self.R)/self.a(self.R))
        # Drop the common R^(ell+1) factor in both Cauchy data.
        return y,c*(yp+k*y/self.R)

    def residual(self,w):
        if w.real<=0 or w.imag>=0:raise ValueError('Require decaying positive-frequency pole')
        q,p=self.regular_cauchy(w)
        u,v=wh_outgoing_cauchy(w,self.R,self.ell,self.bg.mass,**self.exterior)
        # The paper's exterior truncation matches Psi and dPsi/dr (Eq.68),
        # not derivatives in two mismatched tortoise coordinates. Their tiny
        # finite-tail lapse difference must not introduce an artificial shell.
        p/=float(self.alpha(self.R)/self.a(self.R))
        v/=(1-2*self.bg.mass/self.R)
        return (q*v-p*u)/(np.hypot(abs(q),abs(p))*np.hypot(abs(u),abs(v)))

    def pole(self,guess):
        def fun(v):
            w=complex(*v)
            try:z=self.residual(w)
            except ValueError:return [1+abs(v[0]),1+abs(v[1])]
            return [z.real,z.imag]
        sol=root(fun,[complex(guess).real,complex(guess).imag],tol=2e-9)
        w=complex(*sol.x);res=abs(self.residual(w))
        if not sol.success or res>2e-8:
            raise RuntimeError(f'Boson axial pole not converged: {sol.message}; residual={res:g}')
        return w,res

    def tail_metadata(self):
        R=self.R; a=float(self.a(R)); al=float(self.alpha(R));s=float(self.s(R))
        m=.5*R*(1-a**-2); f=1-2*self.bg.mass/R;L=self.ell*(self.ell+1)
        Vin=al*al*(L/R**2-6*m/R**3+s*s)
        Vout=f*(L/R**2-6*self.bg.mass/R**3)
        return dict(match_radius_mu=R,match_radius_over_R99=R/self.bg.radius_99,
                    scalar_at_match=s,missing_mass_fraction=(self.bg.mass-m)/self.bg.mass,
                    relative_potential_jump=abs(Vin-Vout)/abs(Vin),
                    relative_light_speed_jump=abs(al/a-f)/abs(al/a))


def run_boson_axial_pole_audit(bg,artifact_dir,ell=2,refine_background=True):
    """Reproduce both published Table-IV modes and expose all approximations.

    The literature numbers themselves carry few-percent matching-radius
    uncertainty. We demand agreement within 5%, while separately checking
    substantially tighter fixed-model numerical convergence.
    """
    from pathlib import Path
    out=Path(artifact_dir);out.mkdir(parents=True,exist_ok=True)
    if ell!=2:raise ValueError('Published Table-IV comparison here is specifically ell=2')
    if abs(bg.phi_tilde_c-.1916)>5e-5:
        raise ValueError('Published Mini BS I benchmark requires phi_tilde_c=0.1916')
    problem=BosonAxialModeProblem(bg,ell)
    refined=None
    if refine_background:
        refined=solve_mini_boson_star(sigma_c=bg.sigma_c,n=2400,tol=1e-7)
    published=(.277-.388j,.456-.374j)
    modes=[];starts=[];conv=[];tails=[]
    for j,pub in enumerate(published,1):
        w,res=problem.pole(pub)
        for perturb in (1.,.97,1.03):
            ww,rr=problem.pole(pub*perturb)
            starts.append(dict(published_mode=j,seed_omega_R=(pub*perturb).real,
                seed_omega_I=(pub*perturb).imag,omega_R=ww.real,omega_I=ww.imag,
                residual=rr,relative_change=abs(ww-w)/abs(w)))
        for setting in (dict(length=75.),dict(length=130.),dict(angle=1.30),
                        dict(order=22),dict(rtol=5e-12),dict(center_radius=1e-4)):
            test=BosonAxialModeProblem(bg,ell,**setting);ww,rr=test.pole(w)
            conv.append(dict(published_mode=j,setting=str(setting),omega_R=ww.real,
                omega_I=ww.imag,Momega_R=(bg.mass*ww).real,Momega_I=(bg.mass*ww).imag,
                relative_change=abs(ww-w)/abs(w),residual=rr))
        if refined is not None:
            test=BosonAxialModeProblem(refined,ell,match_radius=problem.R)
            ww,rr=test.pole(w)
            conv.append(dict(published_mode=j,setting='EKG n=2400,tol=1e-7; same matching radius',
                omega_R=ww.real,omega_I=ww.imag,Momega_R=(refined.mass*ww).real,
                Momega_I=(refined.mass*ww).imag,relative_change=abs(ww-w)/abs(w),residual=rr))
        for factor in (1.3,1.4,1.5):
            test=BosonAxialModeProblem(bg,ell,match_radius=factor*bg.radius_99)
            ww,rr=test.pole(w)
            tails.append(dict(published_mode=j,omega_R=ww.real,omega_I=ww.imag,
                Momega_R=(bg.mass*ww).real,Momega_I=(bg.mass*ww).imag,
                relative_change=abs(ww-w)/abs(w),residual=rr,**test.tail_metadata()))
        delta=abs(w-pub)/abs(pub)
        modes.append(dict(model='mini boson star I; axial gravitational sector',
            published_mode=j,ell=ell,omega_R=w.real,omega_I=w.imag,
            Momega_R=(bg.mass*w).real,Momega_I=(bg.mass*w).imag,
            published_omega_R=pub.real,published_omega_I=pub.imag,
            published_relative_error=delta,published_required_less_than=.05,
            passed=delta<.05,residual=res,
            matching_radius_relative_envelope=max(t['relative_change'] for t in tails if t['published_mode']==j),
            uncertainty_description='empirical tail-truncation sensitivity, not a rigorous error bound',
            qnm_condition='regular r^(ell+1) at center; outgoing at infinity',
            approximation='Schwarzschild exterior outside 1.4 R99; finite scalar-tail systematic',
            source='Macedo et al.2013 arXiv1307.4812v3 Table IV and Sec V.C',
            source_url='https://arxiv.org/html/1307.4812#S5.T4'))
    mdf=pd.DataFrame(modes);sdf=pd.DataFrame(starts);cdf=pd.DataFrame(conv);tdf=pd.DataFrame(tails)
    for name,df in [('boson_axial_complex_poles',mdf),('boson_axial_pole_multistarts',sdf),
                    ('boson_axial_pole_numerics',cdf),('boson_axial_pole_tail_systematics',tdf)]:
        df.to_csv(out/(name+'.csv'),index=False)
    assert mdf.passed.all(),mdf
    assert max(cdf.relative_change.max(),sdf.relative_change.max())<2e-7,(cdf,sdf)
    # The two published modes are not ordered by damping: do not relabel the
    # first row as the least-damped/fundamental mode of the entire spectrum.
    return mdf,sdf,cdf,tdf




"""Complex poles of the specified OWA/RW effective quantum-horizon model.

Primary source: Oshita, Wang & Afshordi, arXiv:1905.00464v3, Eqs. (14)-(16)
and Appendix A, (27)-(33): https://arxiv.org/html/1905.00464 .
The continuation below is derived from their POSITIVE-real-frequency amplitude
ratio by replacing Gamma(a)^* by Gamma(a^*) before continuing z. Taking abs(z)
or conj(A(z)) in the complex plane is NOT an analytic continuation.

These are numerical outgoing poles of that effective reflectivity model, not
published benchmark frequencies or a derivation of a microscopic horizon.
Embed after freeze_wh_modes.py; its exterior Jost routine is shared.
"""
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.integrate import solve_ivp
from scipy.optimize import root
from scipy.special import lambertw, loggamma


def qh_global_reflection_complex(z,kappa=.25,gamma=1.,EplM=1e8,M=1.):
    """Right-half-plane analytic branch of the global A_out/A_in ratio.

    EplM=E_Pl*M, kappa has units 1/M, gamma is dimensionless. Standard
    Schwarzschild r*=r+2M log(r/(2M)-1) has lapse C exp(kappa r*),
    C=exp(-1/2). Thus the logarithm is gamma*z*M/(C*EplM).
    Principal log is analytic for Re(z)>0. The negative-frequency branch is
    separate; do NOT extend the Boltzmann |omega| by a complex absolute value.
    """
    z=np.asarray(z,complex)
    if np.any(z.real<=0) or not np.all(np.isfinite(z)):
        raise ValueError("positive-real-part complex frequency required")
    if min(kappa,gamma,EplM,M)<=0: raise ValueError("positive physical parameters required")
    s=1j*z/kappa
    logR=(-np.pi*z/kappa-2*s*np.log(gamma*z*M/(np.exp(-.5)*EplM))
          +loggamma(2*s)-loggamma(-2*s)+loggamma(-s)-loggamma(s)
          +loggamma(1-s)-loggamma(1+s))
    return np.exp(logR)


def qh_outgoing_at_plane(z,xH=-40.,ell=2,M=1.,join_radius=None,
                          length=None,angle=.7,order=18,rtol=2e-11):
    """Actual outgoing RW solution, analytically rotated outside, real inside.

    No finite-radius plane-wave outer boundary is used. Start with the
    convergent outgoing sector of the complex-radius exterior Jost solution,
    and propagate its Cauchy data through the real RW potential to xH.
    The arbitrary common normalization cancels from the pole determinant.
    """
    z=complex(z); a=3*M if join_radius is None else float(join_radius)
    if xH>=a+2*M*np.log(a/(2*M)-1): raise ValueError("matching plane must precede the join")
    ray_length=max(140*M,35/abs(z)) if length is None else float(length)
    q,p=wh_outgoing_cauchy(z,a,ell,M,length=ray_length,angle=angle,order=order,rtol=rtol)
    x0=a+2*M*np.log(a/(2*M)-1)
    def rhs(x,y):
        # Stable lapse from W avoids cancellation when r is extremely close to 2M.
        u=float(lambertw(np.exp(x/(2*M)-1)).real)
        r=2*M*(1+u);f=u/(1+u)
        V=f*(ell*(ell+1)/r**2-6*M/r**3)
        return [y[1],(V-z*z)*y[0]]
    sol=solve_ivp(rhs,(x0,xH),[q,p],method="DOP853",rtol=rtol,atol=rtol*.01)
    if not sol.success: raise RuntimeError(sol.message)
    return sol.y[:,-1]


def qh_pole_residual(z,EplM=1e8,gamma=1.,xH=-40.,ell=2,M=1.,**numeric):
    """Normalized complex outgoing determinant, not a real-axis phase crossing."""
    z=complex(z)
    if z.real<=0 or z.imag>=0: raise ValueError("decaying positive-frequency pole required")
    q,p=qh_outgoing_at_plane(z,xH,ell,M,**numeric)
    Rloc=qh_global_reflection_complex(z,1/(4*M),gamma,EplM,M)*np.exp(2j*z*xH)
    outgoing=q+p/(1j*z); incoming=q-p/(1j*z)
    return (outgoing-Rloc*incoming)/(abs(outgoing)+abs(Rloc*incoming)+1e-300)


def qh_complex_pole(guess,EplM=1e8,gamma=1.,xH=-40.,ell=2,M=1.,**numeric):
    def fun(v):
        z=v[0]+1j*v[1]
        # Guard exploratory MINPACK steps, not the final root criterion. Wild
        # steps far outside this selected low-frequency sector needlessly make
        # the real-coordinate ODE highly oscillatory and can stall a multistart.
        if not (.002/M<z.real<1.0/M and -.3/M<z.imag<0):
            return [10+abs(v[0]),10+abs(v[1])]
        try:r=qh_pole_residual(z,EplM,gamma,xH,ell,M,**numeric)
        except (ValueError,FloatingPointError):return [10+abs(v[0]),10+abs(v[1])]
        return [r.real,r.imag]
    result=root(fun,[complex(guess).real,complex(guess).imag],tol=3e-10,options={"maxfev":100})
    z=complex(*result.x)
    try:
        if not (.002/M<z.real<1.0/M and -.3/M<z.imag<0):
            raise ValueError("iteration outside the declared low-frequency root sector")
        res=float(abs(qh_pole_residual(z,EplM,gamma,xH,ell,M,**numeric)))
    except ValueError:res=np.inf
    if not np.isfinite(res) or res>2e-8:
        raise RuntimeError(f"QH complex root not converged: {result.message}; residual={res:g}")
    return z,res


def qh_multistart_poles(EplM=1e8,gamma=1.,ell=2,M=1.,xH=-40.,
                       seed_band=(.03,.45),seed_count=29,**numeric):
    """Selected low-frequency outgoing poles from TWO complex seed ladders.

    Every search outcome is logged. Distinct roots are deduplicated by complex
    distance. Multistart search is not a proof of spectral completeness.
    """
    roots=[];trials=[]
    for real in np.linspace(*seed_band,seed_count):
        for damping in (.04,.11):
            guess=complex(real,-damping*real)/M
            try:
                z,res=qh_complex_pole(guess,EplM,gamma,xH,ell,M,**numeric)
                selected=seed_band[0]<=M*z.real<=seed_band[1]
                new=selected and all(abs(z-w)*M>1e-7 for w in roots)
                if new:roots.append(z)
                trials.append(dict(seed_R=guess.real,seed_I=guess.imag,converged=True,
                    omega_R=z.real,omega_I=z.imag,residual=res,in_selected_band=selected,
                    outcome="new" if new else "duplicate" if selected else "outside declared band"))
            except RuntimeError as exc:
                trials.append(dict(seed_R=guess.real,seed_I=guess.imag,converged=False,
                    omega_R=np.nan,omega_I=np.nan,residual=np.nan,in_selected_band=False,outcome=str(exc)))
    if not roots: raise RuntimeError("no outgoing QH roots found")
    data=[]
    for i,z in enumerate(sorted(roots,key=lambda z:z.real)):
        data.append(dict(selected_index=i,EplM=EplM,gamma=gamma,M=M,ell=ell,xH=xH,
            omega_R=z.real,omega_I=z.imag,frequency_Momega_R=M*z.real,
            frequency_Momega_I=M*z.imag,
            residual=abs(qh_pole_residual(z,EplM,gamma,xH,ell,M,**numeric)),
            method="complex outgoing RW Jost determinant; positive-frequency analytic OWA branch",
            source_url="https://arxiv.org/html/1905.00464",source_equations="14-16;27-33",
            scope="selected poles of the OWA effective boundary model; not an exhaustive spectrum"))
    modes=pd.DataFrame(data); modes["Delta_omega"]=modes.omega_R.diff()
    return modes,pd.DataFrame(trials)


def run_qh_pole_audit(artifact_dir,production_reflection=None,EplM=1e8,gamma=1.,
                       ell=2,M=1.,xH=-40.,seed_count=29):
    """Compute poles and audit real-axis law, matching plane, numerics and hierarchy.

    Optional production_reflection(w) must return the GLOBAL coefficient with
    the standard-r* C factor, not a local interface ratio.
    """
    out=Path(artifact_dir);out.mkdir(parents=True,exist_ok=True)
    wr=np.geomspace(.001,.8,151)/M;kappa=1/(4*M)
    # Independent literal real-axis A*/A from Eq.32, unlike analytic gamma ratio.
    phase=np.log(gamma*wr*M/(np.exp(-.5)*EplM))
    logA=1j*wr/kappa*phase+loggamma(-2j*wr/kappa)-loggamma(-1j*wr/kappa)-loggamma(1-1j*wr/kappa)
    real_exact=np.exp(-np.pi*wr/kappa)*np.exp(-2j*logA.imag)
    analytic=qh_global_reflection_complex(wr,kappa,gamma,EplM,M)
    checks=[dict(check="positive-real-axis analytic law vs literal published A*/A",
                 value=float(np.max(abs(analytic-real_exact))),tolerance=2e-12)]
    checks.append(dict(check="real-axis Boltzmann amplitude",
        value=float(np.max(abs(abs(analytic)-np.exp(-np.pi*wr/kappa)))),tolerance=2e-12))
    if production_reflection is not None:
        checks.append(dict(check="analytic branch vs actual global waveform coefficient",
            value=float(np.max(abs(analytic-production_reflection(wr)))),tolerance=2e-12))
    modes,trials=qh_multistart_poles(EplM,gamma,ell,M,xH,seed_count=seed_count)
    modes.to_csv(out/"qh_complex_poles.csv",index=False)
    trials.to_csv(out/"qh_complex_pole_multistart.csv",index=False)
    convergence=[]
    for row in modes.itertuples():
        w=complex(row.omega_R,row.omega_I);base_length=max(140*M,35/abs(w))
        settings=[("matching plane -50M",dict(xH=-50*M)),("longer exterior contour",dict(length=base_length*1.4)),
                  ("higher asymptotic order",dict(order=22)),("rotated contour",dict(angle=.9)),
                  ("tighter ODE tolerance",dict(rtol=4e-12)),("moved contour join",dict(join_radius=2.1*M))]
        for label,options in settings:
            args=dict(EplM=EplM,gamma=gamma,xH=xH,ell=ell,M=M);args.update(options)
            z,res=qh_complex_pole(w,**args)
            convergence.append(dict(selected_index=row.selected_index,setting=label,
                omega_R=z.real,omega_I=z.imag,relative_change=abs(z-w)/abs(w),residual=res,
                tolerance=2e-6,passed=abs(z-w)/abs(w)<2e-6))
    convergence=pd.DataFrame(convergence)
    convergence.to_csv(out/"qh_complex_pole_convergence.csv",index=False)
    # Continue each branch gradually with hierarchy, recording actual complex
    # zeros at every step. EplM variation changes physics, unlike the xH check.
    hierarchy=[]
    for row in modes.head(3).itertuples():
        for target in (1e6,1e10):
            w=complex(row.omega_R,row.omega_I)
            for hh in np.geomspace(EplM,target,9)[1:]:
                w,res=qh_complex_pole(w,EplM=hh,gamma=gamma,xH=xH,ell=ell,M=M)
            hierarchy.append(dict(base_selected_index=row.selected_index,EplM=target,gamma=gamma,
                omega_R=w.real,omega_I=w.imag,residual=res,method="continued complex outgoing pole"))
        hierarchy.append(dict(base_selected_index=row.selected_index,EplM=EplM,gamma=gamma,
            omega_R=row.omega_R,omega_I=row.omega_I,residual=row.residual,
            method="base complex outgoing pole"))
    hierarchy=pd.DataFrame(hierarchy).sort_values(["base_selected_index","EplM"])
    hierarchy.to_csv(out/"qh_complex_pole_hierarchy.csv",index=False)
    checks=pd.DataFrame(checks);checks["passed"]=checks.value<checks.tolerance
    checks.to_csv(out/"qh_complex_pole_checks.csv",index=False)
    assert checks.passed.all(),checks
    assert convergence.passed.all(),convergence
    assert len(modes)>=6 and modes.residual.max()<2e-8,modes
    assert (hierarchy.omega_I<0).all() and hierarchy.residual.max()<2e-8,hierarchy
    return modes,checks,convergence,hierarchy,trials



bs_modes_df,bs_mode_multistarts,bs_mode_numerics,bs_mode_tail=run_boson_axial_pole_audit(bs_bg,ARTIFACT_DIR,CFG.ell)
qh_modes_df,qh_mode_checks,qh_mode_numerics,qh_mode_hierarchy,qh_mode_trials=run_qh_pole_audit(
    ARTIFACT_DIR,production_reflection=lambda w:quantum_reflection(w,kappa=1/(4*CFG.M),EplM=1e8),
    EplM=1e8,ell=CFG.ell,M=CFG.M,xH=xH)
display(bs_modes_df);display(bs_mode_tail);display(qh_mode_checks)
"""Common units and database-ready selected QNM inventory."""
mode_rows=[]
def add_mode(model,family,index,omegaM,method,scope):
    z=complex(omegaM)
    if not np.isfinite(z.real+z.imag) or z.real<=0 or z.imag>=0:
        raise ValueError('Selected stable mode has invalid frequency')
    mode_rows.append(dict(model=model,family=family,selected_index=index,
        Momega_R=z.real,Momega_I=z.imag,Mf=z.real/(2*np.pi),
        tau_over_M=-1/z.imag,Q=z.real/(2*abs(z.imag)),method=method,scope=scope))
for j,row in bh_qnm_df.iterrows():
    add_mode('BH','axial',j,CFG.M*complex(row.Leaver),'Leaver continued fraction','ell=2, n=0,1,2')
add_mode('GS','axial',0,CFG.M*gs_mode_fd,'regular interior / outgoing exterior matching',
    'verified selected mode of zero-surface-energy Pani shell')
for row in wormhole_qnm_map.itertuples():
    add_mode('WH',row.parity,row.Index,CFG.M*complex(row.omega_R,row.omega_I),row.method,
        'spatial-reflection parity; axial gravity; declared seed band only')
for row in bs_modes_df.itertuples():
    add_mode('BS','axial',row.published_mode,complex(row.Momega_R,row.Momega_I),
        'regular-center / outgoing complex-contour matching',row.approximation)
for row in qh_modes_df.itertuples():
    add_mode('QH','axial',row.selected_index,CFG.M*complex(row.omega_R,row.omega_I),row.method,row.scope)
qnm_comparison_df=pd.DataFrame(mode_rows)
qnm_comparison_df.to_csv(ARTIFACT_DIR/'selected_complex_modes_all_models.csv',index=False)
display(qnm_comparison_df[['model','family','selected_index','Momega_R','Momega_I','Mf','tau_over_M','Q']])
fig,ax=plt.subplots(1,2,figsize=(12,4))
for name,g in qnm_comparison_df.groupby('model',sort=False):
    ax[0].scatter(g.Momega_R,g.Momega_I,label=name,color=colors[name])
    ax[1].semilogy(g.Momega_R,g.tau_over_M,'o',label=name,color=colors[name])
ax[0].set(xlabel=r'$M\Re\omega$',ylabel=r'$M\Im\omega$',title='Selected complex poles, not FFT peaks')
ax[1].set(xlabel=r'$M\Re\omega$',ylabel=r'$\tau/M$',title='Damping times of selected modes')
for a in ax:a.legend()
fig.tight_layout();fig.savefig(ARTIFACT_DIR/'all_models_complex_modes.png')


## 10. Reconstruct the finite-detector waveform before interpreting the echoes

With time dependence $e^{-i\omega t}$, an outward wave is $e^{+i\omega X}$. For the star, $X=r_*-r_*(a)$; for the wormhole, $X=|x|$ increases outward on **each** side of the throat. In both cases the matching plane is $X=0$. The near-horizon input is measured in a separate, absorbing Schwarzschild evolution of the **same source**. The wormhole's native $x=95M$ means $r_*=r_*(r_0)+95M$, not Schwarzschild $r_*=95M$. The quantum surface coordinate plays no part in either construction.

Let the exterior outgoing Jost field have local outward/inward coefficients $A,B$. Then $R_b=B/A$, and $T_E=U(X_E)/A$ includes both radial propagation and finite-distance amplitude. $1/A$ is useful for flux checks, not as a replacement for $T_E$. For one locally incident unit, the first two gravastar returns are $T_E R_s$ and $T_E R_s(R_bR_s)$; the third acquires $(R_bR_s)^2$. The cavity propagation phase is already inside $R_b$.

To predict **ECO minus BH**, write the right exterior as $H+cU$, where $H$ is the BH solution per unit measured monitor field. Its shell characteristics $I_0,O_0$ are generally *both* nonzero because the shell is not in empty space. Matching gives $O_0+cA=R_s(I_0+cB)$, hence $K=T_E(R_s I_0-O_0)/(1-R_s R_b)$. A vanishing local reflector is not necessarily the same boundary condition as extending the BH to the horizon.

For a wormhole let $r,t$ be the exact delta-throat amplitudes. The remote-barrier series is $t^2R_f[1+rR_f+(rR_f)^2+\dots]$; adding $r$ gives $R_{\rm eff}$. Substitute $R_{\rm eff}$ for $R_s$ to sum the **near** returns. Equivalently use the even/odd throat eigenvalues $r+t$ and $r-t$. For a right-hand source the right residual is their half-sum, the transmitted left field their half-difference. There is no fitted multiplier and no common period inferred from alternating parity roots. The earlier phase-crossing QNM estimates are not exact complex-plane pole determinations.

The Laplace line $z=\omega+i\eta$ suppresses circular replicas from long-lived modes: multiply the input by $e^{-\eta t}$, apply $K(z)^*$ with NumPy's positive-frequency convention, then restore $e^{\eta t}$. We vary padding, contour, cutoff, radial resolution and outer boundary and report the change. Primary acceptance is raw relative $L^2<1\%$ in $190\le t/M\le500$; fitted shifts, scales and overlaps are diagnostics only. All reconstructed fields use the original time origin. No strain or detector calibration is assumed.

Physics references: [Pani et al., thin-shell gravastar and axial junction conditions](https://arxiv.org/abs/0909.0287); [Yang, Xia & Miao, Schwarzschild wormhole throat potential](https://doi.org/10.1140/epjc/s10052-025-14446-w); [Mark et al., near-horizon input and return-series method](https://arxiv.org/abs/1706.06155). The finite-observer errors reported below are internal numerical comparisons, not numbers quoted from those papers.

The primary direct wormhole evolution extends to $x=-400M$. A separate three-domain test checks both observers and exposes the late reflected signal when the left domain is cut at $-260M$.

### Corrections 3: distinguish the source field from its incident characteristic

The previous release assumed a pure-ingoing monitor. This release records both the field and its spatial derivative from the independently evolved BH control. With physical $z=\omega+i\eta$, $I=(\widetilde\Psi-\widetilde\Psi_X/(iz))/2$. For the conjugated NumPy transform, define $F=\mathrm{rfft}(e^{-\eta t}\Psi)$ and $D=\mathrm{rfft}(e^{-\eta t}\Psi_X)$. Then **$I_N=[F+D/(\eta+i\omega)]/2$** and $O_N=[F-D/(\eta+i\omega)]/2$. Only $I_N$ multiplies $K(z)^*$. No added spatial phase is needed: the Jost initial data already specify unit incident characteristic at the selected monitor.

The zero-real-frequency bin is at $z=i\eta$, not physical zero frequency; it is finite and is retained. The production interface rejects $\eta=0$ because the directional split of a static field is not uniquely specified. A separate real-axis diagnostic excludes only the static bin. Spectral amplitude ratios on the damped contour are **not energy ratios**.

Controls use known ingoing, outgoing and mixed traveling pulses, independently prescribed derivatives, and deliberately incorrect source/sign alternatives. The derivative recorder is optional and leaves the evolution unchanged. Its sparse weights reproduce the existing fourth-order derivative followed by extraction interpolation.

The source-only domain is enlarged to $x=-400M$; input lasts 650 M while the main comparison remains 190–500 M. Monitor location, source/PDE resolution, cutoff, damping, FFT padding, input duration and left boundary are changed **one at a time**. Coarse/fine TD comparisons use their respective matched BH controls. All amplitude/time/phase fitting is diagnostic only; the fixed raw acceptance threshold remains 1%, and numerical prediction changes must remain below 0.1%.

For the particular retarded BH initial-value problem, the horizon is absorbing and the source lies outside the monitor, so a deep monitor should be almost purely ingoing. Reflection of the external pulse by the main barrier is principally outward on the exterior side, not across that interior monitor. A measured outgoing component can also arise from truncation, numerical dispersion or a boundary return. Its size and the effect of decomposition are measured, not presumed large.

The characteristic convention and horizon-source interpretation follow [Mark et al., sections II.2 and Appendix A](https://arxiv.org/html/1706.06155); the finite-detector implementation and numerical checks here are derived, not claimed to be a published benchmark. [Pani et al.](https://arxiv.org/html/0909.0287) supports the axial thin-shell gravastar junction. The implemented wormhole remains the Schwarzschild thin-shell, throat-jump model: it is **not** the distinct Damour–Solodukhin geometry in the supplied 2007 and 2024 papers. No claim of reproducing those DS results or all possible echo physics is made. The supplied rogue-echo paper addresses interior travel times, not this monitor decomposition. Published model agreement, internal numerical agreement and observational validity are separate claims.

Passing this audit establishes the stated tests and parameter settings, not absence of every possible mistake. The saved field, derivative, incident/outgoing spectra and transfer arrays allow independent reconstruction without trusting the printed validation table.


The gravastar also uses an independent high-precision evaluation of Pani et al.
Eq. (3.8), $r^{\ell+1}(1-H^2r^2)^{-i\omega/(2H)}{}_2F_1(A,B;C;H^2r^2)$,
with $H^2=2M/a^3$, $A=(\ell+2-i\omega/H)/2$, $B=(\ell+1-i\omega/H)/2$,
$C=\ell+3/2$. Its analytically differentiated Cauchy data and metric junctions
are compared with both production ladder data and independent radial integration
at real, zero, upper- and lower-half-plane frequencies. This is the specific
zero-surface-energy-shell axial model, not a universal gravastar prescription.


In [ ]:
# Local characteristic scattering for geometric ECOs (axial RW field).
# This cell is independent of the quantum boundary and its surface coordinate.
from dataclasses import dataclass
import numpy as np


@dataclass(frozen=True)
class ScatteringReference:
    origin_rstar: float              # global Schwarzschild r_* at the seam
    detector_X: float                # detector in local X = r_* - origin_rstar
    monitor_rstar: float             # independent BH source record, global r_*
    outer_X: float                   # finite Sommerfeld Jost truncation, local X
    basis: str = "U=A exp(+izX)+B exp(-izX) in local characteristics at X=0"
    embedded_phases: str = "Rb: barrier and return to seam; Tdet: all propagation to the finite observer"
    time_basis: str = "exp(-izt); use conjugate K for the positive NumPy FFT"
    ell: int = 2
    mass: float = 1.


@dataclass(frozen=True)
class SeamScattering:
    omega: np.ndarray
    Rbar: np.ndarray
    Tinf: np.ndarray
    Tdet: np.ndarray
    jost_seam: np.ndarray
    djost_seam: np.ndarray
    jost_detector: np.ndarray
    incoming_BH: np.ndarray
    outgoing_BH: np.ndarray
    bh_seam: np.ndarray
    bh_dseam: np.ndarray
    outer_X: float
    reference: ScatteringReference


def _seam_advance(z,psi,dpsi,first,last,offset,ell,M,step):
    """Nonautonomous RK4 shooting; all frequencies share the same exact endpoints.

    offset+X is global r_*. The coefficients below are the expanded 2x2 RK4
    propagator for (Psi,Psi')'=((0,1),(V-z²,0))*(Psi,Psi'); no numerical
    radial differentiation of the field or frequency interpolation is used.
    """
    if first==last: return psi.copy(),dpsi.copy()
    n=max(1,int(np.ceil(abs(last-first)/step)))
    X=np.linspace(first,last,n+1); h=(last-first)/n
    V=rw_potential(r_from_schwarzschild_rstar(offset+X,M),ell,M)
    Vm=rw_potential(r_from_schwarzschild_rstar(offset+.5*(X[:-1]+X[1:]),M),ell,M)
    z2=z*z; h2=h*h/6; h3=h*h*h/6; h4=h**4/24; h6=h/6
    q=psi.copy(); p=dpsi.copy(); g0=V[0]-z2
    for vm,v1 in zip(Vm,V[1:]):
        gm=vm-z2; g1=v1-z2
        # The common coefficients are assembled before replacing either state.
        a=1+h2*(g0+2*gm)+h4*gm*g0; b=h+h3*gm
        c=h6*(g0+4*gm+g1)+(.5*h3)*gm*(g0+g1)
        d=1+h2*(2*gm+g1)+h4*gm*g1
        q,p=a*q+b*p,c*q+d*p
        g0=g1
    return q,p


def make_seam_scattering(w,seam_rstar,detector_X,monitor_rstar,ell=2,M=1.,outer_X=400.,step=.025):
    """Two Jost solutions, in one local characteristic basis.

    U is outgoing at the far end of the exterior, normalized to exp(+izX).
    Rbar=B/A and Tinf=1/A for U(0)=A+B, U'(0)=iz(A-B).
    Tinf is the extrapolated plane-wave coefficient at the finite far boundary;
    it is NOT the transfer to the finite detector. Tdet=U(X_E)/A already
    contains the full amplitude and phase to X_E. Never multiply it by another
    plane-wave translation. The outer truncation must be outside the causal
    domain of the observation; real-frequency diagnostics may also vary outer_X.

    H is ingoing at a near-horizon MONITOR for the source-only BH evolution,
    normalized to H=1,H'=-iz there. Initial data/source support must lie OUTSIDE
    the seam and V at the monitor must be negligible. Thus H(0), H'(0) times
    the measured INGOING characteristic are the BH Cauchy data at the actual ECO seam.
    Frequency-independent rescaling of the outgoing Jost solution cancels from
    both Rbar and Tdet. Positive Im(z) enables wrap-resistant Laplace inversion.
    """
    z=np.asarray(w)
    if z.ndim!=1 or z.size==0 or not np.all(np.isfinite(z)) or np.any(z==0):
        raise ValueError("expected a nonempty finite frequency vector; omit real DC or use z=i*eta")
    if np.any(z.real<0) or np.any(z.imag<0): raise ValueError("expected Re(z)>=0 and Im(z)>=0")
    if not (M>0 and step>0 and 0<detector_X<outer_X and monitor_rstar<seam_rstar):
        raise ValueError("invalid physical planes or step")
    vm=float(rw_potential(r_from_schwarzschild_rstar(float(monitor_rstar),M),ell,M))
    if abs(vm)>1e-11/M**2: raise ValueError("incident recorder not in the near-horizon free-wave region")
    Xmon=float(monitor_rstar-seam_rstar)
    ufar=np.exp(1j*z*outer_X); uE,duE=_seam_advance(z,ufar,1j*z*ufar,outer_X,detector_X,seam_rstar,ell,M,step)
    u,du=_seam_advance(z,uE,duE,detector_X,0.,seam_rstar,ell,M,step)
    H,dH=_seam_advance(z,np.ones(z.shape,complex),-1j*z,Xmon,0.,seam_rstar,ell,M,step)
    A=.5*(u+du/(1j*z)); B=.5*(u-du/(1j*z))
    # For the measured BH field O is right-moving, I is left-moving. O is
    # generally NOT zero at the gravastar shell even for an absorbing BH.
    I=.5*(H-dH/(1j*z)); O=.5*(H+dH/(1j*z))
    ref=ScatteringReference(float(seam_rstar),float(detector_X),float(monitor_rstar),float(outer_X),
                            ell=int(ell),mass=float(M))
    return SeamScattering(z.copy(),B/A,1/A,uE/A,u,du,uE,I,O,H,dH,float(outer_X),ref)


def _scattering_return(sc,inner):
    # Add c*U to the BH solution. At X=0: O+c*A = inner*(I+c*B).
    # Hence c*U_E/H_monitor = Tdet*(inner*I-O)/(1-inner*Rbar).
    # The subtraction -O removes the right-moving BH characteristic already
    # present at the seam. Dropping it predicts an ECO field, not ECO minus BH.
    inner=np.broadcast_to(np.asarray(inner,dtype=complex),sc.omega.shape)
    loop=sc.Rbar*inner; incident_first=sc.Tdet*inner
    first=sc.Tdet*(inner*sc.incoming_BH-sc.outgoing_BH)
    return dict(K=first/(1-loop),loop=loop,first=first,second=first*loop,
                first_per_local_incident=incident_first,
                second_per_local_incident=incident_first*loop,R_eff=inner)


def gs_seam_transfer(sc,R_shell):
    """Regular interior including shell matching; R_shell is referenced to X=0.

    The barrier round-trip phase belongs to Rbar. first and second are the
    returned *corrections per unit BH monitor*, not total incident waveforms.
    first_per_local_incident and second_per_local_incident are the ordinary
    shell return series for a unit left-moving characteristic at the shell.
    """
    chain=_scattering_return(sc,R_shell)
    chain["R_shell"]=chain["R_eff"]
    return chain


def wh_seam_transfer(sc,sc_opposite,Lambda):
    """Delta throat + opposite exterior, then near-side returns; all ports at X=0.

    Each local X increases away from the throat. For identical exteriors,
    throat parity eigenvalues are s+=r+t and s-=r-t=-1. The right-incident
    source is the half-sum of the two parity solutions (KW=(K+ + K-)/2).
    No single resonance spacing is inferred from the mixed response.
    """
    if not np.array_equal(sc.omega,sc_opposite.omega): raise ValueError("half-barrier frequency grids differ")
    if abs(sc.reference.origin_rstar-sc_opposite.reference.origin_rstar)>1e-11:
        raise ValueError("inconsistent near/far throat bases")
    if not np.isfinite(Lambda) or Lambda<0: raise ValueError("invalid throat jump")
    z=sc.omega; rd=Lambda/(2j*z-Lambda); td=2j*z/(2j*z-Lambda)
    far=sc_opposite.Rbar
    eff=rd+td**2*far/(1-rd*far); chain=_scattering_return(sc,eff)
    chain.update(r_delta=rd,t_delta=td,far_loop=rd*far,
                 # Actual transmitted first and next remote-barrier returns at throat:
                 opposite_first=td**2*far,opposite_second=td**2*far*(rd*far))
    if not np.allclose(far,sc.Rbar,rtol=1e-7,atol=1e-9):
        raise ValueError("parity splitting requires identical exterior scattering at both throat faces")
    ep=_scattering_return(sc,rd+td); op=_scattering_return(sc,rd-td)
    chain.update(K_even=ep["K"],K_odd=op["K"],loop_even=ep["loop"],loop_odd=op["loop"],
                 even_pole_denominator=1-ep["loop"],odd_pole_denominator=1-op["loop"])
    return chain

def run_characteristic_controls(decompose, artifact_dir):
    """Independent known traveling pulses, including a deliberately mixed monitor.

    At X=0 choose f(t+X)+g(t-X), so spatial derivative is f'(t)-g'(t).
    Their separately known transforms, not an inverse of the decomposition,
    are the oracles. A pure outgoing pulse must NOT feed an ingoing transfer.
    """
    import numpy as np
    import pandas as pd
    from pathlib import Path
    t=np.arange(0.,320.,.05); nf=32768
    def pulse(center,width,carrier,amplitude):
        u=t-center; e=amplitude*np.exp(-.5*(u/width)**2)
        return e*np.cos(carrier*u), e*(-u/width**2*np.cos(carrier*u)-carrier*np.sin(carrier*u))
    f,df=pulse(105.,9.,.38,1.); g,dg=pulse(170.,13.,.21,.62)
    rows=[]
    def check(name,eta,observed,limit):
        value=float(observed); rows.append(dict(test=name,eta=eta,max_error=value,tolerance=limit,passed=bool(np.isfinite(value) and value<limit)))
    for eta in (.0035,.005,.008):
        damp=np.exp(-eta*t)
        known_f=np.fft.rfft(f*damp,nf); known_g=np.fft.rfft(g*damp,nf)
        for label,incident,outgoing,derivative in (("pure ingoing",f,np.zeros_like(g),df),
                  ("pure outgoing",np.zeros_like(f),g,-dg),("mixed",f,g,df-dg)):
            d=decompose(t,incident+outgoing,derivative,nf,eta)
            ei=known_f if label!="pure outgoing" else np.zeros_like(known_f)
            eo=known_g if label!="pure ingoing" else np.zeros_like(known_g)
            norm=max(np.linalg.norm(ei+eo),1e-300)
            check(label+" independent ingoing spectrum",eta,np.linalg.norm(d["ingoing_fft"]-ei)/norm,2e-11)
            check(label+" independent outgoing spectrum",eta,np.linalg.norm(d["outgoing_fft"]-eo)/norm,2e-11)
            check(label+" sum recovers measured field",eta,np.linalg.norm(d["ingoing_fft"]+d["outgoing_fft"]-d["field_fft"])/norm,2e-14)
            # The first rFFT bin is z=i*eta, not the singular static z=0.
            check(label+" imaginary-axis DC",eta,abs(d["ingoing_fft"][0]-ei[0])/norm,2e-11)
        # A sign error would interchange the two independently prescribed pulses.
        bad=.5*(d["field_fft"]-d["derivative_fft"]/(1j*np.conj(d["omega"])))
        wrong_error=np.linalg.norm(bad-known_f)/np.linalg.norm(known_f)
        check("negative control rejects wrong characteristic sign",eta,0. if wrong_error>.5 else 1.,.5)
        total_error=np.linalg.norm(d["field_fft"]-known_f)/np.linalg.norm(known_f)
        check("negative control rejects total-as-incident",eta,0. if total_error>.1 else 1.,.5)
    try:
        decompose(t,f+g,df-dg,nf,0.)
    except ValueError:
        rejected=True
    else:
        rejected=False
    check("real static DC must not be silently split",0.,0. if rejected else 1.,.5)
    # On real nonzero frequency the physical sign is exactly that in the PDF;
    # conjugating physical data flips i and gives the NumPy-convention formula.
    w=np.array([.17,.39,.81]); I=np.array([1.+.4j,.3-.2j,.8+.15j]); O=np.array([.2-.3j,-.1+.5j,.17-.02j])
    psi=I+O; deriv=1j*w*(O-I)
    in_phys=.5*(psi-deriv/(1j*w))
    in_numpy=.5*(np.conj(psi)+np.conj(deriv)/(1j*w))
    check("physical and NumPy real-frequency signs agree",0.,np.max(abs(np.conj(in_phys)-in_numpy)),1e-14)
    result=pd.DataFrame(rows)
    result.to_csv(Path(artifact_dir)/"characteristic_controls.csv",index=False)
    if not result.passed.all(): raise AssertionError(result.loc[~result.passed].to_string(index=False))
    return result

c3_controls_df=run_characteristic_controls(monitor_characteristics,ARTIFACT_DIR); display(c3_controls_df)


In [ ]:
def gs_regular_shell(z,compactness,ell=2,M=1.):
    """Regular de Sitter RW solution via the exact angular-momentum ladder.

    rho=atanh(Hr)/H, V_l=l(l+1) H² csch²(H rho); the regular seed
    sin(z*rho)/z is raised with -d/drho + j H coth(H rho).
    The normalization is arbitrary but BOTH Cauchy data use the same one.
    """
    a=M/compactness; H=np.sqrt(2*M/a**3); rho=np.arctanh(H*a)/H
    z=np.asarray(z,complex); q=rho*np.sinc(z*rho/np.pi); p=np.cos(z*rho)
    h=H/np.tanh(H*rho); v=H**2/np.sinh(H*rho)**2
    for j in range(1,ell+1):
        q,p=j*h*q-p,(z*z-j*j*v)*q+j*h*p
    return q,p

def gs_shell_reflectivity(w,compactness,ell=2,M=1.):
    q,p=gs_regular_shell(w,compactness,ell,M)
    return (1j*w*q+p)/(1j*w*q-p)

def run_gs_reconstruction_checks():
    mu=CFG.gravastar_compactness; a=CFG.M/mu; xs=gs.metadata["shell_x"]
    w=np.array([.17,.28,.37,.44,.61,.78,.91]); f=1-2*CFG.M/a
    sc=make_seam_scattering(w,xs,CFG.extraction_x-xs,-65.,step=.025)
    R=gs_shell_reflectivity(w,mu,CFG.ell,CFG.M); chain=gs_seam_transfer(sc,R)
    rows=[]
    def check(name,u,tolerance):
        err=float(np.max(np.abs(u)))
        rows.append(dict(test=name,max_error=err,tolerance=tolerance,passed=bool(err<tolerance)))
    # Independent radial-coordinate adaptive regular-center solve followed by
    # a simultaneous 2x2 junction solve for the external reflector and internal scale.
    rin=[]; dpsi=[]; psi=[]; te=[]
    for i,z in enumerate(w):
        q,pr=gravastar_regular_data(a,z,CFG.ell,CFG.M); p=f*pr
        rr,C=np.linalg.solve([[1.,-q],[1j*z,-p]],[-1.,1j*z])
        rin.append(rr); psi.append(1+R[i]-C*q); dpsi.append(1j*z*(R[i]-1)-C*p)
        def rhs(X,y):
            V=rw_potential(r_from_schwarzschild_rstar(X+xs,CFG.M),CFG.ell,CFG.M)
            return [y[1],(V-z*z)*y[0]]
        sol=solve_ivp(rhs,(0.,sc.reference.detector_X),[1+sc.Rbar[i],1j*z*(1-sc.Rbar[i])],
            method="DOP853",rtol=5e-12,atol=1e-13)
        if not sol.success: raise RuntimeError(sol.message)
        te.append(sol.y[0,-1])
    check("independent 2x2 shell junction R",R-np.asarray(rin),5e-8)
    check("continuity of field",psi,5e-8); check("continuity of tortoise derivative",dpsi,5e-8)
    check("lossless regular interior",abs(R)-1,1e-11)
    check("unit-shell-incident IVP vs finite detector",sc.Tdet-te,8e-7)
    check("exterior Wronskian/flux",abs(sc.Rbar)**2+abs(sc.Tinf)**2-1,2e-7)
    check("ratio of successive returns",chain["second_per_local_incident"]/chain["first_per_local_incident"]-sc.Rbar*R,1e-12)
    check("series vs direct matching",chain["K"]-sc.jost_detector*(R*sc.incoming_BH-sc.outgoing_BH)/
          (.5*(sc.jost_seam+sc.djost_seam/(1j*w))-R*.5*(sc.jost_seam-sc.djost_seam/(1j*w))),1e-10)
    # Weak reflection relative to a locally absorbing seam; finite-seam BH
    # outgoing data are not part of a shell's first return for unit incidence.
    eps=1e-3; weak=gs_seam_transfer(sc,eps*R)
    check("linear weak first-return amplitude",weak["first_per_local_incident"]/eps-chain["first_per_local_incident"],1e-12)
    check("weak total vs first, O(eps) relative",weak["first_per_local_incident"]/(1-weak["loop"])/weak["first_per_local_incident"]-1,.00101)
    df=pd.DataFrame(rows); df.to_csv(ARTIFACT_DIR/"gs_scattering_checks.csv",index=False)
    assert df.passed.all(),df
    return df

def _gs_td_port(t, x, potential, *, center=False, initial=None, left_drive=None,
                right_drive=None, sensor=0):
    """Independent real leapfrog RW solver, with characteristic ports; returns ONLY its TD trace.

    At a left (right) port enforce q_t - q_x = 2 g' (q_t + q_x = 2 g'), where
    g is the prescribed incoming characteristic field, and a zero drive absorbs.
    Eliminate the ghost with this condition and the wave equation at the endpoint.
    Local port velocity/field amplitudes are defined with k=omega, not sqrt(omega²-V).
    A regular stellar center is physical: q=0 there, rather than an absorbing port.
    """
    h=float(x[1]-x[0]); dt0=float(t[1]-t[0]); c=dt0/h
    if c*c*(4+h*h*np.max(potential))>=4: raise ValueError("unstable port experiment")
    if initial is None: q=np.zeros(len(x)); p=np.zeros(len(x))
    else: q,p=initial
    q=q.copy(); d2=np.zeros_like(q); d2[1:-1]=(q[:-2]-2*q[1:-1]+q[2:])/h**2
    prev=q-dt0*p+.5*dt0**2*(d2-potential*q)
    buf=np.empty_like(q); trace=np.empty_like(t); trace[0]=q[sensor]
    l=np.zeros_like(t) if left_drive is None else left_drive
    r=np.zeros_like(t) if right_drive is None else right_drive
    b=2.-2*c*c-dt0**2*potential; z=2*c*c; coef=1./(1.+c)
    for j in range(len(t)-1):
        buf[1:-1]=b[1:-1]*q[1:-1]+c*c*(q[:-2]+q[2:])-prev[1:-1]
        dl=l[j+1]-(l[j-1] if j else 0.)
        dr=r[j+1]-(r[j-1] if j else 0.)
        buf[0]=0. if center else coef*(b[0]*q[0]+z*q[1]-(1.-c)*prev[0]+2*c*dl)
        buf[-1]=coef*(b[-1]*q[-1]+z*q[-2]-(1.-c)*prev[-1]+2*c*dr)
        trace[j+1]=buf[sensor]; prev,q,buf=q,buf,prev
    return trace


def _gs_envelope_peak(t,y):
    # Unfitted envelope maximum over the whole record; a three-point interpolation
    # only removes time-sampling quantisation. No theoretical peak window is supplied.
    z=abs(hilbert(y)); j=int(np.argmax(z))
    if not 1<j<len(z)-2: raise ValueError("truncated first-return packet")
    offset=.5*(z[j-1]-z[j+1])/(z[j-1]-2*z[j]+z[j+1])
    return float(t[j]+np.clip(offset,-.5,.5)*(t[1]-t[0]))


def gs_narrowband_checks():
    """Single shell-interior return without subsequent shell/barrier feedback.

    A fixed, initially ingoing Gaussian carrier in global r* is evolved with an
    open shell port, recording its arriving incident characteristic I0(t).
    On the independent TD branch a second evolution drives the *regular interior*
    by I0; the outgoing characteristic at the shell is q_shell-I0. A third
    evolution launches just this measured outgoing characteristic into the RW
    exterior and absorbs its later barrier reflection at the shell. This open-loop
    experiment rigorously isolates a single interior return even if pulses overlap.
    The parallel FD branch sees only I0 and independently calculated T_E and
    R_shell, never the returned traces. It predicts T_E R_shell I0, not a
    repeated-return transfer or the global BH-subtracted production waveform.
    """
    # Canonical dimensionless M=1, ell=2 test at fixed physical planes. Repeating
    # the independent TD experiment on a coarser grid measures the discretization
    # error; the finer solution, not an extrapolation, is compared and plotted.
    mass=1.; ell=2
    launch=115.; detector=95.; outer=380.; carrier=.61; sigma=15.
    rows=[]; figure,axes=plt.subplots(2,1,figsize=(10,6),sharex=True,layout="constrained")
    def experiment(mu,hgoal,dtp,spectra):
        a=mass/mu; xs=float(schwarzschild_rstar(a,mass))
        L=detector-xs; nE=int(round(L/hgoal)); h=L/nE
        xe=xs+h*np.arange(int(np.ceil((outer-xs)/h))+1)
        t=dtp*np.arange(int(365./dtp)+1); nf=1<<(2*len(t)-1).bit_length()
        om=2*np.pi*np.fft.rfftfreq(nf,dtp); keep=(om>=.13)&(om<=1.12); w=om[keep]
        # A finite detector, a fixed exterior source, and matched characteristic input.
        V=rw_potential(r_from_schwarzschild_rstar(xe,mass),ell,mass)
        d=xe-launch; en=np.exp(-.5*(d/sigma)**2)
        q=en*np.cos(carrier*d)
        p=en*(-d/sigma**2*np.cos(carrier*d)-carrier*np.sin(carrier*d))
        incident=_gs_td_port(t,xe,V,initial=(q,p),sensor=0)
        H=np.sqrt(2*mass/a**3); length=np.arctanh(H*a)/H
        ni=int(round(length/hgoal)); rho=np.linspace(0.,length,ni+1); ri=np.tanh(H*rho)/H
        vi=np.zeros_like(ri); vi[1:]=(1-H*H*ri[1:]**2)*ell*(ell+1)/ri[1:]**2
        shell_total=_gs_td_port(t,rho,vi,center=True,right_drive=incident,sensor=-1)
        returned=shell_total-incident
        direct=_gs_td_port(t,xe,V,left_drive=returned,sensor=nE)
        # Each non-negligible positive-frequency bin is evaluated directly: no phase interpolation.
        # The horizon monitor field in sc is unused in this open-port test.
        if spectra is None or not np.array_equal(w,spectra[0]):
            sc=make_seam_scattering(w,seam_rstar=xs,detector_X=L,monitor_rstar=xs-65.,
                                    ell=ell,M=mass)
            rs=gs_shell_reflectivity(w,mu,ell=ell,M=mass)
            spectra=(w,sc.Tdet*rs)
        Z=np.fft.rfft(incident,nf); Y=np.zeros_like(Z); Y[keep]=Z[keep]*np.conj(spectra[1])
        fd=np.fft.irfft(Y,nf)[:len(t)]
        missing=float(np.sum(abs(Z[~keep])**2)/np.sum(abs(Z)**2))
        peak_td=_gs_envelope_peak(t,direct); peak_fd=_gs_envelope_peak(t,fd)
        err=np.linalg.norm(fd-direct)/np.linalg.norm(direct)
        result=dict(compactness=mu,shell_global_rstar=xs,exterior_source_global_rstar=launch,
            detector_global_rstar=detector,detector_relative_to_shell=L,carrier=carrier,
            envelope_sigma_M=sigma,incident_power_fraction_outside_FD_band=missing,
            t_peak_TD=peak_td,t_peak_FD=peak_fd,peak_error_M=abs(peak_td-peak_fd),
            raw_relative_L2=err,TD_outer_boundary=xe[-1],spatial_step_M=h,
            temporal_step_M=dtp,window=f"0..{t[-1]:.3f} M, full record; unfitted; L2 / TD norm")
        return result,t,direct,fd,spectra
    for ax,mu in zip(axes,(.44,.49)):
        coarse,*_,spectra=experiment(mu,.08,.032,None)
        fine,t,direct,fd,_=experiment(mu,.04,.016,spectra)
        fine.update(coarse_raw_relative_L2=coarse["raw_relative_L2"],
                    coarse_peak_error_M=coarse["peak_error_M"],
                    coarse_spatial_step_M=coarse["spatial_step_M"],coarse_temporal_step_M=.032,
                    observed_L2_error_ratio=coarse["raw_relative_L2"]/fine["raw_relative_L2"])
        rows.append(fine); xs=fine["shell_global_rstar"]
        ax.plot(t,direct,label="TD: open-loop interior return",lw=1.3)
        ax.plot(t,fd,"--",label="FD: finite-detector prediction",lw=.9)
        ax.set_ylabel("field"); ax.set_title(f"M/a={mu:g}; shell at r*={xs:.3f} M")
        ax.set_xlim(140,305); ax.legend()
    axes[-1].set_xlabel("t/M, same exterior launch and observer in both experiments")
    figure.savefig(ARTIFACT_DIR/"gs_first_return_narrowband.png",bbox_inches="tight")
    df=pd.DataFrame(rows); dtd=float(np.diff(df.t_peak_TD)[0]); dfd=float(np.diff(df.t_peak_FD)[0])
    df["shell_change_peak_shift_TD_M"]=dtd; df["shell_change_peak_shift_FD_M"]=dfd
    df["shell_change_shift_error_M"]=abs(dtd-dfd)
    assert np.isfinite(df.raw_relative_L2).all() and df.raw_relative_L2.max()<.005, df
    assert df.observed_L2_error_ratio.between(3.,5.).all(), df
    assert df.incident_power_fraction_outside_FD_band.max()<1e-5, df
    assert df.peak_error_M.max()<.05 and abs(dtd-dfd)<.05 and abs(dtd)>1., df
    df.to_csv(ARTIFACT_DIR/"gs_first_return_narrowband.csv",index=False)
    return df

gs_component_df=run_gs_reconstruction_checks(); gs_timing_df=gs_narrowband_checks(); display(gs_component_df); display(gs_timing_df)

"""Independent published gravastar solution: Pani et al. arXiv:0909.0287v2 Eq. (3.8).

This is NOT a numerical QNM value read from a plot. It evaluates the published
analytic regular solution at high precision, differentiates its hypergeometric
form analytically, and compares normalization-invariant Cauchy data with the
production ladder and the independent radial ODE. Requires numpy, scipy,
pandas, mpmath (all used explicitly; no network access during execution).
"""
import mpmath as mp


def pani_regular_hypergeometric(r,omega,a,ell=2,M=1.,dps=70):
    """Pani Eq3.8 and d/dx=f*d/dr, in the zero-surface-energy shell model.

    H²=2M/a³, z=H²r², alpha=-i omega/(2H),
    A=(ell+2-i omega/H)/2, B=(ell+1-i omega/H)/2, C=ell+3/2.
    Psi=r^(ell+1)*(1-z)^alpha*2F1(A,B;C;z).
    The analytic derivative uses d2F1/dz=(A*B/C)*2F1(A+1,B+1;C+1;z).
    """
    with mp.workdps(dps):
        r=mp.mpf(str(r));a=mp.mpf(str(a));M=mp.mpf(str(M));omega=mp.mpc(omega)
        H=mp.sqrt(2*M/a**3);z=H**2*r*r;f=1-z
        alpha=-1j*omega/(2*H);A=(ell+2-1j*omega/H)/2;B=(ell+1-1j*omega/H)/2;C=mp.mpf(ell)+mp.mpf('1.5')
        F=mp.hyp2f1(A,B,C,z);Fz=A*B/C*mp.hyp2f1(A+1,B+1,C+1,z)
        pref=r**(ell+1)*f**alpha
        psi=pref*F
        deriv=pref*(((ell+1)/r-alpha*2*H**2*r/f)*F+2*H**2*r*Fz)
        return complex(psi),complex(f*deriv)


def run_pani_gravastar_benchmark(artifact_dir):
    from pathlib import Path
    import numpy as np
    import pandas as pd
    rows=[]
    # Frequencies include real scattering, the physical imaginary-axis DC bin,
    # upper inversion contour and the lower-half-plane QNM shooting region.
    for mu in (.40,.44,.47,.49,.495):
        for om in (.0,.17,.37,.61,.91,.008j,.17+.008j,.61+.008j,.405-.025j):
            a=CFG.M/mu
            q,p=pani_regular_hypergeometric(a,om,a,CFG.ell,CFG.M,70)
            q2,p2=pani_regular_hypergeometric(a,om,a,CFG.ell,CFG.M,100)
            ql,pl=gs_regular_shell(np.asarray([om]),mu,CFG.ell,CFG.M)
            ql,pl=complex(ql[0]),complex(pl[0])
            # Scaling-invariant mismatch; unlike a logarithmic derivative it is
            # also regular when the field happens to pass through zero.
            cross=abs(ql*p-pl*q)/(np.hypot(abs(ql),abs(pl))*np.hypot(abs(q),abs(p)))
            qode,pode=gravastar_regular_data(a,om,CFG.ell,CFG.M);pode*=(1-2*CFG.M/a)
            cross_ode=abs(qode*p-pode*q)/(np.hypot(abs(qode),abs(pode))*np.hypot(abs(q),abs(p)))
            reflect=(1j*om*q+p)/(1j*om*q-p)
            reflect_ladder=complex(gs_shell_reflectivity(np.asarray([om]),mu,CFG.ell,CFG.M)[0])
            # Both references are rounded to complex128 only on return. This
            # measures stability of the float64 reference under increased
            # working precision, not a claim of 70-digit float64 accuracy.
            precision=max(abs(q-q2)/(1+abs(q2)),abs(p-p2)/(1+abs(p2)))
            # Metric-variable junction from Pani3.9 and3.29, not just the
            # equivalent master-field reflection formula. Use unit incident
            # exterior Psi=1+R, Psi_x=iω(R-1) and interior scale C0.
            # At omega=0 h0 is not defined by3.9; test only Cauchy data there.
            shell_h0=shell_sqrt_h_h1=np.nan
            if om!=0:
                # Exterior data use the production reflection coefficient;
                # interior data use the independent published solution.
                C0=(1+reflect_ladder)/q
                pout=1j*om*(reflect_ladder-1);qout=1+reflect_ladder;f=1-2*CFG.M/a
                h0in=-1j/om*(f*C0*q+a*C0*p)
                h0out=-1j/om*(f*qout+a*pout)
                hh1in=np.sqrt(f)*a/f*C0*q;hh1out=np.sqrt(f)*a/f*qout
                shell_h0=abs(h0in-h0out)/(1+abs(h0in))
                shell_sqrt_h_h1=abs(hh1in-hh1out)/(1+abs(hh1in))
            passed=(cross<2e-10 and cross_ode<2e-8 and abs(reflect-reflect_ladder)<2e-10
                    and precision<1e-14 and (om==0 or max(shell_h0,shell_sqrt_h_h1)<2e-12))
            rows.append(dict(model='GS',reference='Pani et al.0909.0287v2 Eq3.8; junction3.29',
                compactness=mu,ell=CFG.ell,M=CFG.M,omega_real=np.real(om),omega_imag=np.imag(om),
                published_analytic_R_real=reflect.real,published_analytic_R_imag=reflect.imag,
                numerical_ladder_R_real=reflect_ladder.real,numerical_ladder_R_imag=reflect_ladder.imag,
                reflection_absolute_error=abs(reflect-reflect_ladder),ladder_projective_Cauchy_error=cross,
                radial_ODE_projective_Cauchy_error=cross_ode,precision70_vs100=precision,
                junction_h0_scaled_error=shell_h0,junction_sqrt_h_h1_scaled_error=shell_sqrt_h_h1,
                comparison='exact published analytic solution evaluated numerically; NOT a digitized QNM',passed=passed))
    df=pd.DataFrame(rows);df.to_csv(Path(artifact_dir)/'pani_gravastar_analytic_benchmark.csv',index=False)
    if not df.passed.all():raise AssertionError(df.loc[~df.passed].to_string(index=False))
    return df


pani_df=run_pani_gravastar_benchmark(ARTIFACT_DIR)
display(pani_df)


In [ ]:
# Wormhole transfer: independent signed-coordinate and time-domain tests.
from scipy.integrate import solve_ivp
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


def _wh_oracle_jost(z,ref,side):
    """Shoot in physical signed x from the appropriate far boundary toward x=0.

    This DOP853 reference uses no production step matrix, recurrence or
    characteristic decomposition along the trajectory. The outgoing derivative
    has the opposite sign at the LEFT boundary. Results are returned in each
    side's OUTWARD coordinate so the matching equations below have a plus sign
    between the two outward derivatives.
    """
    z=np.asarray(z,complex); m=z.size; outer=ref.outer_X
    u0=np.exp(1j*z*outer); y0=np.r_[u0,side*1j*z*u0]
    def rhs(x,y):
        radius=r_from_schwarzschild_rstar(ref.origin_rstar+abs(x),ref.mass)
        V=float(rw_potential(radius,ref.ell,ref.mass))
        return np.r_[y[m:],(V-z*z)*y[:m]]
    sol=solve_ivp(rhs,[side*outer,0.],y0,method="DOP853",
                  t_eval=[side*ref.detector_X,0.],rtol=3e-11,atol=1e-12)
    if not sol.success: raise RuntimeError(sol.message)
    U,P,D=sol.y[:m,1],side*sol.y[m:,1],sol.y[:m,0]
    A=.5*(U+P/(1j*z)); B=.5*(U-P/(1j*z))
    return dict(U=U,P=P,D=D,R=B/A,Tinf=1/A,Tdet=D/A)


def _wh_oracle_bh(z,ref):
    """Independent normalization of H,H' from the source-only BH monitor."""
    z=np.asarray(z,complex); n=z.size
    def rhs(x,y):
        V=float(rw_potential(r_from_schwarzschild_rstar(x,ref.mass),ref.ell,ref.mass))
        return np.r_[y[n:],(V-z*z)*y[:n]]
    sol=solve_ivp(rhs,[ref.monitor_rstar,ref.origin_rstar],np.r_[np.ones(n),-1j*z],
                  method="DOP853",rtol=3e-11,atol=1e-12,t_eval=[ref.origin_rstar])
    if not sol.success: raise RuntimeError(sol.message)
    return sol.y[:n,0],sol.y[n:,0]


def audit_wh_transfer(near,far,chain,Lambda,t,direct_right,direct_left,
                      pred_eigen_even,pred_eigen_odd,*,artifact_dir):
    """Four unshifted, unscaled waveform tests and independent component controls.

    direct_right = total on the driven side MINUS its matched BH; direct_left =
    total at the opposite detector, where the reference signal is zero.
    pred_eigen_* are FULL (not half-weighted) inverse transforms of K_even/odd.
    Return (checks, parity_table). All thresholds are fixed, not fit to the data.
    """
    root=Path(artifact_dir); root.mkdir(exist_ok=True,parents=True)
    z=near.omega
    picks=np.unique([np.argmin(abs(z.real-w/near.reference.mass))
                     for w in (.12,.23,.33,.42,.59,.81)])
    if len(picks)<5: raise ValueError("WH scattering audit requires resolved frequencies in 0.12..0.81 / M")
    zz=z[picks]; right=_wh_oracle_jost(zz,near.reference,1); left=_wh_oracle_jost(zz,far.reference,-1)
    H,HP=_wh_oracle_bh(zz,near.reference)
    checks=[]
    def ck(label,observed,tol):
        v=float(np.max(np.abs(observed)))
        checks.append(dict(check=label,max_error=v,tolerance=tol,passed=bool(np.isfinite(v) and v<tol)))
    def scaled(a,b): return (a-b)/(1+np.abs(b))
    for sc,oracle,tag in ((near,right,"near"),(far,left,"opposite")):
        for name,pkey in (("R","Rbar"),("Tinf","Tinf"),("Tdet","Tdet")):
            ck(tag+" exterior signed DOP853 "+pkey,scaled(getattr(sc,pkey)[picks],oracle[name]),2e-5)
    ck("BH monitor to seam field",scaled(near.bh_seam[picks],H),2e-5)
    ck("BH monitor to seam derivative",scaled(near.bh_dseam[picks],HP),2e-5)
    # Flux is meaningful on the REAL axis only, not on the damped inverse contour.
    real_z=zz.real
    for ref,side,tag in ((near.reference,1,"near"),(far.reference,-1,"opposite")):
        v=_wh_oracle_jost(real_z,ref,side)
        ck(tag+" real flux |R|^2+|Tinf|^2-1",np.abs(v["R"])**2+np.abs(v["Tinf"])**2-1,1e-7)
    # Isolated delta: unit incidence; solve continuity and the derivative jump as
    # a 2x2 system for reflected and transmitted coefficients, without the series.
    dmat=np.zeros((len(zz),2,2),complex); dmat[:,0,0]=1; dmat[:,0,1]=-1
    dmat[:,1,0]=1j*zz-Lambda; dmat[:,1,1]=1j*zz
    rhs=np.column_stack((-np.ones(len(zz)),1j*zz+Lambda))
    rt=np.linalg.solve(dmat,rhs[...,None])[...,0]
    ck("isolated delta reflected piece",scaled(chain["r_delta"][picks],rt[:,0]),1e-12)
    ck("isolated delta transmitted piece",scaled(chain["t_delta"][picks],rt[:,1]),1e-12)
    rreal=Lambda/(2j*real_z-Lambda); treal=1+rreal
    ck("isolated real delta conserves flux",abs(rreal)**2+abs(treal)**2-1,1e-12)
    # Terminate the far side directly by its outgoing logarithmic derivative.
    # Multiplication by V avoids division by a Jost zero.
    U,P,D=right["U"],right["P"],right["D"]
    V,Q,E=left["U"],left["P"],left["D"]
    eff=((1j*zz+Lambda)*V-Q)/((1j*zz-Lambda)*V+Q)
    ck("effective reflector from opposite-side field and jump",scaled(chain["R_eff"][picks],eff),2e-5)
    # Fully assembled solution from continuity and derivative jump. No delta
    # reflectivity, effective reflector or geometric denominator in this branch.
    # On the right y=H+c U, on the left y=d V, at unit BH monitor amplitude.
    matrix=np.zeros_like(dmat); matrix[:,0,0]=U; matrix[:,0,1]=-V
    matrix[:,1,0]=P-Lambda*U; matrix[:,1,1]=Q
    c,d=np.linalg.solve(matrix,np.column_stack((-H,Lambda*H-HP))[...,None])[...,0].T
    ke,ko=chain["K_even"],chain["K_odd"]
    ck("assembled right transfer vs two-sided Cauchy match",scaled(chain["K"][picks],c*D),3e-5)
    ck("assembled left transfer vs two-sided Cauchy match",scaled((ke[picks]-ko[picks])/2,d*E),3e-5)
    ck("even transfer vs direct Robin y'=Lambda*y/2",scaled(ke[picks],(.5*Lambda*H-HP)*D/(P-.5*Lambda*U)),3e-5)
    ck("odd transfer vs direct Dirichlet y=0",scaled(ko[picks],-H*D/U),3e-5)
    ck("parity half-sum on the entire supplied frequency grid",scaled(.5*(ke+ko),chain["K"]),1e-8)
    loc=chain["first"][picks]; far1=chain["opposite_first"][picks]
    ck("near second return/first return is the near loop",chain["second"][picks]/loc-chain["loop"][picks],2e-12)
    ck("opposite second return/first return is the remote loop",chain["opposite_second"][picks]/far1-chain["far_loop"][picks],2e-12)
    checks=pd.DataFrame(checks); checks.to_csv(root/"wh_scattering_checks.csv",index=False)
    # For a one-sided source, R+L and R-L have the same incident normalization
    # as each K eigenchannel. Taking halves only when reconstructing the detectors
    # preserves absolute amplitudes without fitted factors.
    t,R,L,pe,po=map(np.asarray,(t,direct_right,direct_left,pred_eigen_even,pred_eigen_odd))
    if any(a.shape!=t.shape or not np.isfinite(a).all() for a in (R,L,pe,po)):
        raise ValueError("WH parity check requires four finite same-grid waveforms")
    channels={"even":(R+L,pe),"odd":(R-L,po),"near_sum":(R,(pe+po)/2),"far_difference":(L,(pe-po)/2)}
    rows=[]; fig,axes=plt.subplots(2,2,figsize=(11,6),sharex=True,layout="constrained")
    for ax,(label,(data,pred)) in zip(axes.flat,channels.items()):
        for low,high in ((150.,300.),(190.,500.),(300.,500.)):
            take=(t>=low)&(t<=high)
            if not np.any(take): raise ValueError("insufficient WH observation duration")
            rms0=float(np.sqrt(np.mean(data[take]**2)))
            rms=float(np.sqrt(np.mean((data[take]-pred[take])**2)))
            rows.append(dict(channel=label,t_start=low,t_end=high,direct_rms=rms0,
                             raw_abs_rms=rms,raw_relative_L2=rms/rms0 if rms0 else np.nan,
                             raw_peak_ratio=max(abs(pred[take]))/max(abs(data[take])) if rms0 else np.nan))
        ax.plot(t,data,label="direct TD",lw=1.4); ax.plot(t,pred,"--",label="raw FD",lw=1)
        ax.set(title=label.replace("_"," "),ylabel="field correction",xlim=(115.,500.)); ax.grid(alpha=.2)
    for ax in axes[-1]: ax.set_xlabel("time / M")
    axes[0,0].legend(); fig.savefig(root/"wh_transfer_parity.png",dpi=150); plt.show(); plt.close(fig)
    parity=pd.DataFrame(rows); parity.to_csv(root/"wh_transfer_parity.csv",index=False)
    return checks,parity

In [ ]:
def interp_complex_pchip(xnew, x, z, fill=0j):
    xnew=np.asarray(xnew,float); x=np.asarray(x,float); z=np.asarray(z,complex)
    out=np.full(xnew.shape,fill,complex); q=(xnew>=x[0])&(xnew<=x[-1])
    if np.any(q):
        mag=PchipInterpolator(x,abs(z))(xnew[q])
        phase=PchipInterpolator(x,np.unwrap(np.angle(z)))(xnew[q])
        out[q]=mag*np.exp(1j*phase)
    return out


def transfer_comparison_metrics(t,pred,direct,t1=190,t2=500):
    q=(t>=t1)&(t<=t2)
    raw=float(np.linalg.norm(pred[q]-direct[q])/max(np.linalg.norm(direct[q]),1e-300))
    overlap=optimized_overlap(pred[q],direct[q])
    cc=correlate(hilbert(direct[q]),hilbert(pred[q]),mode="full",method="fft")
    nq=np.count_nonzero(q); dtloc=float(np.mean(np.diff(t)))
    lags=np.arange(-nq+1,nq)*dtloc; lag=float(lags[np.argmax(abs(cc))])
    shifted=CubicSpline(t,pred,extrapolate=False)(t-lag); ok=q&np.isfinite(shifted)
    zpred=hilbert(shifted[ok]); zdirect=hilbert(direct[ok])
    beta=np.vdot(zpred,zdirect)/max(float(np.vdot(zpred,zpred).real),1e-300)
    fitted=float(np.linalg.norm(beta*zpred-zdirect)
                 /max(np.linalg.norm(zdirect),1e-300))
    return {"raw_relative_L2":raw,"optimized_overlap":overlap,
            "best_time_shift":lag,"best_amplitude_scale":float(abs(beta)),
            "best_phase_shift":float(np.angle(beta)),
            "aligned_complex_relative_L2":fitted}



# 1% is the pre-set maximum raw late-time relative error (not an overlap target).
RAW_TARGET=.01
mu=CFG.gravastar_compactness; xs=gs.metadata["shell_x"]; xt=wh.metadata["rstar0"]
lam_th=wh.metadata["throat_Lambda"]; neutral_monitor=-65.*CFG.M
# Evaluate the last mutually covered sample, never extrapolating the GS TD record.
rt=bh.t[bh.t<=min(gs.t[-1],wh.t[-1],bh.t[-1])]
nr=len(rt); rdt=rt[1]-rt[0]; fft_n=1<<(4*nr-1).bit_length()
def build_geometric_responses(z,step=.035,outer=400.):
    g=make_seam_scattering(z,xs,CFG.extraction_x-xs,neutral_monitor,CFG.ell,CFG.M,outer,step)
    w=make_seam_scattering(z,xt,CFG.extraction_x,neutral_monitor,CFG.ell,CFG.M,outer,step)
    # Exteriors are identical functions of outward distance. Symmetry is
    # independently checked in signed global coordinates below.
    return g,w,gs_seam_transfer(g,gs_shell_reflectivity(z,mu,CFG.ell,CFG.M)),wh_seam_transfer(w,w,lam_th)

_C3_SCATTER={}
def characteristic_prediction(model,records,source_times,*,monitor=-65.,nf=32768,eta=.008,cutoff=2.,step=.025,outer=400.):
    """Only the independently measured ingoing characteristic drives K.

    All matching planes and source/observer positions remain unchanged. The
    outgoing monitor component is retained as a diagnostic, never added to I.
    """
    rec=records[monitor]
    parts=monitor_characteristics(source_times,rec['total'],rec['dpsi'],nf,eta)
    use=parts['omega'].real<=cutoff; z=parts['omega'][use]
    key=(model,nf,float(source_times[1]-source_times[0]),eta,cutoff,step,outer,monitor)
    if key not in _C3_SCATTER:
        origin=xs if model=='GS' else xt
        det=CFG.extraction_x-origin if model=='GS' else CFG.extraction_x
        sc=make_seam_scattering(z,origin,det,monitor,CFG.ell,CFG.M,outer,step)
        chain=gs_seam_transfer(sc,gs_shell_reflectivity(z,mu,CFG.ell,CFG.M)) if model=='GS' else wh_seam_transfer(sc,sc,lam_th)
        _C3_SCATTER[key]=(sc,chain)
    sc,chain=_C3_SCATTER[key]; waves={}; Ks={}
    for name,kname in [(model,'K')]+([('even','K_even'),('odd','K_odd')] if model=='WH' else []):
        K=np.zeros(parts['omega'].shape,complex); K[use]=chain[kname]
        waves[name]=np.exp(eta*rt)*np.fft.irfft(np.conj(K)*parts['ingoing_fft'],nf)[:nr]
        Ks[name]=K
    old=np.exp(eta*rt)*np.fft.irfft(np.conj(Ks[model])*parts['field_fft'],nf)[:nr]
    return dict(waves=waves,Ks=Ks,parts=parts,sc=sc,chain=chain,total_field_prediction=old,
                diagnostics=characteristic_diagnostics(parts,cutoff))

recon_eta=.008
c3_results={model:characteristic_prediction(model,records,source_t,nf=fft_n)
    for model,records in [('GS',gs_monitor_records),('WH',wh_monitor_records)]}
rwaves={k:v for result in c3_results.values() for k,v in result['waves'].items()}
rKs={k:v for result in c3_results.values() for k,v in result['Ks'].items()}
rfraction={k:r['diagnostics']['excluded_ingoing_spectral_power'] for k,r in c3_results.items()}
wcont=c3_results['WH']['sc'];wcont_chain=c3_results['WH']['chain']
# Padding only: all other settings are held fixed, unlike the previous joint test.
rwaves_alt={model:characteristic_prediction(model,records,source_t,nf=2*fft_n)['waves'][model]
    for model,records in [('GS',gs_monitor_records),('WH',wh_monitor_records)]}


gref,wref,gchain,wchain=build_geometric_responses(np.linspace(.15,.95,601),.025)
Rwh_eff=wh_chain["R_throat_effective"]  # older real-frequency grid: delay plot only

@dataclass(frozen=True)
class ReflectionDiagnostic:
    omega:np.ndarray
    R_eco:np.ndarray
    T_barrier:np.ndarray
    physical_definition:str
    reference_surface:str
    def phase(self): return np.unwrap(np.angle(self.R_eco))
    def delay(self): return np.gradient(self.phase(),self.omega,edge_order=2)

reflection_responses={
    "GS":ReflectionDiagnostic(gref.omega,gchain["R_eff"],gref.Tinf,"regular de Sitter interior and continuous axial shell data","X=r*-r*(a)=0"),
    "WH":ReflectionDiagnostic(wref.omega,wchain["R_eff"],wref.Tinf,"exact delta throat and opposite exterior","native x=0, both exterior bases outward"),
    "QH":ReflectionDiagnostic(wgrid,Rq_grid,Tbh,"OWA surface reflectivity","local x-xH; asymptotic barrier basis")}
undefined_reflections={"BS":"This stellar potential has no separable, physically specified inner reflection coefficient."}
reference_rows=[]
for key,ss,offset in (("GS",gref,0.),("WH",wref,xt)):
    s=ss.reference; peak=schwarzschild_rstar(rw_peak_radius(CFG.ell,CFG.M),CFG.M)-s.origin_rstar
    reference_rows.append(dict(model=key,seam_global_rstar=s.origin_rstar,seam_local_X=0.,
        barrier_peak_local=peak,opposite_peak_signed=(-peak if key=="WH" else np.nan),
        detector_local=s.detector_X,detector_global_rstar=CFG.extraction_x+offset,
        source_global_rstar=CFG.pulse_center+offset,source_local=CFG.pulse_center+offset-s.origin_rstar,
        BH_monitor_global_rstar=s.monitor_rstar,basis=s.basis,embedded_phases=s.embedded_phases,
        transmission="outgoing Jost field at finite observer; Tinf for flux only",
        fourier=s.time_basis))
refplanes=pd.DataFrame(reference_rows); refplanes.to_csv(ARTIFACT_DIR/"reconstruction_reference_planes.csv",index=False); display(refplanes)
aligned_waveforms={k:CubicSpline(wf.t,wf.psi,extrapolate=False)(bh.t) for k,wf in waveforms.items()}
direct_corrections={k:aligned_waveforms[k]-reference_common[k] for k in ["GS","WH"]}
direct_corrections["QH"]=td_owa_fine
right_raw=direct_corrections["WH"][:nr]; left_raw=CubicSpline(wh.t,wh_left,extrapolate=False)(rt)
wh_checks_df,wh_parity_df=audit_wh_transfer(wcont,wcont,wcont_chain,lam_th,rt,right_raw,left_raw,
    rwaves["even"],rwaves["odd"],artifact_dir=ARTIFACT_DIR)
late_mask=(bh.t>=190)&(bh.t<=500); prompt_mask=(bh.t>=100)&(bh.t<190)
prompt_per_model={k:float(np.max(abs(y[prompt_mask]))) for k,y in reference_common.items()}
A_prompt=prompt_per_model["BH"]
def scalar_at(w,x,y): return complex(np.interp(w,x,y.real)+1j*np.interp(w,x,y.imag))

transfer_predictions={**{k:rwaves[k] for k in ("GS","WH")}}
# Independently interpolate the complete K and drive it with measured ingoing data.
# Compare against a separate time-stepped boundary-history calculation.
K_qh_control=interp_complex_pchip(wfft,wgrid,surface_K(wgrid,Rbh,Tdet,Rq_grid,xH,xE))
transfer_predictions["QH"]=np.fft.irfft(np.conj(K_qh_control)*qh_ingoing_spectrum(qh_source_records[xH],nfft),nfft)[:nr]
transfer_rows=[]; stability_rows=[]
for key,pred in transfer_predictions.items():
    direct=direct_corrections[key][:nr]
    transfer_rows.append({"model":key,**transfer_comparison_metrics(rt,pred,direct)})
    if key in ("GS","WH"):
        mask=(rt>=190)&(rt<=500); early=rt<75
        stability_rows.append(dict(model=key,relative_L2_changed_numerics=np.linalg.norm((rwaves_alt[key]-pred)[mask])/np.linalg.norm(direct[mask]),
            precursor_peak_over_late_peak=np.max(abs(pred[early]))/np.max(abs(direct[mask])),
            omitted_incident_power_fraction=rfraction[key],nfft=fft_n,repeat_nfft=2*fft_n,eta=recon_eta,
            repeat_eta=recon_eta,cutoff=2.,repeat_cutoff=2.,radial_step=.025,repeat_radial_step=.025))
transfer_reconstruction_df=pd.DataFrame(transfer_rows)
transfer_reconstruction_df["window_and_scale"]="190<=t/M<=500; raw L2 / direct norm; no fitting; alignment columns diagnostic only"
stability_transfer_df=pd.DataFrame(stability_rows)
transfer_reconstruction_df.to_csv(ARTIFACT_DIR/"transfer_reconstruction_comparison.csv",index=False)
stability_transfer_df.to_csv(ARTIFACT_DIR/"reconstruction_numerical_sensitivity.csv",index=False)
display(transfer_reconstruction_df);display(stability_transfer_df)
qh_transfer_reconstruction_error=float(transfer_reconstruction_df.set_index("model").loc["QH","raw_relative_L2"])
np.savez_compressed(ARTIFACT_DIR/"reconstruction_records.npz",t=rt,source_t=source_t,fft_n=fft_n,
    **{model.lower()+'_'+key:val for model,records in [('GS',gs_monitor_records),('WH',wh_monitor_records)]
        for key,val in [('monitor_field',records[-65.]['total']),('monitor_derivative',records[-65.]['dpsi']),
            ('ingoing_fft',c3_results[model]['parts']['ingoing_fft']),('outgoing_fft',c3_results[model]['parts']['outgoing_fft']),
            ('total_fft',c3_results[model]['parts']['field_fft'])]},
    gs_total=aligned_waveforms["GS"][:nr],gs_bh=reference_common["GS"][:nr],
    wh_total=aligned_waveforms["WH"][:nr],wh_bh=reference_common["WH"][:nr],
    qh_control_K=K_qh_control,gs_K=rKs["GS"],wh_K=rKs["WH"],gs_eta=recon_eta,wh_eta=recon_eta,
    wh_left=left_raw,wh_even_K=rKs["even"],wh_odd_K=rKs["odd"],
    **{k.lower()+"_direct":direct_corrections[k][:nr] for k in transfer_predictions},
    **{k.lower()+"_prediction":v for k,v in transfer_predictions.items()})
assert (transfer_reconstruction_df.raw_relative_L2<RAW_TARGET).all(),transfer_reconstruction_df
assert (stability_transfer_df.relative_L2_changed_numerics<.001).all(),stability_transfer_df
assert (stability_transfer_df.precursor_peak_over_late_peak<.001).all(),stability_transfer_df
assert (stability_transfer_df.omitted_incident_power_fraction<1e-8).all(),stability_transfer_df

# Independent one-factor tests: none of these changes is tuned to align waves.
c3_rows=[]; c3_wave_records={}; late=(rt>=190)&(rt<=500)
def c3_trial(model,axis,value,records=None,times=None,reference=None,direct=None,**kw):
    records=(gs_monitor_records if model=='GS' else wh_monitor_records) if records is None else records
    times=source_t if times is None else times
    result=characteristic_prediction(model,records,times,nf=kw.pop('nf',fft_n),**kw)
    pred=result['waves'][model]; direct=direct_corrections[model][:nr] if direct is None else direct
    norm=np.linalg.norm(direct[late]); delta=pred-rwaves[model]
    row=dict(model=model,axis=axis,value=value,raw_relative_L2=np.linalg.norm((pred-direct)[late])/norm,
        prediction_change_relative_L2=np.linalg.norm(delta[late])/np.linalg.norm(direct_corrections[model][:nr][late]),
        **result['diagnostics'])
    c3_rows.append(row);c3_wave_records[f'{model}_{axis}_{value}']=pred
    return result

for model,offset,default in [('GS',0.,gs_monitor_records),('WH',xt,wh_monitor_records)]:
    print('Characteristic convergence:',model,flush=True)
    for monitor in (-55.,-65.,-75.):c3_trial(model,'monitor',monitor,monitor=monitor)
    for cutoff in (1.5,2.,2.4):c3_trial(model,'cutoff',cutoff,cutoff=cutoff)
    for eta in (.006,.008,.010):c3_trial(model,'eta',eta,eta=eta)
    for nf in (fft_n,2*fft_n):c3_trial(model,'padding',nf,nf=nf)
    for duration in (650.,700.):
        tt=rdt*np.arange(int(np.floor(duration/rdt))+1)
        ref,rr=matched_source_control(offset,sample_t=tt,t_max=duration)
        c3_trial(model,'input_duration',duration,records=rr,times=tt)
    for boundary in (-400.,-480.):
        ref,rr=matched_source_control(offset,sample_t=source_t,x_left=boundary)
        c3_trial(model,'source_left',boundary,records=rr)
    for dx in (.2,.1,.05):
        ref,rr=matched_source_control(offset,dx=dx,sample_t=source_t)
        wf=(run_gravastar(dx=dx,t_max=CFG.t_max+.2)[0] if model=='GS' else run_wormhole(dx=dx,t_max=CFG.t_max+.2)[0])
        direct=CubicSpline(wf.t,wf.psi,extrapolate=False)(rt)-CubicSpline(ref.t,ref.psi,extrapolate=False)(rt)
        result=c3_trial(model,'dx',dx,records=rr,direct=direct)
        c3_rows[-1]['direct_change_vs_production_L2']=np.linalg.norm((direct-direct_corrections[model][:nr])[late])/np.linalg.norm(direct[late])
characteristic_convergence_df=pd.DataFrame(c3_rows)
characteristic_convergence_df['norm_and_scope']='raw late [190,500] / direct norm; O/I is damped spectral amplitude, not energy'
characteristic_convergence_df.to_csv(ARTIFACT_DIR/'characteristic_convergence.csv',index=False)
np.savez_compressed(ARTIFACT_DIR/'characteristic_convergence_waveforms.npz',t=rt,**c3_wave_records)
display(characteristic_convergence_df)
assert (characteristic_convergence_df.raw_relative_L2<.01).all(),characteristic_convergence_df
assert (characteristic_convergence_df.prediction_change_relative_L2<.001).all(),characteristic_convergence_df
assert (characteristic_convergence_df.identity_relative_L2<1e-12).all()
assert (characteristic_convergence_df.derivative_identity_relative_L2<1e-12).all()
# Report a real-axis diagnostic separately: the ambiguous STATIC bin is excluded.
c3_real=[]
for model,records in [('GS',gs_monitor_records),('WH',wh_monitor_records)]:
    f=np.fft.rfft(records[-65.]['total'],fft_n); d=np.fft.rfft(records[-65.]['dpsi'],fft_n)
    w=2*np.pi*np.fft.rfftfreq(fft_n,rdt); band=(w>0)&(w<=2.)
    inc=.5*(f[band]+d[band]/(1j*w[band])); out=.5*(f[band]-d[band]/(1j*w[band]))
    c3_real.append(dict(model=model,real_axis_nonzero_band_amplitude_ratio=np.linalg.norm(out)/np.linalg.norm(inc),
        real_axis_nonzero_band_flux_ratio=np.sum(w[band]**2*abs(out)**2)/np.sum(w[band]**2*abs(inc)**2),
        static_DC_field_real=f[0].real,static_DC_derivative_real=d[0].real,
        static_policy='not directionally split; excluded only from real-axis diagnostic; production retains z=i eta'))
pd.DataFrame(c3_real).to_csv(ARTIFACT_DIR/'characteristic_real_axis_diagnostic.csv',index=False)
fig,axes=plt.subplots(2,3,figsize=(13,7),layout='constrained')
for ax,axis in zip(axes.flat,('monitor','dx','cutoff','eta','input_duration','source_left')):
    for model,g in characteristic_convergence_df.query('axis==@axis').groupby('model'):
        g=g.sort_values('value'); ax.semilogy(g.value,np.maximum(g.raw_relative_L2,1e-16),'o-',label=model)
    ax.set(title=axis,xlabel=axis,ylabel='raw relative L2');ax.legend()
fig.suptitle('One factor at a time; no amplitude, phase or time fit')
fig.savefig(ARTIFACT_DIR/'characteristic_convergence.png')


central_rows=[]
for key,response in reflection_responses.items():
    prompt_for_key=prompt_per_model[key]
    Rwc=scalar_at(wc,response.omega,response.R_eco)
    Twc=scalar_at(wc,wgrid,Tbh) if key=="QH" else scalar_at(wc,response.omega,response.T_barrier)
    phase=response.phase(); tau=response.delay(); signal=direct_corrections[key]
    Aecho=float(np.max(abs(signal[late_mask])))
    late_m=1-optimized_overlap(aligned_waveforms[key][late_mask],reference_common[key][late_mask])
    proxy=float(abs(Twc)**2*abs(Rwc))
    central_rows.append({
        "model":key,"reflection_defined":True,"abs_R_at_wc":abs(Rwc),
        "phi_R_at_wc":float(np.interp(wc,response.omega,phase)),
        "tau_R_at_wc":float(np.interp(wc,response.omega,tau)),
        "A_echo_over_A_prompt":Aecho/max(prompt_for_key,1e-300),
        "transmission_squared_times_abs_R":proxy,
        "observed_over_amplitude_proxy":Aecho/max(prompt_for_key*proxy,1e-300),
        "M_late":late_m,"physical_definition":response.physical_definition,
        "reference_surface":response.reference_surface,
    })

central_rows.append({
    "model":"BS","reflection_defined":False,"abs_R_at_wc":np.nan,
    "phi_R_at_wc":np.nan,"tau_R_at_wc":np.nan,"T_echo":np.nan,
    "T_recurrence":np.nan,
    "A_echo_over_A_prompt":float(np.max(abs((aligned_waveforms["BS"]-reference_common["BS"])[late_mask]))/prompt_per_model["BS"]),
    "transmission_squared_times_abs_R":np.nan,
    "observed_over_amplitude_proxy":np.nan,
    "M_late":1-optimized_overlap(aligned_waveforms["BS"][late_mask],reference_common["BS"][late_mask]),
    "physical_definition":undefined_reflections["BS"],
    "reference_surface":"not applicable",
})
eco_central_comparison_df=pd.DataFrame(central_rows).rename(columns={"A_echo_over_A_prompt":"A_late_residual_over_BH_prompt"})
eco_central_comparison_df["window_and_normalization"]="max |ECO-BH| in [190,500] / max |BH| in [100,190); QH: independent TD"
print("Timing and classifications join these measurements in section 13.")
display(eco_central_comparison_df.drop(columns=["T_echo","T_recurrence"],errors="ignore"))


# Required amplitude/phase/group-delay figure with the actual packet carrier.
fig,ax=plt.subplots(3,1,figsize=(11,11),sharex=True)
for key,response in reflection_responses.items():
    band=(response.omega>=.15)&(response.omega<=.9)
    ax[0].plot(response.omega[band],abs(response.R_eco[band]),label=key)
    ax[1].plot(response.omega[band],response.phase()[band],label=key)
    ax[2].plot(response.omega[band],response.delay()[band],label=key)
for i,a in enumerate(ax):
    a.axvline(wc,c="k",ls="--",lw=1,label=r"$\omega_c$" if i==0 else None)
    a.grid(alpha=.2)
ax[0].set(ylabel=r"$|R_{\rm ECO}|$",title="Physically defined ECO reflection responses")
ax[0].legend(ncol=4); ax[1].set(ylabel=r"$\arg R_{\rm ECO}$")
ax[2].set(ylabel=r"$d\arg R_{\rm ECO}/d\omega$",xlabel=r"$M\omega$",ylim=(-100,350))
fig.tight_layout(); fig.savefig(ARTIFACT_DIR/"unified_eco_reflection_response.png")


fig,ax=plt.subplots(3,1,figsize=(12,9),sharex=True)
for a,key in zip(ax,["GS","WH","QH"]):
    scale=max(np.max(abs(direct_corrections[key][late_mask])),1e-300)
    a.plot(rt,direct_corrections[key][:nr]/scale,label="direct TD correction",lw=.9)
    a.plot(rt,transfer_predictions[key]/scale,
           label="frequency-response prediction",lw=.8,alpha=.8)
    a.set(ylabel=key,title=f"{key}: direct versus transfer reconstruction")
    a.legend(fontsize=8)
ax[-1].set(xlim=(120,500),xlabel=r"$t/M$")
fig.suptitle("Both traces divided by the direct-correction peak in [190,500] M; no alignment"); fig.tight_layout(rect=[0,0,1,.96]); fig.savefig(ARTIFACT_DIR/"transfer_vs_direct_waveforms.png")


# Wormhole timing discrepancy audit.
tau_barrier=np.gradient(np.unwrap(np.angle(Rhalf)),wgrid,edge_order=2)
tau_throat=np.gradient(np.unwrap(np.angle(Rwh_eff)),wgrid,edge_order=2)
wt=phase_weight[relevant]

def packet_weighted(z):
    return float(np.sum(wt*z[relevant])/np.sum(wt))

Tbarrier_weighted=packet_weighted(tau_barrier)
Tthroat_weighted=packet_weighted(tau_throat)
even_roots=wormhole_qnm_map.loc[
    wormhole_qnm_map.parity=="even","omega_R"].to_numpy()
odd_roots=wormhole_qnm_map.loc[
    wormhole_qnm_map.parity=="odd","omega_R"].to_numpy()

Tqnm_even=family_times["even"]; Tqnm_odd=family_times["odd"]
window_rows=[]
for tend in [300.,380.,460.,float(wh.t[-1])]:
    q=wh.t<=tend
    tr,_=recurrence_delay(
        wh.t[q],wh.psi[q],template_window[0],abs(Tweighted),.38)
    window_rows.append({"observation_end":tend,"T_recurrence":tr})
wormhole_window_sensitivity_df=pd.DataFrame(window_rows)
wormhole_timing_audit_df=pd.DataFrame([
    {"quantity":"naive geometric round trip","value":Tgeom,"role":"geometry only"},
    {"quantity":"barrier phase contribution","value":Tbarrier_weighted,
     "role":"packet-weighted d arg R_half/domega"},
    {"quantity":"effective throat contribution","value":Tthroat_weighted,
     "role":"delta throat plus opposite exterior"},
    {"quantity":"full loop group delay","value":Tweighted,
     "role":"sum of phase derivatives across packet"},
    {"quantity":"matched-packet recurrence","value":Tmatched,
     "role":"mixed return-family peak estimator"},
    {"quantity":"autocorrelation recurrence","value":Tacf,
     "role":"finite-window estimator"},
    {"quantity":"envelope recurrence","value":Tenvelope,
     "role":"envelope-only diagnostic, not classifier"},
    {"quantity":"even-family QNM spacing time","value":Tqnm_even,
     "role":"same-parity resonance family"},
    {"quantity":"odd-family QNM spacing time","value":Tqnm_odd,
     "role":"same-parity resonance family"},
    {"quantity":"packet-bandwidth delay sigma","value":bandwidth_delay_sigma,
     "role":"frequency spread"},
])
wormhole_simple_echo_picture_valid=bool(
    wh_quality["separated_pulses"] and wh_quality["stable_spacing"]
    and wormhole_spacing_resolution_fraction<.05
    and wormhole_delay_error<.10 and wormhole_qnm_spacing_error<.10)
wormhole_timing_conclusion=(
    "single echo-time picture supported" if wormhole_simple_echo_picture_valid
    else "multiple overlapping return/resonance families; retain T_recurrence and do not report T_echo")
display(wormhole_timing_audit_df)
display(wormhole_window_sensitivity_df)
print("Timing conclusion:",wormhole_timing_conclusion)

fig,ax=plt.subplots(2,2,figsize=(15,9))
plotrows=wormhole_timing_audit_df.iloc[:10]
ax[0,0].barh(plotrows.quantity,plotrows.value)
ax[0,0].set(title="Wormhole timing scales",xlabel=r"$T/M$")
ax[0,1].plot(wgrid,tau_barrier,label="barrier phase")
ax[0,1].plot(wgrid,tau_throat,label="throat/opposite exterior")
ax[0,1].plot(wgrid,tau_w,"k--",label="full loop")
ax[0,1].axvline(wc,c="k",ls=":")
ax[0,1].set(xlim=(.15,.9),ylim=(-100,250),
            title="Group-delay decomposition",xlabel=r"$M\omega$")
ax[0,1].legend(fontsize=8)
ax[1,0].plot(wh.t,wh.psi,lw=.7)
ax[1,0].scatter(wh_packets.time,wh_packets.amplitude,s=20,c="tab:red",
                label="matched candidates")
ax[1,0].set(xlim=(120,430),title="Overlapping return candidates",xlabel=r"$t/M$")
ax[1,0].legend()
for (parity,g),color in zip(wormhole_qnm_map.groupby("parity"),
                            ["tab:blue","tab:orange"]):
    ax[1,1].vlines(g.omega_R,0,np.exp(g.omega_I*40),
                   colors=color,label=parity)
ax[1,1].axvline(wc,c="k",ls="--")
ax[1,1].set(xlim=(.1,1.05),title="Interlaced QNM families",xlabel=r"$M\omega_R$")
ax[1,1].legend()
fig.tight_layout(); fig.savefig(ARTIFACT_DIR/"wormhole_timing_discrepancy_audit.png")


fig,ax=plt.subplots(figsize=(7,5))
finite=eco_central_comparison_df[eco_central_comparison_df.reflection_defined]
ax.scatter(finite.transmission_squared_times_abs_R,
           finite.A_late_residual_over_BH_prompt,s=60)
for _,row in finite.iterrows():
    ax.annotate(row.model,
                (row.transmission_squared_times_abs_R,row.A_late_residual_over_BH_prompt),
                xytext=(5,5),textcoords="offset points")
lo=min(finite.transmission_squared_times_abs_R.min(),finite.A_late_residual_over_BH_prompt.min())
hi=max(finite.transmission_squared_times_abs_R.max(),finite.A_late_residual_over_BH_prompt.max())
ax.plot([lo,hi],[lo,hi],"k--",label="unit proportionality")
ax.set(xlabel=r"$|T'_{BH}(\omega_c)|^2|R_{ECO}(\omega_c)|$",
       ylabel=r"$A_{late,res}/A_{BH,prompt}$",
       title="Late residual peak [190,500] / BH prompt peak [100,190)")
ax.legend(); fig.tight_layout()
fig.savefig(ARTIFACT_DIR/"reflection_echo_amplitude_connection.png")


## 11. BH-subtracted residuals, masked instantaneous frequency, mismatch, and energy flux

Two different questions are retained:

- **intrinsic morphology**: a white-noise, time/phase-optimized shape mismatch;
- **detector distinguishability proxy**: the same windows weighted by an analytic aLIGO design PSD after mapping $M$ to a stated $60M_\odot$ total mass.


The evolved odd master field is the Regge-Wheeler variable $Q=f h_1/r$ in the
exterior, not the Cunningham-Price-Moncrief (CPM) strain master. In vacuum
$Q=\tfrac12\partial_t\Psi_{CPM}$ up to sign. Hence the **radiation-limit**
luminosity per mode is $(\ell+2)!/(16\pi(\ell-2)!)|Q|^2$, equivalently
$(\ell+2)!/(64\pi(\ell-2)!)|\dot\Psi_{CPM}|^2$; applying a second time
derivative to the RW waveform would be incorrect. We evaluate this at a finite,
large observer radius with unit incident amplitude; these are not calibrated
energies at null infinity. See [Martel & Poisson](https://doi.org/10.1103/PhysRevD.71.104003)
and [Macedo et al., Sec. VI.1 (axial flux)](https://arxiv.org/html/1307.4812#S6.SS1).

Intrinsic mismatches compare RW fields. For the detector comparison the windowed
RW amplitudes acquire the frequency-domain integration factor $1/(2\pi i f)$
in the positive detector band. Its squared modulus enters both norms and the
cross spectrum; no DC division, unknown distance factor, or fitted amplitude
is required. These are analytic design-PSD comparisons, not detection claims.


All cross-model comparisons below use **one controlled waveform bank**: equal
ADM mass, multipole, unit ingoing Gaussian strength and width, initial time,
source areal radius and detector areal radius. The WH throat-centered and BS
center-based coordinates are inverted to locate those same radii. Native
TD/FD-validation experiments above are retained separately and are not mixed
into this mismatch matrix. Every exported matrix pair is reconstructed from
the saved controlled bank, including pairs with no BH member.


In [ ]:
# The earlier generic derivative-squared master-flux helper is for CPM input.
# Here the evolved odd variable is Q_RW; use the vacuum radiation-limit formula.
def rw_radiation_flux(wf,ell=2):
    return math.factorial(ell+2)/math.factorial(ell-2)/(16*np.pi)*np.abs(wf.psi)**2

def rw_strain_detector_overlap(a,b,dt_M,total_mass_solar=60.0,fmin=20.0,fmax=1024.0):
    from scipy.signal.windows import tukey
    n=len(a); nfft=1<<(2*n-1).bit_length(); Msec=4.925490947e-6*total_mass_solar; dt=dt_M*Msec
    win=tukey(n,.1); A=np.fft.rfft((a-np.mean(a))*win,nfft)*dt; B=np.fft.rfft((b-np.mean(b))*win,nfft)*dt
    f=np.fft.rfftfreq(nfft,dt); band=(f>=fmin)&(f<=min(fmax,f[-1])); Sn=aligo_design_psd(f)*(2*np.pi*f)**2  # Q_RW = 0.5*d(CPM)/dt; f=0 excluded by band
    df=f[1]-f[0]; aa=4*df*np.sum(abs(A[band])**2/Sn[band]); bb=4*df*np.sum(abs(B[band])**2/Sn[band])
    q=np.zeros(nfft,complex); q[np.flatnonzero(band)]=A[band]*np.conj(B[band])/Sn[band]
    corr=4*df*nfft*np.fft.ifft(q)
    return float(np.clip(np.max(abs(corr))/np.sqrt(max(aa*bb,1e-300)),0,1))


# A distinct, controlled experiment for comparisons across DIFFERENT models.
# The native production bank remains untouched for its TD/FD reconstruction.
# All masses are scaled to ADM M=1, with the same source/observer areal radii.
def boson_native_tortoise_at_radius(bg, radius_over_M):
    """Invert the actual equilibrium tortoise map; continue its vacuum exterior."""
    rr=np.asarray(radius_over_M,float)
    if np.any(rr<=0): raise ValueError("Positive areal radius required")
    rM=bg.r/bg.mass
    xM=cumulative_trapezoid(bg.a/bg.alpha,bg.r,initial=0.)/bg.mass
    flat=rr.reshape(-1); xx=np.empty_like(flat)
    inside=flat<=rM[-1]
    if np.any(flat[inside]<rM[0]): raise ValueError("Radius below background grid")
    xx[inside]=PchipInterpolator(rM,xM,extrapolate=False)(flat[inside])
    if np.any(~inside):
        xx[~inside]=xM[-1]+schwarzschild_rstar(flat[~inside])-schwarzschild_rstar(rM[-1])
    return xx.reshape(rr.shape)


def build_controlled_comparison_bank():
    if CFG.M!=1.: raise ValueError("BS comparison uses ADM-scaled M=1 coordinates")
    source_global=float(CFG.pulse_center); observer_global=float(CFG.extraction_x)
    radii=r_from_schwarzschild_rstar(np.array([source_global,observer_global]),1.)
    wh_offset=float(wh.metadata["rstar0"])
    wh_cfg=replace(CFG,pulse_center=source_global-wh_offset,extraction_x=observer_global-wh_offset)
    wh_control,*_=run_wormhole(cfg=wh_cfg,extraction_xs=(wh_cfg.extraction_x,))
    bs_positions=boson_native_tortoise_at_radius(bs_bg,radii)
    bs_cfg=replace(CFG,pulse_center=float(bs_positions[0]),extraction_x=float(bs_positions[1]))
    bs_control,*_=run_boson_star(cfg=bs_cfg)
    originals={"BH":bh,"GS":gs,"BS":bs_control,"WH":wh_control,"QH":qh}
    native_positions={"BH":(source_global,observer_global),"GS":(source_global,observer_global),
        "BS":tuple(bs_positions),"WH":(wh_cfg.pulse_center,wh_cfg.extraction_x),
        "QH":(source_global,observer_global)}
    bank={}; geometry=[]
    for name,wave in originals.items():
        sx,ox=map(float,native_positions[name])
        # QH is a linear response to the explicitly measured BH experiment;
        # it has no separate Cauchy-data initialization of its own.
        source_metadata=bh.metadata if name=="QH" else wave.metadata
        source_initial_F_in=float(source_metadata["F_ingoing_left_u_plus"] if
            "F_ingoing_left_u_plus" in source_metadata else source_metadata["F_in"])
        if name=="BS":
            # Independent inverse of the coordinate map used to set positions.
            rM=bs_bg.r/bs_bg.mass
            xM=cumulative_trapezoid(bs_bg.a/bs_bg.alpha,bs_bg.r,initial=0.)/bs_bg.mass
            actual=[]
            for xx in (sx,ox):
                if xx<=xM[-1]:
                    lo=max(0,int(np.searchsorted(xM,xx))-1); hi=min(len(rM)-1,lo+1)
                    actual.append(brentq(lambda rr:float(boson_native_tortoise_at_radius(bs_bg,rr))-xx,
                                         float(rM[lo]),float(rM[hi]),xtol=1e-12))
                else:
                    actual.append(float(r_from_schwarzschild_rstar(schwarzschild_rstar(rM[-1])+xx-xM[-1])))
            actual=np.asarray(actual)
        else:
            offset=wh_offset if name=="WH" else 0.
            actual=r_from_schwarzschild_rstar(np.array([sx,ox])+offset,1.)
        meta={**wave.metadata,"comparison_bank":"common_ADM_mass_and_areal_radii",
            "comparison_ADM_mass":1.,"comparison_source_native_tortoise":sx,
            "comparison_observer_native_tortoise":ox,
            "comparison_source_areal_radius_over_M":float(actual[0]),
            "comparison_observer_areal_radius_over_M":float(actual[1]),
            "comparison_source_global_Schwarzschild_rstar":source_global,
            "comparison_observer_global_Schwarzschild_rstar":observer_global,
            "comparison_initial_data":"unit Gaussian, left-going; fixed native tortoise width",
            "comparison_pulse_width_M":float(CFG.pulse_width),
            "comparison_initial_F_in":source_initial_F_in,
            "comparison_initial_characteristic_source":"BH source experiment" if name=="QH" else "own measured initial data",
            "comparison_time_alignment":"initial Cauchy surface t=0; no fitted shifts"}
        bank[name]=Waveform(wave.model,wave.t,wave.psi,wave.x_extract,meta)
        geometry.append(dict(model=name,ADM_mass=1.,source_native=sx,observer_native=ox,
            source_areal_radius_over_M=actual[0],observer_areal_radius_over_M=actual[1],
            source_radius_error=abs(actual[0]-radii[0]),observer_radius_error=abs(actual[1]-radii[1]),
            source_global_rstar=source_global,observer_global_rstar=observer_global,
            pulse_width_M=CFG.pulse_width,initial_F_in=source_initial_F_in,
            initial_time=float(wave.t[0]),last_time=float(wave.t[-1])))
    geometry=pd.DataFrame(geometry)
    assert geometry.source_radius_error.max()<1e-9
    assert geometry.observer_radius_error.max()<1e-9
    assert np.allclose(geometry.initial_F_in,1.,rtol=0,atol=1e-12)
    assert np.all(geometry.initial_time==0.) and np.all(geometry.last_time>=500.)
    assert np.all(geometry.ADM_mass==1.) and np.all(geometry.pulse_width_M==CFG.pulse_width)
    assert all(np.isfinite(w.psi).all() and np.all(np.diff(w.t)>0) for w in bank.values())
    geometry.to_csv(ARTIFACT_DIR/"controlled_comparison_geometry.csv",index=False)
    return bank,geometry


comparison_waveforms,comparison_geometry=build_controlled_comparison_bank()
comparison_reference_native={name:comparison_waveforms["BH"] for name in comparison_waveforms}
comparison_reference_common={name:bh.psi for name in comparison_waveforms}
_comparison_prompt=(bh.t>=100)&(bh.t<190)
comparison_prompt_per_model={name:float(np.max(abs(bh.psi[_comparison_prompt]))) for name in comparison_waveforms}
display(comparison_geometry)


names=list(comparison_waveforms); t_common=bh.t


def aligned_to_bh(wf):
    return CubicSpline(wf.t,wf.psi,extrapolate=False)(t_common)


# Every ECO gets an aligned BH subtraction and early/late metrics.
residual_rows=[]; residual_series={}; instfreq_rows=[]
early_common=(t_common>=100)&(t_common<190); late_common=(t_common>=190)&(t_common<=500)
for name,wf in comparison_waveforms.items():
    yy=aligned_to_bh(wf); residual=yy-comparison_reference_common[name]; residual_series[name]=residual
    re=float(np.sqrt(np.mean(residual[early_common]**2))); rl=float(np.sqrt(np.mean(residual[late_common]**2)))
    residual_rows.append({"model":name,"residual_rms_early":re,"residual_rms_late":rl,"late_RMS_over_BH_prompt_peak":rl/comparison_prompt_per_model[name],"BH_prompt_peak":comparison_prompt_per_model[name],"windows_M":"early [100,190); late [190,500]"})
    for eps in [1e-2,1e-3,1e-4]:
        fi=wf.masked_instantaneous_frequency(eps)
        for label,window in [("prompt",(wf.t>=100)&(wf.t<190)),("late",(wf.t>=190)&(wf.t<=500))]:
            valid=np.isfinite(fi)&window
            instfreq_rows.append({"model":name,"window":label,"time_M":"[100,190)" if label=="prompt" else "[190,500]","epsilon_of_global_envelope_peak":eps,
                "valid_fraction_in_window":valid.sum()/window.sum(),"valid_samples":int(valid.sum()),
                "masked_IQR_cycles_per_M":float(np.ptp(np.percentile(fi[valid],[25,75]))) if valid.any() else np.nan})
residual_df=pd.DataFrame(residual_rows).set_index("model"); instfreq_sensitivity_df=pd.DataFrame(instfreq_rows)
display(residual_df); display(instfreq_sensitivity_df)

figr,ar=plt.subplots(4,4,figsize=(19,13))
for row,name in enumerate(["GS","BS","WH","QH"]):
    wf=comparison_waveforms[name]; yy=aligned_to_bh(wf); rr=residual_series[name]
    ar[row,0].plot(t_common,comparison_reference_common[name],lw=.7,label="source-matched BH"); ar[row,0].plot(t_common,yy,lw=.7,label=name); ar[row,0].set(xlim=(100,500),title=f"BH and {name}"); ar[row,0].legend(fontsize=7)
    ar[row,1].plot(t_common,rr,lw=.7); ar[row,1].set(xlim=(100,500),title=f"{name} - BH")
    m=(t_common>=100)&(t_common<=500); f,A,_=spectrum(t_common[m],rr[m]); ar[row,2].semilogy(2*np.pi*f,np.maximum(A,1e-16)); ar[row,2].set(xlim=(0,1.2),title="Tukey residual spectrum",xlabel=r"$M\omega$")
    ar[row,3].plot(wf.t,wf.masked_instantaneous_frequency(1e-3),lw=.6); ar[row,3].set(xlim=(100,500),ylim=(0,.25),title=r"masked $f_{inst}$, $\epsilon=10^{-3}$")
for a in ar.ravel(): a.set_xlabel(a.get_xlabel() or r"$t/M$")
figr.suptitle("Residuals and Hilbert morphology, 100–500 M: show f only where |analytic signal| > 0.001 × global envelope peak; not QNM fits"); figr.tight_layout(rect=[0,0,1,.97]); figr.savefig(ARTIFACT_DIR/"bh_subtracted_residual_diagnostics.png")


def mismatch_matrix(t1,t2,detector=False):
    sig={n:windowed_signal(wf,t_common,t1,t2)[1] for n,wf in comparison_waveforms.items()}; out=np.zeros((len(names),len(names)))
    dtM=float(np.mean(np.diff(t_common[(t_common>=t1)&(t_common<=t2)])))
    for i,a in enumerate(names):
        for j,b in enumerate(names):
            aa,bb=sig[a],sig[b]
            ov=rw_strain_detector_overlap(aa,bb,dtM,CFG.detector_total_mass_solar) if detector else optimized_overlap(aa,bb)
            out[i,j]=1-ov
    return out


prompt_mismatch=mismatch_matrix(100,190); late_mismatch=mismatch_matrix(190,500)
prompt_detector_mismatch=mismatch_matrix(100,190,True); late_detector_mismatch=mismatch_matrix(190,500,True)
energy=[]
for name,wf in comparison_waveforms.items():
    flux=rw_radiation_flux(wf,CFG.ell); p=(wf.t>=100)&(wf.t<190); l=(wf.t>=190)&(wf.t<=500)
    Ep=np.trapezoid(flux[p],wf.t[p]); El=np.trapezoid(flux[l],wf.t[l]); Et=np.trapezoid(flux[(wf.t>=100)&(wf.t<=500)],wf.t[(wf.t>=100)&(wf.t<=500)])
    energy.append({"model":name,"prompt_flux_energy":Ep,"late_flux_energy":El,"E_prompt":Ep,"E_late":El,
                   "late_over_prompt_flux_energy":El/Ep,"late_fraction":El/Et})
energy_df=pd.DataFrame(energy).set_index("model"); energy_df["excess_late_vs_BH"]=energy_df.late_flux_energy-energy_df.loc["BH","late_flux_energy"]
for name in ("GS","WH"):
    ref=comparison_reference_native[name]; m=(ref.t>=190)&(ref.t<=500)
    energy_df.loc[name,"excess_late_vs_BH"]=energy_df.loc[name,"late_flux_energy"]-np.trapezoid(rw_radiation_flux(ref,CFG.ell)[m],ref.t[m])
qh_corr_flux=math.factorial(CFG.ell+2)/math.factorial(CFG.ell-2)/(16*np.pi)*np.abs(qh_echo)**2
energy_df.loc["QH","isolated_correction_late_energy"]=np.trapezoid(qh_corr_flux[late_common],qh.t[late_common])
display(energy_df); print("Intrinsic late mismatch"); display(pd.DataFrame(late_mismatch,index=names,columns=names).round(6))
print("Detector-weighted late mismatch (analytic aLIGO PSD, 60 Msun)"); display(pd.DataFrame(late_detector_mismatch,index=names,columns=names).round(6))

fig,ax=plt.subplots(2,2,figsize=(12,10))
for a,mat,title in zip(ax.ravel(),[prompt_mismatch,late_mismatch,prompt_detector_mismatch,late_detector_mismatch],
                       ["Intrinsic prompt","Intrinsic late","PSD-weighted prompt","PSD-weighted late"]):
    vmax=max(.01,float(mat.max())); im=a.imshow(mat,vmin=0,vmax=vmax,cmap="magma"); a.set(xticks=range(len(names)),yticks=range(len(names)),xticklabels=names,yticklabels=names,title=title)
    for i in range(len(names)):
        for j in range(len(names)): a.text(j,i,f"{mat[i,j]:.3f}",ha="center",va="center",fontsize=8,color="white" if mat[i,j]>.55*vmax else "black")
    fig.colorbar(im,ax=a,fraction=.046)
fig.tight_layout(); fig.savefig(ARTIFACT_DIR/"intrinsic_detector_mismatch.png")

fig,ax=plt.subplots(figsize=(11,4))
for name,wf in comparison_waveforms.items(): ax.semilogy(wf.t,np.maximum(rw_radiation_flux(wf,CFG.ell),1e-18),lw=.7,label=name)
ax.set(xlim=(100,500),xlabel=r"$t/M$",ylabel=r"$dE/dt$ (master normalization)",title="Common source/observer experiment: prompt and late master-function flux"); ax.legend(ncol=3)
fig.tight_layout(); fig.savefig(ARTIFACT_DIR/"energy_flux.png")

# Compare exactly the samples admitted by each threshold, including the isolated
# correction, which need not share the amplitude scale of the total QH waveform.
if_signals={**comparison_waveforms,"QH correction":Waveform("isolated correction",qh.t,qh_echo,qh.x_extract,{})}
if_rows=[]
fig,axes=plt.subplots(2,3,figsize=(13,6),sharex=True,layout="constrained")
for ax,(label,wave) in zip(axes.flat,if_signals.items()):
    for eps,color in [(1e-4,"#aaaaaa"),(1e-3,"#1984b6"),(1e-2,"#e77726")]:
        f=wave.masked_instantaneous_frequency(eps); win=(wave.t>=190)&(wave.t<=500)
        allowed=wave.amplitude>eps*wave.amplitude.max()
        assert np.all(allowed[np.isfinite(f)])
        valid=np.isfinite(f)&win
        if_rows.append(dict(model=label,epsilon=eps,window="190<=t/M<=500",normalization="own global analytic-envelope peak",
                            n_window=int(win.sum()),n_valid=int(valid.sum()),retained_fraction=valid.sum()/win.sum()))
        ax.plot(wave.t[win],f[win],lw=.75,color=color,label=f"{eps:g}")
    ax.set(title=label,xlabel="t/M",ylabel="cycles/M",ylim=(-.05,.2))
axes.flat[0].legend(title="threshold",fontsize=7)
fig.suptitle("Hilbert frequency, 190–500 M: retain only stencils above the stated fraction of each signal's envelope peak. Not QNM estimates.")
fig.savefig(ARTIFACT_DIR/"hilbert_threshold_comparison.png",bbox_inches="tight")
if_evidence=pd.DataFrame(if_rows); if_evidence.to_csv(ARTIFACT_DIR/"hilbert_mask_counts.csv",index=False)


"""Controlled-bank spectra: reproducible PSD normalization, leakage and distances."""
from scipy.signal import stft

def positive_periodogram(samples,step,taper,padding=4):
    y=np.asarray(samples,float); win=np.asarray(taper,float)
    if step<=0 or len(y)<4 or y.shape!=win.shape: raise ValueError('invalid spectral grid')
    sig=(y-y.mean())*win
    n=1<<(int(padding*len(sig))-1).bit_length()
    bins=np.fft.rfftfreq(n,step); transform=step*np.fft.rfft(sig,n)
    psd=abs(transform)**2/(step*np.sum(win**2)); psd[1:-1]*=2.
    expected=np.sum(sig**2)/np.sum(win**2)
    observed=psd.sum()*(bins[1]-bins[0])
    if not np.isclose(expected,observed,rtol=3e-12,atol=1e-30):raise ArithmeticError('PSD Parseval normalization')
    return bins,transform,psd,abs(expected-observed)/max(expected,1e-300)

def audit_controlled_spectra(bank):
    rows=[];dist=[]; arrays={}
    fig,axes=plt.subplots(2,1,figsize=(9,7),layout='constrained')
    for lo,hi in [(100.,190.),(190.,420.),(190.,500.)]:
        tt=bh.t[(bh.t>=lo)&(bh.t<=hi)];h=float(tt[1]-tt[0]);freq_res=1/(len(tt)*h)
        # Offset by a noninteger number of record bins to expose leakage. The
        # four-bin width is fixed in intrinsic resolution, not padded grid size.
        carrier=(max(3,round(.4/(2*np.pi*freq_res)))+.37)*freq_res
        injection=np.cos(2*np.pi*carrier*(tt-tt[0]))
        for alpha in (.15,1.):
            taper=tukey(len(tt),alpha)
            f,_,p,_=positive_periodogram(injection,h,taper)
            leakage=p[abs(f-carrier)>4*freq_res].sum()/p.sum()
            by_model={}
            for name,wf in bank.items():
                y=CubicSpline(wf.t,wf.psi,extrapolate=False)(tt)
                f,X,P,err=positive_periodogram(y,h,taper)
                support=(2*np.pi*f>.04)&(2*np.pi*f<1.3)
                assert np.any(support) and np.max(P[support])>0
                peak=float(f[support][np.argmax(P[support])]*2*np.pi)
                # Fourier phases are relative to the window start; do not plot
                # phase where the magnitude is below the declared floor.
                phase=np.angle(X);phase[abs(X)<1e-4*np.max(abs(X))]=np.nan
                rows.append(dict(model=name,t_start=lo,t_end=hi,tukey_alpha=alpha,
                    resolution_cycles_per_M=freq_res,padded_bin_spacing=f[1]-f[0],
                    peak_Momega=peak,parseval_relative_error=err,
                    calibration_tone_leakage_outside_four_bins=leakage))
                if hi==500 and alpha==.15:
                    arrays[name+'_windowed_fourier']=X;arrays[name+'_one_sided_PSD']=P
                    arrays[name+'_phase_local_time_origin']=phase;arrays['frequency_cycles_per_M']=f
                    axes[0].semilogy(2*np.pi*f,np.maximum(P,1e-32),label=name,color=colors[name]);
                    axes[1].plot(2*np.pi*f,phase,'.',ms=1.4,color=colors[name],label=name)
                _,_,spec=stft(y-y.mean(),fs=1/h,window='hann',nperseg=256,noverlap=192,
                    boundary=None,padded=False)
                by_model[name]=(y,np.sqrt(P),abs(spec))
            b0,amp0,tf0=by_model['BH']
            for name,(y,amp,tf) in by_model.items():
                # No shifts/fits/per-model normalization in these distances.
                z=hilbert(y); z0=hilbert(b0); mask=(abs(z)>1e-3*abs(z).max())&(abs(z0)>1e-3*abs(z0).max())
                ph=np.angle(z*np.conj(z0)); wt=abs(z[mask])*abs(z0[mask])
                use=mask[1:]&mask[:-1];dfreq=np.angle(z[1:]*np.conj(z[:-1]))-np.angle(z0[1:]*np.conj(z0[:-1]))
                dist.append(dict(model=name,t_start=lo,t_end=hi,tukey_alpha=alpha,
                    relative_L2_vs_BH=np.linalg.norm(y-b0)/np.linalg.norm(b0),max_abs_difference=np.max(abs(y-b0)),
                    amplitude_spectral_distance_vs_BH=np.linalg.norm(amp-amp0)/np.linalg.norm(amp0),
                    STFT_magnitude_distance_vs_BH=np.linalg.norm(tf-tf0)/np.linalg.norm(tf0),
                    amplitude_weighted_wrapped_phase_RMS=np.sqrt(np.average(ph[mask]**2,weights=wt)) if wt.sum()>0 else np.nan,
                    valid_frequency_intervals=int(use.sum()),
                    masked_IF_difference_RMS_cycles_per_M=float(np.sqrt(np.mean(dfreq[use]**2))/(2*np.pi*h)) if use.any() else np.nan))
    axes[0].set(xlim=(0,1.5),ylim=(1e-15,1e2),ylabel='one-sided RW PSD',title='Controlled bank, 190–500 M; same taper and scales');axes[0].legend(ncol=5)
    axes[1].set(xlim=(0,1.5),xlabel=r'$M\omega$',ylabel='masked Fourier phase [rad]')
    fig.savefig(ARTIFACT_DIR/'controlled_spectra.png');plt.show();plt.close(fig)
    a=pd.DataFrame(rows);d=pd.DataFrame(dist)
    assert a.parseval_relative_error.max()<3e-12
    assert np.max(np.abs(d.query("model=='BH'")[['relative_L2_vs_BH','STFT_magnitude_distance_vs_BH']].values))==0
    assert np.all((a.calibration_tone_leakage_outside_four_bins>=0)&(a.calibration_tone_leakage_outside_four_bins<1))
    a.to_csv(ARTIFACT_DIR/'controlled_spectral_windows.csv',index=False);d.to_csv(ARTIFACT_DIR/'controlled_distances.csv',index=False)
    np.savez_compressed(ARTIFACT_DIR/'controlled_late_spectra.npz',**arrays)
    return a,d

spectral_windows_df,controlled_distances_df=audit_controlled_spectra(comparison_waveforms)
display(spectral_windows_df.query('t_end==500 and tukey_alpha==.15'))


## 12. Scan: convergence, extraction and classification

Compare maxima of the **total**, **ECO minus BH**, and (only for QH) the **isolated boundary correction**, each divided by the BH prompt peak. A common retarded-time coordinate is labelled by its time at exterior tortoise radius 95. Prompt: [100,190); late: [190,500] M. Source and detectors use the same right-exterior tortoise coordinates at every scan point, including the throat-centred WH grids. Peak frequencies describe a finite-window spectrum; only complex time-domain fits are compared with FD QNMs. An exploratory point cannot establish a clean echo.

The scan fixes right-exterior source/observer coordinates. Section 9 uses throat-centred source/observer coordinates and a different packet estimator for the wormhole. These are separate experiments, not interchangeable measurements.

In [ ]:
# t95 is retarded time referred to x*=95; observers share a common analysis interval.
from scipy.signal.windows import tukey
from scipy.optimize import minimize_scalar

SCAN_P=(100.,190.); SCAN_L=(190.,500.); radii=(95.,120.)
scan_norm="peak |BH at same observer| in 100<=t95<190; late 190<=t95<=500"
scan_levels=[.20,.10,.05]
representatives={"gravastar":{.40,.47,.495},"wormhole":{2.0005,2.02,2.1}}
def measurement_grid(wf,ref,radius):
    # common physical windows, no extrapolation, fixed quadrature step across resolutions
    tt=np.arange(1000,5001,dtype=float)/10; tlocal=tt+radius-95.
    if min(tlocal)<max(wf.t[0],ref.t[0]) or max(tlocal)>min(wf.t[-1],ref.t[-1]):
        raise ValueError("observer does not cover the stated time window")
    total=CubicSpline(wf.t,wf.psi,extrapolate=False)(tlocal)
    base=CubicSpline(ref.t,ref.psi,extrapolate=False)(tlocal)
    return tt,total,base

def window_peak_frequency(tt,y):
    """Continuous maximum of the tapered finite-record periodogram near its largest bin."""
    h=tt[1]-tt[0]; z=(y-np.mean(y))*tukey(len(y),.2)
    w=2*np.pi*np.fft.rfftfreq(len(z),h); power=abs(np.fft.rfft(z))**2
    ids=np.flatnonzero((w>.08)&(w<1.3)); j=ids[np.argmax(power[ids])]
    p=minimize_scalar(lambda om:-abs(np.dot(z,np.exp(1j*om*(tt-tt[0]))))**2,
                      bounds=(w[j-1],w[j+1]),method="bounded")
    return float(p.x)

def scan_observe(wf,ref,rad,hint,isolated=None):
    t,y,b=measurement_grid(wf,ref,rad); p=t<190.; l=~p; d=y-b
    peak=np.max(abs(b[p])); s=y if isolated is None else CubicSpline(wf.t,isolated)(t+rad-95)
    if not np.isfinite(peak) or peak<=0: raise ValueError("undefined BH prompt")
    # A peak on a lag-window edge is not evidence for recurrence. Require two cycles of coverage.
    tl=t[l]; sl=s[l]; lag=np.nan
    qual=dict(separated_pulses=False,stable_spacing=False,spacing_cv=np.nan,median_trough_ratio=np.nan,n_packets=0)
    if np.isfinite(hint) and hint>0 and np.ptp(tl)>=1.8*hint:
        packs=matched_packet_observables(t,s,(190.,190.+min(.7*hint,65.)),max(5.,.6*hint),190.)
        qual=pulse_train_quality(t,s,packs)
        z=hilbert(sl-np.mean(sl)); ac=abs(correlate(z,z,method="fft")[len(z)-1:]); lags=(tl-tl[0])
        peaks,_=find_peaks(ac,prominence=.005*max(ac[0],1e-300))
        cand=peaks[(lags[peaks]>.65*hint)&(lags[peaks]<1.35*hint)]
        if len(cand): lag=float(lags[cand[np.argmax(ac[cand])]])
    f,A,_=spectrum(tl,sl); w=2*np.pi*f; band=(w>.02)&(w<1.5); wt=A[band]**2
    return dict(T_recurrence=lag,A_late_total_over_BH_prompt=np.max(abs(y[l]))/peak,
        A_late_residual_over_BH_prompt=np.max(abs(d[l]))/peak,
        A_late_QH_correction_over_BH_prompt=np.nan if isolated is None else np.max(abs(s[l]))/peak,
        spectral_peak_omega=window_peak_frequency(tl,sl),
        spectral_bandwidth=np.sqrt(np.average((w[band]-np.average(w[band],weights=wt))**2,weights=wt)),
        M_early_intrinsic=1-optimized_overlap(y[p],b[p]),M_late_intrinsic=1-optimized_overlap(y[l],b[l]),
        BH_prompt_peak=peak,normalization=scan_norm,**qual),(y[l],d[l])

def gs_fd_td_at_point(mu,wf):
    # Complex TD fit is chosen from the measured (not FD-seeded) ringdown, then checked over windows.
    fits=[]
    for lo,hi in [(165,210),(170,220),(175,235)]:
        sel=(wf.t>=lo)&(wf.t<=hi)
        om=window_peak_frequency(wf.t[sel],wf.psi[sel])
        fit=damped_sinusoid_fit(wf.t,wf.psi,lo,hi,(np.clip(om,.16,.79),.035))
        if fit["success"] and np.isfinite(fit["omega_R"]+fit["omega_I"]):
            fits.append(complex(fit["omega_R"],fit["omega_I"]))
    if len(fits)<3:
        return dict(TD_omega_R=np.nan,TD_omega_I=np.nan,FD_omega_R=np.nan,FD_omega_I=np.nan,
                    TD_window_spread=np.nan,TD_FD_relative_error=np.nan,TD_FD_supported=False)
    td=complex(np.median(np.real(fits)),np.median(np.imag(fits)))
    scatter=max(abs(z-td) for z in fits)/abs(td)
    choices=[]
    for guess in [(td.real,td.imag),(.40,-.02),(.32,-.012),(.50,-.07)]:
        try:
            a=gravastar_qnm(mu,guess,45.,12)
            b=gravastar_qnm(mu,(a.real,a.imag),60.,14)
            if 0<b.real<1 and -.5<b.imag<0 and abs(a-b)/abs(b)<.01:
                choices.append(b)
        except (ValueError,RuntimeError,FloatingPointError,OverflowError):
            continue
    fd=min(choices,key=lambda z:abs(z-td)) if choices else complex(np.nan,np.nan)
    err=abs(td-fd)/abs(fd) if choices else np.nan
    return dict(fit_windows_M="165–210, 170–220, 175–235 at observer 95",frequency_units="M omega; ratios dimensionless",TD_omega_R=td.real,TD_omega_I=td.imag,FD_omega_R=fd.real,FD_omega_I=fd.imag,
                TD_window_spread=scatter,TD_FD_relative_error=err,
                TD_FD_supported=bool(np.isfinite(err) and err<.10 and scatter<.10))

def wh_predictions(r0):
    rc,tc,*_=half_barrier_coefficients_vectorized(wgrid,r0,CFG.ell,CFG.M)
    modes,chain=wormhole_qnm_map_for_r0(r0,rc,tc)
    periods=family_periods(modes)

    tau=np.gradient(np.unwrap(np.angle(chain["loop"])),wgrid,edge_order=2)
    ww=Pw*abs(tc**2*chain["R_throat_effective"])**2
    return float(np.average(tau[relevant],weights=ww[relevant])),periods

def relative_change(a,b):
    return abs(a-b)/max(abs(b),1e-15) if np.isfinite(a) and np.isfinite(b) else np.nan

def gs_loop_delay(mu):
    """Local shell characteristics use k=omega on BOTH sides of the decomposition."""
    w=np.linspace(.15,.90,181); a=CFG.M/mu; f=1-2*CFG.M/a
    rb,tb,*_=half_barrier_coefficients_vectorized(w,a,CFG.ell,CFG.M)
    psi,dr=np.array([gravastar_regular_data(a,om,CFG.ell,CFG.M) for om in w]).T
    ri=(1j*w*psi+f*dr)/(1j*w*psi-f*dr)
    tau=np.gradient(np.unwrap(np.angle(rb*ri)),w,edge_order=2)
    weights=np.interp(w,wpacket,Ppacket,left=0,right=0)*abs(tb)**4
    return float(np.average(tau,weights=weights))

def decide(row,criteria,resonance,negative_label="INCONCLUSIVE"):
    """One rule for scans and production summaries. Missing evidence never passes."""
    flags={k:(None if pd.isna(v) else bool(v)) for k,v in criteria.items()}
    flags["resolved_recurrence"]=bool(np.isfinite(row["T_recurrence"]) and row["T_recurrence"]>0)
    failed=[k for k,v in flags.items() if v is False]
    missing=[k for k,v in flags.items() if v is None]
    clean=not failed and not missing
    row.update({"Echo classification":"CLEAN_ECHO" if clean else ("TRAPPED_RESONANCE" if resonance else negative_label),
                "failed_criteria":"; ".join(failed) or "none","untested_criteria":"; ".join(missing) or "none",
                "failed_or_unavailable":"; ".join(failed+[k+" (untested)" for k in missing]) or "none",
                "T_echo":row["T_recurrence"] if clean else np.nan})
    row.update({f"pass_{k}":v for k,v in flags.items()})

def classifier_control(period,flags,evidence=False):
    row={"T_recurrence":period}; decide(row,flags,evidence); return row["Echo classification"]

decision_controls=all([
    classifier_control(35.,dict(shape_grid=True,spacing=True))=="CLEAN_ECHO",
    classifier_control(np.nan,dict(shape_grid=True,spacing=True))=="INCONCLUSIVE",
    classifier_control(35.,dict(shape_grid=None,spacing=True))=="INCONCLUSIVE",
    classifier_control(35.,dict(shape_grid=True,spacing=False),True)=="TRAPPED_RESONANCE"])
assert decision_controls


raw=[]; comparison=[]; result=[]
for model,values in (("gravastar",[.40,.44,.47,.485,.495]),("wormhole",[2.0005,2.005,2.02,2.1])):
    for value in values:
        print(f"Scan: {model} {value:g}",flush=True)
        audit=value in representatives[model]; levels=scan_levels if audit else scan_levels[:1]
        observers=radii if audit else radii[:1]; point={}; residuals={}; meta_fd={}
        gdelay=sspace=np.nan
        if model=="wormhole":
            gdelay,family=wh_predictions(value)
            meta_fd.update(spectral_time_even=family["even"],spectral_time_odd=family["odd"])
        elif audit: gdelay=gs_loop_delay(value)
        for dx in levels:
            # 95 and 120 correspond to the same exterior tortoise coordinates in all geometries.
            if model=="gravastar":
                base,x,V,r,records=run_gravastar(value,dx=dx,t_max=526,extraction_xs=observers)
                offset=0.; prefix="x"; m=base.metadata; shell=m["shell_index"]
                barrier=shell+1+np.argmax(V[shell+1:]); bottom=1+np.argmin(V[1:barrier+1])
                sphere=m["a"]<3*CFG.M
                cavity=bool(sphere and bottom<barrier and V[bottom]<V[barrier] and x[barrier]>m["shell_x"])
                hint=2*(x[barrier]-m["center_x"])
                if audit and dx==levels[-1]: meta_fd=gs_fd_td_at_point(value,base)
            else:
                offset=float(schwarzschild_rstar(value)); locs=tuple(q-offset for q in observers)
                cfg=replace(CFG,pulse_center=CFG.pulse_center-offset,extraction_x=locs[0])
                base,_,x,V,records=run_wormhole(value,cfg=cfg,dx=dx,t_max=526,extraction_xs=locs)
                prefix="right"; m=base.metadata; sphere=value<3*CFG.M
                cavity=bool(sphere and m["x_peak"]>0 and m["throat_Lambda"]>0)
                hint=abs(gdelay)
            bh_ref,_,_,_,refs=run_black_hole(dx=dx,t_max=526,x_left=-260.,extraction_xs=radii)
            for rad in observers:
                wf=Waveform(model,base.t,records[f"{prefix}{rad-offset:g}"],rad,m)
                ref=Waveform("BH",bh_ref.t,refs[f"x{rad:g}"],rad,bh_ref.metadata)
                ob,res=scan_observe(wf,ref,rad,hint)
                # Include 8 pulse widths on the outer-facing tail, and the far WH boundary if present.
                local_source=CFG.pulse_center-offset
                end=500+rad-95
                margins=[2*m["x_right"]-(local_source+8*CFG.pulse_width)-(rad-offset)-end,
                         2*bh_ref.metadata["x_right"]-(CFG.pulse_center+8*CFG.pulse_width)-rad-end,
                         CFG.pulse_center-8*CFG.pulse_width+rad-2*bh_ref.metadata["x_left"]-end]
                if model=="wormhole":
                    margins.append(local_source-8*CFG.pulse_width+rad-offset-2*m["x_left"]-end)
                # The regular stellar center is physical and is not an artificial boundary.
                ob.update(model=model,parameter=value,dx_requested=dx,dx_actual=m["dx"],radius=rad,
                          boundary_margin=min(margins),stable=m["stability_pass"] and bh_ref.metadata["stability_pass"],
                          photon_sphere=sphere,cavity=cavity,geometric_delay=hint if model=="gravastar" else m["geometric_roundtrip"])
                raw.append(ob); point[dx,rad]=ob; residuals[dx,rad]=res
        row=dict(point[levels[-1],95.]); row.update(meta_fd)
        row.update(parameter_name="M/a" if model=="gravastar" else "r0/M",group_delay=gdelay,
                   spectral_spacing_time=sspace,Delta_omega_qnm=2*np.pi/sspace if np.isfinite(sspace) else np.nan,
                   study="three grids, two radii" if audit else "exploratory: one grid, one radius")
        def compare(ka,kb,kind):
            a,b=point[ka],point[kb]; (ta,ya),(tb,yb)=residuals[ka],residuals[kb]
            recerr=relative_change(a["T_recurrence"],b["T_recurrence"])
            amp=relative_change(a["A_late_residual_over_BH_prompt"],b["A_late_residual_over_BH_prompt"])
            spec=relative_change(a["spectral_peak_omega"],b["spectral_peak_omega"])
            shape=np.linalg.norm(ya-yb)/max(np.linalg.norm(yb),1e-300)
            total_shape=np.linalg.norm(ta-tb)/max(np.linalg.norm(tb),1e-300)
            comparison.append(dict(model=model,parameter=value,comparison=kind,dx_a=ka[0],radius_a=ka[1],
                dx_b=kb[0],radius_b=kb[1],recurrence_fraction=recerr,late_residual_peak_fraction=amp,
                T_candidate_a=a["T_recurrence"],T_candidate_b=b["T_recurrence"],raw_total_shape_L2=total_shape,
                spectral_peak_fraction=spec,raw_retarded_residual_shape_L2=shape,normalization=scan_norm))
            return bool(amp<.10 and spec<.10 and shape<.10 and total_shape<.10), bool(recerr<.05)
        tests=[]; ext=[]
        if audit:
            for rad in observers:
                for da,db in zip(levels,levels[1:]): tests.append(compare((da,rad),(db,rad),"adjacent dx"))
            for dx in levels: ext.append(compare((dx,95.),(dx,120.),"observer, retarded"))
        conv=bool(tests and all(z[0] for z in tests)); extrap=bool(ext and all(z[0] for z in ext))
        timing=bool(tests and all(z[1] for z in tests+ext))
        rec=row["T_recurrence"]
        # A single complex frequency is not a spacing prediction: use a separate loop delay.
        theory=relative_change(rec,gdelay)<.15 and (all(relative_change(rec,v)<.15 for v in family.values()) if model=="wormhole"
                                                    else meta_fd.get("TD_FD_supported",False))
        crit=dict(photon_sphere=sphere,cavity=cavity,separated=row["separated_pulses"],spacing=row["stable_spacing"],
                  resolved_recurrence=np.isfinite(rec),domain=all(z["stable"] and z["boundary_margin"]>0 for z in point.values()) and boundary_error<1e-3,
                  shape_grid=conv if audit else None,shape_extraction=extrap if audit else None,
                  timing_stability=(timing if audit and all(np.isfinite(z["T_recurrence"]) for z in point.values()) else None),
                  theory_consistency=theory if np.isfinite(rec) and np.isfinite(gdelay) else None)
        resonance=meta_fd.get("TD_FD_supported",False) if model=="gravastar" else bool(cavity and any(np.isfinite(list(family.values()))))
        decide(row,crit,resonance)
        vals=[z["T_recurrence"] for z in point.values()]; finite=np.asarray(vals)[np.isfinite(vals)]
        row["echo_time_uncertainty"]=np.ptp(finite) if np.isfinite(row["T_echo"]) and finite.size>1 else np.nan
        result.append(row)

for hierarchy,(wf,echo) in quantum_scan_waveforms.items():
    rq=quantum_local_reflection(wgrid,xH,EplM=hierarchy); k=surface_K(wgrid,Rbh,Tdet,rq,xH,xE)
    period,packs,loop=qh_packet_audit(wf,echo,rq)
    row,_=scan_observe(wf,bh,95.,loop,echo)
    row.update(model="quantum_boundary",parameter_name="Epl*M/gamma",parameter=hierarchy,T_recurrence=period,
        study="frequency-domain boundary; independent TD at production hierarchy" if hierarchy==1e8 else "exploratory hierarchy",
        Delta_omega_qnm=np.nan,geometric_delay=L_cavity_roundtrip,spectral_spacing_time=np.nan,
        group_delay=np.interp(wc,wgrid,np.gradient(np.unwrap(np.angle(k)),wgrid)),
        abs_Rq_at_0_37=np.interp(.37,wgrid,abs(rq)),arg_Rq_at_0_37=np.interp(.37,wgrid,np.unwrap(np.angle(rq))),
        tau_Q_at_0_37=np.interp(.37,wgrid,np.gradient(np.unwrap(np.angle(rq)),wgrid)),echo_time_uncertainty=np.nan)
    qual=pulse_train_quality(wf.t,echo,packs)
    for key in qual: row[key]=qual[key]
    decide(row,dict(separated=qual["separated_pulses"],spacing=qual["stable_spacing"],resolved_recurrence=np.isfinite(period),
                    loop_agreement=relative_change(period,loop)<.15,verified_boundary=(hierarchy==1e8 and qh_transfer_reconstruction_error<.05)),False)
    result.append(row)

scan_df=pd.DataFrame(result); scan_raw=pd.DataFrame(raw); scan_compare=pd.DataFrame(comparison)
assert len(scan_raw)==6*3*2+3 and len(scan_compare)==6*(4+3)
assert np.isfinite(scan_df[["A_late_total_over_BH_prompt","A_late_residual_over_BH_prompt"]]).all().all()
q=scan_df.model=="quantum_boundary"
assert np.allclose(scan_df.loc[q,"A_late_residual_over_BH_prompt"],scan_df.loc[q,"A_late_QH_correction_over_BH_prompt"],rtol=1e-7)
assert scan_df.loc[scan_df["Echo classification"]!="CLEAN_ECHO","T_echo"].isna().all()
assert scan_df.loc[scan_df.study.str.contains("exploratory"),"T_echo"].isna().all()
for df,name in [(scan_df,"waveform_parameter_map"),(scan_raw,"scan_measurements"),(scan_compare,"scan_resolution_and_extraction")]:
    df.to_csv(ARTIFACT_DIR/(name+".csv"),index=False)
display(scan_df[["model","parameter","study","T_recurrence","T_echo","A_late_total_over_BH_prompt",
                 "A_late_residual_over_BH_prompt","A_late_QH_correction_over_BH_prompt","Echo classification","failed_or_unavailable","normalization"]])
display(scan_df.loc[scan_df.TD_omega_R.notna(),["parameter","fit_windows_M","frequency_units","TD_omega_R","TD_omega_I","FD_omega_R","FD_omega_I","TD_FD_relative_error","TD_window_spread","TD_FD_supported"]])
# A flat total amplitude must not be magnified into an apparent hierarchy trend.
fig,axes=plt.subplots(3,4,figsize=(15,8),layout="constrained")
columns=[("A_late_total_over_BH_prompt","Total peak"),("A_late_residual_over_BH_prompt","Residual peak"),
         ("A_late_QH_correction_over_BH_prompt","Isolated QH peak"),("T_recurrence","Recurrence [M]")]
labels={"gravastar":"Gravastar","wormhole":"Wormhole","quantum_boundary":"Quantum boundary"}
for aa,(model,g) in zip(axes,scan_df.groupby("model",sort=False)):
    for ax,(col,title) in zip(aa,columns):
        for _,r in g.iterrows():
            if np.isfinite(r[col]): ax.plot(r.parameter,r[col],"o",color="tab:blue",mfc="none" if "exploratory" in r.study else "tab:blue")
        if model=="quantum_boundary": ax.set_xscale("log")
        ax.set_xlim(g.parameter.min()*(.75 if model=="quantum_boundary" else 1)-(.0 if model=="quantum_boundary" else np.ptp(g.parameter)*.08),
                    g.parameter.max()*(1.3 if model=="quantum_boundary" else 1)+(.0 if model=="quantum_boundary" else np.ptp(g.parameter)*.08))
        good=g[col].dropna()
        if good.empty:
            ax.set_yticks([]); ax.text(.5,.5,"Not applicable" if col.startswith("A_late_QH") else "Not resolved",transform=ax.transAxes,ha="center")
        else:
            ax.set_ylim(0,1.16*max(good.max(),1e-12)); ax.ticklabel_format(style="plain",axis="y",useOffset=False)
        ax.set(title=title,xlabel=g.parameter_name.iloc[0])
    aa[0].set_ylabel(labels[model])
fig.suptitle("Late peaks [190,500] / BH prompt peak [100,190) M at the same observer\nOpen symbols: exploratory; blank recurrence is not a zero period.")
fig.savefig(ARTIFACT_DIR/"waveform_parameter_maps.png",bbox_inches="tight")


## 13. Spatial resolution, independent time-step convergence, and fail-fast validation

Two three-level spatial order estimates use $\Delta x=(0.4,0.2,0.1)M$ and $(0.2,0.1,0.05)M$. Cubic time alignment prevents a second-order interpolant from corrupting the fourth-order test. A separate fixed-grid run varies CFL as $(0.35,0.175,0.0875)$ so temporal RK4 error is not hidden by a simultaneous spatial refinement.

The final table collects every key validation and raises an exception if a required test fails.


In [ ]:
# Spatial convergence through dx=0.05.
spatial_dx=[.40,.20,.10,.05]; spatial_waves=[]; spatial_rows=[]
for dx in spatial_dx:
    wf,*_=run_black_hole(dx=dx,t_max=240,x_left=-140,x_right=300,extraction_xs=(95.,)); spatial_waves.append(wf)
    spatial_rows.append({"dx":dx,"dt":wf.metadata["dt"],"Nx":wf.metadata["Nx"],"Nt":wf.metadata["Nt"],"CFL":wf.metadata["cfl"]})
ref=spatial_waves[-1]; m=(ref.t>=130)&(ref.t<=220); ys=[CubicSpline(w.t,w.psi)(ref.t[m]) for w in spatial_waves]; scale=np.linalg.norm(ys[-1])
pair_errors=[np.linalg.norm(ys[i]-ys[i+1])/scale for i in range(3)]
p_coarse=float(np.log2(pair_errors[0]/pair_errors[1])); p_fine=float(np.log2(pair_errors[1]/pair_errors[2]))
spatial_df=pd.DataFrame(spatial_rows); spatial_df["adjacent_pair_L2"]=pair_errors+[np.nan]

# Independent time convergence at fixed dx.
cfls=[.35,.175,.0875]; time_waves=[]
for cfl in cfls: time_waves.append(run_black_hole(dx=.20,t_max=220,x_left=-140,x_right=300,extraction_xs=(95.,),cfl=cfl)[0])
tref=time_waves[-1]; mt=(tref.t>=130)&(tref.t<=210); ytime=[CubicSpline(w.t,w.psi)(tref.t[mt]) for w in time_waves]
time_pair=[np.linalg.norm(ytime[0]-ytime[1]),np.linalg.norm(ytime[1]-ytime[2])]; p_time=float(np.log2(time_pair[0]/time_pair[1]))
time_df=pd.DataFrame({"CFL":cfls,"dt":[w.metadata["dt"] for w in time_waves],"adjacent_pair_L2":[time_pair[0],time_pair[1],np.nan]})
display(spatial_df); print("Spatial p coarse/fine:",p_coarse,p_fine); display(time_df); print("Temporal p:",p_time)

# Gravastar shell-aligned/regular-center waveform refinement check.
gs_coarse,*_=run_gravastar(CFG.gravastar_compactness,dx=.20,t_max=420,extraction_xs=(95.,))
mg=(gs.t>=160)&(gs.t<=400); gsc=CubicSpline(gs_coarse.t,gs_coarse.psi)(gs.t[mg])
gravastar_grid_refinement_error=float(np.linalg.norm(gsc-gs.psi[mg])/max(np.linalg.norm(gs.psi[mg]),1e-300))

bh_td_fund_error=float(td_qnm_df.loc[0,"TD_relative_error"]) if bool(td_qnm_df.loc[0,"resolved_free_TD"]) else np.inf
all_stable=all(bool(w.metadata["stability_pass"]) for w in [bh,gs,bs,wh,wh_coarse,wh_fine])
primary_causal_margins=[2*w.metadata["x_right"]-CFG.pulse_center-w.x_extract-w.metadata["actual_t_max"] for w in [bh,gs,bs,wh]]
minimum_boundary_causal_margin=float(min(primary_causal_margins))
boson_lit_error=float(bs_benchmark.relative_error.max())
wormhole_delay_error=abs(Tmatched-Tweighted)/abs(Tweighted)
wormhole_resolution_stable=bool(wormhole_spacing_resolution_fraction<.05 and wormhole_extraction_spacing_fraction<.05
    and wormhole_pulse_convergence_df.separated_pulses.all() and wormhole_pulse_convergence_df.stable_spacing.all())
numerical_resolution_stable=bool(3.5<p_coarse<4.5 and 3.5<p_fine<4.5 and 3.4<p_time<4.6)

# Use production waveforms for measurements, and the matching scan for independent
# grid/observer evidence. Do not import the fine-grid scan's label as the answer.
gs_s=scan_df[(scan_df.model=="gravastar") & np.isclose(scan_df.parameter,CFG.gravastar_compactness)]
gs_s=gs_s.iloc[0] if len(gs_s) else None
si=gs.metadata["shell_index"]; ib=si+1+np.argmax(V_gs[si+1:]); iw=1+np.argmin(V_gs[1:ib+1])
gs_sphere=gs.metadata["a"]<3*CFG.M
gs_cavity=bool(gs_sphere and iw<ib and V_gs[iw]<V_gs[ib] and x_gs[ib]>gs.metadata["shell_x"])
gs_hint=2*(x_gs[ib]-gs.metadata["center_x"])
gs_measured,_=scan_observe(gs,bh,gs.x_extract,gs_hint)
gs_rec=gs_measured["T_recurrence"]
gs_theory=(relative_change(gs_rec,gs_s.group_delay)<.15 if gs_s is not None and
           np.isfinite(gs_rec) and np.isfinite(gs_s.group_delay) else None)
gs_supported=bool(gravastar_fundamental_support>=2 and candidate.TD_supported and
                  candidate.FD_relative_spread<1e-6 and gs_td_fd_error<.08)
def gs_check(key): return gs_s.get("pass_"+key,None) if gs_s is not None else None
boundary_ok=boundary_error<1e-3
reports=[]
def evidence_report(model,period,photon,cavity,reflective,trapped,sep,spacing,group,spectral,
                    domain,grid,extra,description,negative="INCONCLUSIVE"):
    r={"model":model,"T_recurrence":period,"Photon sphere present?":photon,"Cavity identified?":cavity,
       "Reflective boundary?":reflective,"Trapped modes present?":trapped,"Separated pulses identified?":sep,
       "Stable echo spacing?":spacing,"Group-delay agreement?":group,"QNM-spacing agreement?":spectral,
       "Boundary stable?":domain,"Resolution stable?":grid,"dominant_late_structure":description}
    # A single TD/FD mode can establish a resonance, not an intermode spacing.
    decide(r,dict(photon_sphere=photon,cavity=cavity,reflective=reflective,separated=sep,spacing=spacing,
                  domain=domain,shape_grid=grid,**extra),trapped,negative)
    reports.append(r)
evidence_report("BH",np.nan,True,False,False,False,None,None,None,None,boundary_ok,
                numerical_resolution_stable,{},"absorbing Schwarzschild reference","NO_CLEAN_ECHO")
evidence_report("GS",gs_rec,gs_sphere,gs_cavity,True,gs_supported,gs_measured["separated_pulses"],
                gs_measured["stable_spacing"],gs_theory,None,boundary_ok and gs.metadata["stability_pass"] and
                bool(gs_check("domain")),gs_check("shape_grid"),dict(shape_extraction=gs_check("shape_extraction"),
                timing_stability=gs_check("timing_stability"),theory_consistency=gs_theory,TD_FD=gs_supported),
                "regular-center axial response; complex TD/FD root tested")
evidence_report("BS",np.nan,bs.metadata["photon_sphere"],False,False,False,None,None,None,None,
                boundary_ok,None,{},"axial stellar response; published damped modes computed; no trapped photon-sphere cavity")
wg=wormhole_delay_error<.10; ws=wormhole_qnm_spacing_error<.10
evidence_report("WH",Tmatched,wh.metadata["r0"]<3*CFG.M,wh.metadata["x_peak"]>0,True,
                bool(len(wormhole_qnm_map)>0),wh_quality["separated_pulses"],wh_quality["stable_spacing"],
                wg,ws,boundary_ok,wormhole_resolution_stable,dict(group=wg,spectral=ws),
                "throat-centred experiment: parity-resolved resonances / matched-packet recurrence")
qg=relative_change(qh_delay_obs,qh_loop_hint)<.15 if np.isfinite(qh_delay_obs) else None
evidence_report("QH",qh_delay_obs,True,xH<0,bool(np.any(abs(Rq_grid)>0)),bool(len(qh_modes_df)>0),
                qh_quality["separated_pulses"],qh_quality["stable_spacing"],qg,None,
                boundary_ok,qh_transfer_reconstruction_error<.05,dict(loop_agreement=qg),
                "isolated OWA boundary correction; complex outgoing poles independently verified")
echo_report_df=pd.DataFrame(reports)
display(echo_report_df); echo_report_df.to_csv(ARTIFACT_DIR/"standard_echo_validity_report.csv",index=False)
keys=["Echo classification","T_echo","T_recurrence","failed_criteria","untested_criteria"]
eco_central_comparison_df=eco_central_comparison_df.drop(columns=keys,errors="ignore").merge(
    echo_report_df[["model",*keys]],on="model",validate="one_to_one",how="left")
by_m=echo_report_df.set_index("model").loc[eco_central_comparison_df.model]
report_agrees=(np.array_equal(by_m["Echo classification"],eco_central_comparison_df["Echo classification"]) and
               np.allclose(by_m[["T_echo","T_recurrence"]],eco_central_comparison_df[["T_echo","T_recurrence"]],equal_nan=True))
display(eco_central_comparison_df)


qh_lowfreq_final=float(qh_lowfreq_df.late_correction_relative_L2.max())
qh_lowfreq_phase_final=float(qh_lowfreq_df.late_weighted_phase_rms.max())
echo_label_consistent=bool(np.all(echo_report_df.loc[echo_report_df["Echo classification"]!="CLEAN_ECHO","T_echo"].isna()))
wormhole_is_clean=bool(echo_report_df.loc[echo_report_df.model=="WH","Echo classification"].iloc[0]=="CLEAN_ECHO")

validation_specs=[
    ("Boson published axial mode benchmark","both Table IV values within our 5% gate; paper estimates few-percent tail uncertainty",bs_modes_df.published_relative_error.max(),bs_modes_df.passed.all(),True),
    ("Boson complex mode numerical convergence","fixed-tail problem relative change <2e-7",bs_mode_numerics.relative_change.max(),bs_mode_numerics.relative_change.max()<2e-7,True),
    ("Quantum complex pole real-axis prescription","analytic continuation agrees with actual global coefficient",qh_mode_checks.value.max(),qh_mode_checks.passed.all(),True),
    ("Quantum complex pole convergence","interface/contour/order/tolerance change <2e-6",qh_mode_numerics.relative_change.max(),qh_mode_numerics.passed.all(),True),
    ("Gravastar published analytic interior and metric junctions","45/45 high-precision reference comparisons",pani_df.reflection_absolute_error.max(),pani_df.passed.all(),True),
    ("Wormhole published complex frequencies","3 independent Table 6 values within 1%",wh_literature_df.relative_complex_error.max(),wh_literature_df.passed.all(),True),
    ("Wormhole complex-pole convergence","contour/distance/order/tolerance change <2e-6",wh_pole_convergence_df.relative_change.max(),wh_pole_convergence_df.relative_change.max()<2e-6,True),
    ("Controlled comparison geometry","same areal radii within1e-9 M",max(comparison_geometry.source_radius_error.max(),comparison_geometry.observer_radius_error.max()),max(comparison_geometry.source_radius_error.max(),comparison_geometry.observer_radius_error.max())<1e-9,True),
    ("Quantum fixed-physics interface invariance","unfitted waveform change <1e-4",qh_plane_df.raw_waveform_change.max(),qh_plane_df.raw_waveform_change.max()<1e-4,True),
    ("Quantum global/local coefficient identity","relative algebraic residual <1e-12",qh_plane_df.global_local_identity_error.max(),qh_plane_df.global_local_identity_error.max()<1e-12,True),
    ("Quantum overlap-region kernel control","negative-lag norm <0.1% and independent PDE error <1%",qh_plane_df.raw_TD_FD_error.max(),qh_plane_df.negative_lag_kernel_fraction.max()<.001 and qh_plane_df.raw_TD_FD_error.max()<.01,True),
    ("Quantum moved-surface independent PDE","raw late error < 1% at second surface, fixed source and detector",alt_error,alt_error<.01,True),
    ("Quantum zero wall end to end","both PDE and spectrum correction exactly zero",zero_error,zero_error==0.,True),
    ("Echo published timing agreement","all central and final timing columns agree",float(report_agrees),report_agrees,True),
    ("Echo classifier controls","positive, negative and missing-evidence controls",float(decision_controls),decision_controls,True),
    ("BH QNM n=0 literature","relative error < 1e-6",float(bh_qnm_df.relative_error.iloc[0]),float(bh_qnm_df.relative_error.iloc[0])<1e-6,True),
    ("BH QNM n=1 literature","relative error < 1e-6",float(bh_qnm_df.relative_error.iloc[1]),float(bh_qnm_df.relative_error.iloc[1])<1e-6,True),
    ("BH QNM n=2 literature","relative error < 1e-6",float(bh_qnm_df.relative_error.iloc[2]),float(bh_qnm_df.relative_error.iloc[2])<1e-6,True),
    ("BH free TD fundamental","relative complex error < 8%",bh_td_fund_error,bh_td_fund_error<.08,True),
    ("Initial ingoing fraction","> 0.999",float(bh.metadata["F_ingoing_left_u_plus"]),float(bh.metadata["F_ingoing_left_u_plus"])>.999,True),
    ("All primary evolutions stable","no exponential growth",float(all_stable),all_stable,True),
    ("Boundary contamination","max relative L2 < 1e-3",boundary_error,boundary_error<1e-3,True),
    ("Production boundary causal margin","earliest right-boundary return after t_max",minimum_boundary_causal_margin,minimum_boundary_causal_margin>0,True),
    ("Extraction-radius raw agreement","retarded raw L2 < 5e-2",extraction_raw_error,extraction_raw_error<5e-2,True),
    ("Extraction-radius shape agreement","amplitude-fitted retarded L2 < 3e-2",extraction_error,extraction_error<3e-2,True),
    ("Extraction-radius late agreement","late retarded shape L2 < 10%",extraction_late_error,extraction_late_error<.10,True),
    ("Spatial order coarse","3.5 < p < 4.5",p_coarse,3.5<p_coarse<4.5,True),
    ("Spatial order fine","3.5 < p < 4.5",p_fine,3.5<p_fine<4.5,True),
    ("Temporal RK4 order","3.4 < p < 4.6",p_time,3.4<p_time<4.6,True),
    ("BH barrier flux","max |R^2+T^2-1| < 2e-6",flux_residual,flux_residual<2e-6,True),
    ("Gravastar FD convergence","rmax/order spread < 1e-6",gravastar_fd_spread,gravastar_fd_spread<1e-6,True),
    ("Gravastar multistart attempts","5--10 independent complex guesses",gravastar_multistart_attempt_count,5<=gravastar_multistart_attempt_count<=10,True),
    ("Gravastar fundamental basin support","at least 2 independent guesses reach the physical fundamental",gravastar_fundamental_support,gravastar_fundamental_support>=2,True),
    ("Gravastar multistart fundamental","clustered root agrees with converged FD mode < 1e-6",gravastar_multistart_fund_error,gravastar_multistart_fund_error<1e-6,True),
    ("Gravastar TD/FD","relative complex error < 8%",gs_td_fd_error,gs_td_fd_error<.08,True),
    ("Gravastar shell metric","continuity residual < 1e-12",gravastar_shell_residual,gravastar_shell_residual<1e-12,True),
    ("Gravastar shell alignment","coordinate residual < 1e-12",gravastar_shell_alignment_error,gravastar_shell_alignment_error<1e-12,True),
    ("Gravastar grid refinement","coarse/fine waveform L2 < 10%",gravastar_grid_refinement_error,gravastar_grid_refinement_error<.10,True),
    ("Boson BVP residual","< 1e-4",bs_bg.residual,bs_bg.residual<1e-4,True),
    ("Boson literature background","max relative error < 1e-3",boson_lit_error,boson_lit_error<1e-3,True),
    ("Boson photon sphere","absent for mini-BS I",float(len(photon_spheres)),len(photon_spheres)==0,True),
    ("Boson photon metadata scope","recomputed for evolved background",float(bs.metadata["photon_sphere_metadata_scope"]=="recomputed for this background"),bs.metadata["photon_sphere_metadata_scope"]=="recomputed for this background",True),
    ("Wormhole symmetry","max residual < 1e-8",symmetry_residual,symmetry_residual<1e-8,True),
    ("Wormhole throat scattering flux","max residual < 1e-12",throat_flux_residual,throat_flux_residual<1e-12,True),
    ("Wormhole throat analytic jump","max residual < 1e-12",throat_scattering_jump_residual,throat_scattering_jump_residual<1e-12,True),
    ("Wormhole numerical throat jump","final residual < 1e-10",abs(wh.metadata["internal_interface"]["final_Psi_jump_residual"]),abs(wh.metadata["internal_interface"]["final_Psi_jump_residual"])<1e-10,True),
    ("Wormhole finite-band delay","<10% required only for CLEAN_ECHO; disagreement must downgrade label",wormhole_delay_error,(wormhole_delay_error<.10) or not wormhole_is_clean,True),
    ("Wormhole QNM spacing","<10% required only for CLEAN_ECHO; split families remain recurrence",wormhole_qnm_spacing_error,(wormhole_qnm_spacing_error<.10) or not wormhole_is_clean,True),
    ("Wormhole pulse resolution","<5% required only for CLEAN_ECHO; dx=.20,.10,.05 retained",wormhole_spacing_resolution_fraction,(wormhole_spacing_resolution_fraction<.05) or not wormhole_is_clean,True),
    ("Wormhole extraction spacing","x=75,95,120 spacing spread < 5%",wormhole_extraction_spacing_fraction,wormhole_extraction_spacing_fraction<.05,True),
    ("Wormhole timing interpretation","full timing audit and echo label agree",float(wormhole_simple_echo_picture_valid==wormhole_is_clean),wormhole_simple_echo_picture_valid==wormhole_is_clean,True),
    ("Unified ECO reflection definitions","GS/WH/QH physical; BS explicitly undefined",float(len(reflection_responses)==3 and "BS" in undefined_reflections),len(reflection_responses)==3 and "BS" in undefined_reflections,True),
    ("Unified central metrics","all defined responses have finite amplitude/phase/delay",float(np.all(np.isfinite(eco_central_comparison_df.loc[eco_central_comparison_df.reflection_defined,["abs_R_at_wc","phi_R_at_wc","tau_R_at_wc"]].to_numpy(float)))),np.all(np.isfinite(eco_central_comparison_df.loc[eco_central_comparison_df.reflection_defined,["abs_R_at_wc","phi_R_at_wc","tau_R_at_wc"]].to_numpy(float))),True),
    *[(f"{k} raw transfer reconstruction","unshifted/unscaled late L2 < 1%",float(v),v<RAW_TARGET,True) for k,v in transfer_reconstruction_df.set_index("model").raw_relative_L2.items() if k in ("GS","WH")],
    ("Measured characteristic analytic controls","pure ingoing/outgoing/mixed/DC/negative controls",float(c3_controls_df.passed.mean()),c3_controls_df.passed.all(),True),
    ("Measured characteristic one-factor convergence","36 trials; raw error <1%, prediction change <0.1%",characteristic_convergence_df.prediction_change_relative_L2.max(),(characteristic_convergence_df.raw_relative_L2<.01).all() and (characteristic_convergence_df.prediction_change_relative_L2<.001).all(),True),
    ("GS junction, unit-incident, flux and return controls","all component tolerances",float(gs_component_df.passed.mean()),gs_component_df.passed.all(),True),
    ("GS moved-shell first returns","absolute arrival < 0.05 M; delay change < 0.05 M",gs_timing_df.peak_error_M.max(),gs_timing_df.peak_error_M.max()<.05 and gs_timing_df.shell_change_shift_error_M.max()<.05,True),
    ("WH left domain and both observers","move -400 to -480 M; raw change < 1e-7",wh_domain_fine.relative_domain_change.max(),wh_domain_fine.relative_domain_change.max()<1e-7,True),
    ("WH component, symmetry and two-sided matching","all component tolerances",float(wh_checks_df.passed.mean()),wh_checks_df.passed.all(),True),
    ("WH raw even and odd plus both observers","late window raw L2 < 1%",wh_parity_df.query("t_start==190 and t_end==500").raw_relative_L2.max(),wh_parity_df.query("t_start==190 and t_end==500").raw_relative_L2.max()<.01,True),
    ("Geometric inversion convergence","padding-only change < 0.1%",stability_transfer_df.relative_L2_changed_numerics.max(),stability_transfer_df.relative_L2_changed_numerics.max()<.001,True),
    ("Quantum reflectivity identity","max residual < 1e-10",quantum_identity,quantum_identity<1e-10,True),
    ("Quantum phase decomposition","derivative residual < 1e-6",phase_decomposition_residual,phase_decomposition_residual<1e-6,True),
    ("Quantum analytic causality","relative L2 < 1e-10",causality_l2,causality_l2<1e-10,True),
    ("Quantum arrival convention","arrival error <= dt",causality_arrival_error,causality_arrival_error<=.05,True),
    ("Quantum reflection phase sign","wrapped phase error < 1e-10",causality_phase_error,causality_phase_error<1e-10,True),
    ("Quantum full-chain amplitude","controlled relative error < 2%",chain_amplitude_error,chain_amplitude_error<.02,True),
    ("Quantum full-chain delay","controlled absolute error < 0.2M",chain_delay_error,chain_delay_error<.2,True),
    ("Quantum full-chain phase","controlled wrapped error < 0.03 rad",chain_phase_error,chain_phase_error<.03,True),
    ("Quantum omega_min convergence","late correction L2 < 5%",qh_lowfreq_final,qh_lowfreq_final<.05,True),
    ("Quantum low-frequency phase convergence","late weighted phase RMS < 0.05 rad",qh_lowfreq_phase_final,qh_lowfreq_phase_final<.05,True),
    ("Quantum phase interpolation","PCHIP vs linear late L2 < 5%",phase_interpolation_error,phase_interpolation_error<.05,True),
    ("Quantum time reconstruction","FFT padding late L2 < 5%",time_reconstruction_error,time_reconstruction_error<.05,True),
    ("Quantum physical location tests","max pointwise error < 1e-10 (zero, linearity, shifted arrivals and round trips)",plane_tests.max_error.max(),plane_tests.max_error.max()<1e-10,True),
    ("Quantum constant and dispersive TD boundary","both dx=.05 unfitted PDE errors < 5%",qh_td_tests.loc[qh_td_tests.dx==.05,"raw_relative_L2"].max(),qh_td_tests.loc[qh_td_tests.dx==.05,"raw_relative_L2"].max()<.05,True),
    ("Quantum independent PDE refinement","raw discrepancy decreases from dx=.10 to .05 for both wall laws",float((qh_td_tests.groupby("law").raw_relative_L2.last()/qh_td_tests.groupby("law").raw_relative_L2.first()).max()),(qh_td_tests.groupby("law").raw_relative_L2.last()<qh_td_tests.groupby("law").raw_relative_L2.first()).all(),True),
    ("Quantum unified transfer reconstruction","direct vs common-K late L2 < 5%",qh_transfer_reconstruction_error,qh_transfer_reconstruction_error<.05,True),
    ("Echo labels physically consistent","T_echo only for CLEAN_ECHO",float(echo_label_consistent),echo_label_consistent,True),
]
def validation_category(name):
    if name.startswith("Quantum") or name.startswith("BH barrier"): return "Quantum"
    if name.startswith("Wormhole") or name.startswith("Echo") or name.startswith("Unified"): return "Echo physics"
    if name.startswith("Gravastar") or name.startswith("Boson"): return "Background-model"
    return "Numerical"
validation_df=pd.DataFrame([{"Category":validation_category(a),"Test":a,"Requirement":b,"Result":c,"Pass":bool(d),"Required":bool(e)} for a,b,c,d,e in validation_specs])
for category,group in validation_df.groupby("Category",sort=False): print("\n",category); display(group)
validation_df.to_csv(ARTIFACT_DIR/"validation_summary.csv",index=False); boundary_df.to_csv(ARTIFACT_DIR/"boundary_tests.csv",index=False)

fig,ax=plt.subplots(1,2,figsize=(12,4)); ax[0].loglog(spatial_dx[:-1],pair_errors,"o-"); ax[0].set(xlabel=r"$\Delta x/M$",ylabel="adjacent relative L2",title=f"Spatial p={p_coarse:.3f}, {p_fine:.3f}")
ax[1].loglog(cfls[:-1],time_pair,"o-"); ax[1].set(xlabel="CFL at fixed dx",ylabel="adjacent L2",title=f"Temporal p={p_time:.3f}")
fig.tight_layout(); fig.savefig(ARTIFACT_DIR/"spatial_temporal_convergence.png")

failed=validation_df[validation_df.Required & ~validation_df.Pass]
if len(failed): raise AssertionError("Required validation failed:\n"+failed[["Test","Result","Requirement"]].to_string(index=False))
print("All required validations passed.")


## 14. Structured outputs

The HDF5 database stores raw waveforms, analytic-signal quantities, physical master flux, numerical metadata, echo observables, mismatch matrices, validation results, uncertainty components, QNM tables, and parameter maps. CSV mirrors are written for quick inspection.


In [ ]:
h5_path=ARTIFACT_DIR/"ECO_waveforms.h5"
refs={"black_hole":"Leaver 1985; Regge-Wheeler 1957","thin_shell_gravastar":"arXiv:0909.0287",
      "mini_boson_star_axial_fixed_background":"arXiv:1307.4812",
      "schwarzschild_thin_shell_wormhole_axial_matching":"Yang, Xia & Miao, Eur. Phys. J. C 85, 742 (2025), doi:10.1140/epjc/s10052-025-14446-w; arXiv:1706.06155; arXiv:1608.08637",
      "owa_boltzmann_quantum_horizon":"arXiv:1905.00464"}
with h5py.File(h5_path,"w") as h5:
    h5.attrs['flux_convention']='finite-radius radiation-limit estimate; Q_RW=f*h1/r; (ell+2)!/(ell-2)!*|Q_RW|^2/(16*pi)'
    h5.attrs['weighted_overlap_convention']='windowed RW field converted to strain shape in positive-frequency detector band before analytic PSD weighting'
    h5.attrs["project"]="Model-specific linear ECO responses with unified physical reflection and fail-fast validation"; h5.attrs["convention"]="G=c=1, exp(-i omega t), stable Im(omega)<0"
    h5.attrs['selected_complex_modes_json']=qnm_comparison_df.to_json(orient='records')
    for short,wf in waveforms.items():
        g=h5.create_group(wf.model); z=hilbert(wf.psi)
        for key,data in {"time":wf.t,"waveform_real":wf.psi,"waveform_imaginary":z.imag,"amplitude":abs(z),"phase":np.unwrap(np.angle(z)),
                         "masked_instantaneous_frequency":wf.masked_instantaneous_frequency(1e-3),"BH_subtracted_residual":wf.psi-CubicSpline(reference_native[short].t,reference_native[short].psi,extrapolate=False)(wf.t),"master_flux":rw_radiation_flux(wf,CFG.ell)}.items(): g.create_dataset(key,data=data,compression="gzip")
        g.attrs["residual_scope"]="native production record minus its native reference; controlled cross-model residual is in /controlled_comparison"
        selected=qnm_comparison_df[qnm_comparison_df.model==short]
        g.create_dataset('selected_QNM_Momega',data=selected.Momega_R.to_numpy()+1j*selected.Momega_I.to_numpy())
        g.attrs['selected_QNM_scope_json']=selected.to_json(orient='records')
        g.attrs["literature_references"]=refs[wf.model]; g.attrs["metadata_json"]=json.dumps(wf.metadata,default=float)
        if short=="QH":
            g.create_dataset("isolated_transfer_correction",data=qh_echo)
            g.create_dataset("independent_TD_correction_on_BH_grid",data=td_owa_fine)
        if short=="WH" and len(wh_packets):
            eg=g.create_group("echo_observables")
            for col in wh_packets.columns: eg.create_dataset(col,data=wh_packets[col].to_numpy())

    # These are the EXACT records entering every cross-model metric.  A single
    # BH member is used for every comparison; no opponent-dependent substitution.
    cg=h5.create_group("controlled_comparison")
    cg.attrs["protocol"]="same ADM M=1, source global Schwarzschild r*=55, observer r*=95; radii from inverse r*"
    cg.attrs["geometry_json"]=comparison_geometry.to_json(orient="records")
    cg.attrs["native_bank_relationship"]="native root records retained separately for model-specific TD/FD validation"
    for short,wf in comparison_waveforms.items():
        group=cg.create_group(short)
        reference=CubicSpline(bh.t,bh.psi,extrapolate=False)(wf.t)
        # GS takes a shell-aligned grid with a slightly different last sample.
        # Out-of-reference coverage is represented honestly as NaN, never fitted.
        for key,data in {"time":wf.t,"waveform_real":wf.psi,"master_flux":rw_radiation_flux(wf,CFG.ell),
                         "BH_subtracted_residual":wf.psi-reference}.items():
            group.create_dataset(key,data=data,compression="gzip")
        group.attrs["metadata_json"]=json.dumps(wf.metadata,default=float)
    h5.attrs["mismatch_source_group"]="controlled_comparison"
    h5.attrs["mismatch_experiment"]="one waveform per label at common ADM mass, source areal radius and observer areal radius"
    h5.attrs["root_waveform_scope"]="native production coordinates for model-specific reconstruction; not cross-model matrix input"

    for key,data in {"prompt_intrinsic_mismatch":prompt_mismatch,"late_intrinsic_mismatch":late_mismatch,"prompt_detector_mismatch":prompt_detector_mismatch,"late_detector_mismatch":late_detector_mismatch}.items(): h5.create_dataset(key,data=data)
    h5.attrs["mismatch_labels"]=json.dumps(names); h5.attrs["validation_json"]=validation_df.to_json(orient="records"); h5.attrs["echo_uncertainty_json"]=echo_uncertainty_components.to_json(orient="records")
    h5.attrs["echo_validity_json"]=echo_report_df.to_json(orient="records"); h5.attrs["qh_low_frequency_convergence_json"]=qh_lowfreq_df.to_json(orient="records")
    h5.attrs["eco_central_comparison_json"]=eco_central_comparison_df.to_json(orient="records"); h5.attrs["transfer_reconstruction_json"]=transfer_reconstruction_df.to_json(orient="records")
    h5.attrs["wormhole_timing_audit_json"]=wormhole_timing_audit_df.to_json(orient="records"); h5.attrs["gravastar_multistart_json"]=gravastar_multistart_df.to_json(orient="records")
bh_qnm_df.to_csv(ARTIFACT_DIR/"schwarzschild_qnms.csv",index=False); td_qnm_df.to_csv(ARTIFACT_DIR/"schwarzschild_td_modes.csv",index=False)
gravastar_fd_df.to_csv(ARTIFACT_DIR/"gravastar_fd_convergence.csv",index=False); echo_uncertainty_components.to_csv(ARTIFACT_DIR/"echo_time_uncertainty.csv",index=False)
wormhole_qnm_map.to_csv(ARTIFACT_DIR/"wormhole_qnm_map.csv",index=False); qh_lowfreq_df.to_csv(ARTIFACT_DIR/"qh_low_frequency_convergence.csv",index=False)
wormhole_pulse_convergence_df.to_csv(ARTIFACT_DIR/"wormhole_pulse_convergence.csv",index=False); qh_transfer_chain_test.to_csv(ARTIFACT_DIR/"qh_transfer_chain_test.csv",index=False)
quantum_scan_df.to_csv(ARTIFACT_DIR/"quantum_hierarchy_diagnostics.csv",index=False)
gravastar_multistart_df.to_csv(ARTIFACT_DIR/"gravastar_qnm_multistart.csv",index=False); gravastar_root_clusters_df.to_csv(ARTIFACT_DIR/"gravastar_qnm_root_clusters.csv",index=False)
eco_central_comparison_df.to_csv(ARTIFACT_DIR/"eco_central_reflection_comparison.csv",index=False); transfer_reconstruction_df.to_csv(ARTIFACT_DIR/"transfer_reconstruction_comparison.csv",index=False)
wormhole_timing_audit_df.to_csv(ARTIFACT_DIR/"wormhole_timing_audit.csv",index=False); wormhole_window_sensitivity_df.to_csv(ARTIFACT_DIR/"wormhole_window_sensitivity.csv",index=False)
residual_df.to_csv(ARTIFACT_DIR/"early_late_residual_metrics.csv"); instfreq_sensitivity_df.to_csv(ARTIFACT_DIR/"instantaneous_frequency_threshold_sensitivity.csv",index=False)
energy_df.to_csv(ARTIFACT_DIR/"energy_by_window.csv")
print("Wrote",h5_path.resolve(),"size",h5_path.stat().st_size/1e6,"MB")

# Verify outputs as delivered, not only the Python objects used to create them.
with h5py.File(h5_path,"r") as readback:
    b=readback[bh.model]; q=readback[qh.model]
    tt=b["time"][:]; bdata=b["waveform_real"][:]; qdata=q["waveform_real"][:]
    early=(tt>=100)&(tt<190); late=(tt>=190)&(tt<=500); delta=qdata-bdata
    peak=np.max(np.abs(bdata[early]))
    norms=np.array([np.sqrt(np.mean(delta[early]**2)),np.sqrt(np.mean(delta[late]**2))])
    measured=residual_df.loc["QH"]
    norm_ok=(np.allclose(norms,measured[["residual_rms_early","residual_rms_late"]].to_numpy(float),atol=1e-15)
             and np.isclose(norms[1]/peak,measured.late_RMS_over_BH_prompt_peak))
    qi=names.index("QH"); bi=names.index("BH")
    weighted=readback["late_detector_mismatch"][qi,bi]
    intrinsic=readback["late_intrinsic_mismatch"][qi,bi]
    _,qs=windowed_signal(qh,tt,190,500); _,rs=windowed_signal(bh,tt,190,500)
    mismatch_ok=(np.isclose(intrinsic,1-optimized_overlap(qs,rs)) and
                 np.isclose(weighted,1-rw_strain_detector_overlap(qs,rs,np.mean(np.diff(tt[late])),CFG.detector_total_mass_solar)))
    energy_ok=np.isclose(np.trapezoid(q["master_flux"][:][late],tt[late]),energy_df.loc["QH","late_flux_energy"])
    waveform_ok=np.allclose(delta,q["isolated_transfer_correction"][:],atol=2e-15,rtol=1e-10)
    direct=q["independent_TD_correction_on_BH_grid"][:]
    independent_l2=np.linalg.norm((delta-direct)[late])/np.linalg.norm(direct[late])

hierarchy_rows=quantum_scan_df.set_index("EplM_over_gamma")
hierarchy_ok=True
for hh,(ww,ee) in quantum_scan_waveforms.items():
    saved=hierarchy_rows.loc[hh]
    hierarchy_ok &= np.isclose(saved.A_late_total_over_BH_prompt,np.max(abs(ww.psi[late]))/peak)
    hierarchy_ok &= np.isclose(saved.A_late_QH_correction_over_BH_prompt,np.max(abs(ee[late]))/peak)
    hierarchy_ok &= np.isclose(saved.A_late_residual_over_BH_prompt,np.max(abs(ww.psi[late]-bdata[late]))/peak)
    kg=surface_K(wgrid,Rbh,Tdet,quantum_local_reflection(wgrid,xH,EplM=hh),xH,xE)
    hierarchy_ok &= np.isclose(saved["total_group_delay_at_0.37"],np.interp(.37,wgrid,np.gradient(np.unwrap(np.angle(kg)),wgrid,edge_order=2)))
nz=plane_tests[plane_tests.R>0]; grids_ok=True
for (model,value),g in scan_raw.groupby(["model","parameter"]):
    expected={(d,r) for d in scan_levels for r in radii} if value in representatives[model] else {(.20,95.)}
    grids_ok &= len(g)==len(expected) and set(zip(g.dx_requested,g.radius))==expected
fitrows=scan_df[(scan_df.model=="gravastar") & scan_df.TD_omega_R.notna()]
mustcols=["raw_total_shape_L2","raw_retarded_residual_shape_L2","spectral_peak_fraction","late_residual_peak_fraction"]
monotone=all(np.all(np.diff(g.sort_values("epsilon").n_valid)<=0) for _,g in if_evidence.groupby("model"))
gsbase=echo_report_df.set_index("model").loc["GS"]
gsflags_ok=(gsbase["Separated pulses identified?"]==gs_measured["separated_pulses"] and
            gsbase["Stable echo spacing?"]==gs_measured["stable_spacing"] and
            np.isclose(gsbase.T_recurrence,gs_rec,equal_nan=True))
audit_rows=[]
def verify(item,requirement,evidence,observed,passed):
    audit_rows.append(dict(item=item,requirement=requirement,evidence=evidence,observed=str(observed),passed=bool(passed)))
verify("1.1–1.2","Local surface basis; one-way detector path distinct from round trip","section 8; qh_coordinate_tests.csv",
       (L_surface_to_extraction,L_cavity_roundtrip),L_surface_to_extraction==xE-xH and L_cavity_roundtrip==-2*xH and np.allclose(nz.first_delay,95.-nz.xH))
verify("1.3.1","Zero reflector yields zero correction","qh_shifted_surface_pde.csv",zero_error,zero_error==0)
verify("1.3.2","First-return amplitude linear in reflectivity","qh_coordinate_tests.csv",tuple(nz.R.unique()),np.allclose(nz.first_height,.8*nz.R))
verify("1.3.3","Move wall; fixed source/detector; compare arrival and return spacing","qh_coordinate_tests.csv; qh_shifted_surface_pde.csv",
       alt_error,nz.xH.nunique()>=2 and np.allclose(nz.spacing,-2*nz.xH) and np.allclose(nz.first_delay_from_fixed_barrier_source,95.-2*nz.xH) and alt_error<.01)
verify("1.3.4","Independent local-boundary PDE vs transfer, no fitted shift or normalization","qh_independent_td_checks.csv; HDF5",
       independent_l2,independent_l2<.05 and np.isclose(independent_l2,qh_td_tests.query("law=='OWA' and dx==.05").raw_relative_L2.iloc[0]))
verify("1.3.5","Fourier convention preserves causal delay and reflection phase","qh_transfer_chain_test.csv",
       (causality_l2,chain_phase_error),causality_l2<1e-10 and chain_phase_error<.03)
verify("1.3.6","Recompute waveform, delay, hierarchy, residuals, flux energy and both mismatches",
       "HDF5 readback; quantum_hierarchy_diagnostics.csv; energy_by_window.csv",len(hierarchy_rows),
       waveform_ok and hierarchy_ok and norm_ok and energy_ok and mismatch_ok)
verify("2","Independent input guesses, residuals, complex clustering, damping order, independent TD and one-at-a-time numerical stability",
       "gravastar_qnm_multistart.csv; gravastar_qnm_root_clusters.csv; gravastar_root_stability.csv",
       (len(gravastar_multistart_df),len(clusters),gravastar_fundamental_support),
       gravastar_multistart_df.guess.nunique()==len(gravastar_multistart_df) and gravastar_root_clusters_df.n_starts.sum()==gravastar_multistart_df.converged.sum()
       and len(stability_trials)==4*len(clusters) and gs_supported and np.all(np.diff(verified.damping)>=0))
verify("3.1","Separate total, residual and isolated QH peaks; explicit common BH prompt and windows",
       "waveform_parameter_map.csv; waveform_parameter_maps.png",scan_norm,
       hierarchy_ok and scan_df.normalization.eq(scan_norm).all() and np.isfinite(scan_df[["A_late_total_over_BH_prompt","A_late_residual_over_BH_prompt"]]).all().all())
verify("3.2","Measure gravastar packets, cavity, light ring, QNM, boundary/grid checks; record failed or untested criteria",
       "waveform_parameter_map.csv; standard_echo_validity_report.csv",tuple(fitrows.parameter),
       gsflags_ok and decision_controls and report_agrees and set(fitrows.parameter)==representatives["gravastar"] and scan_df.failed_or_unavailable.notna().all())
verify("3.3","Low/mid/high at three grids and two observers; recurrence, peak, amplitude and both waveform shapes",
       "scan_measurements.csv; scan_resolution_and_extraction.csv",(len(scan_raw),len(scan_compare)),
       grids_ok and len(scan_compare)==42 and np.isfinite(scan_compare[mustcols]).all().all()
       and scan_df.loc[scan_df.study.str.contains("exploratory"),"T_echo"].isna().all())
verify("4.1","Absolute early/late RMS and late RMS / BH prompt peak, including tiny early residuals",
       "early_late_residual_metrics.csv; HDF5 readback",norms[1]/peak,norm_ok and "late_vs_early_residual_ratio" not in residual_df)
verify("4.2","Fitted complex QNMs; masked Hilbert values only; threshold sensitivity with labelled mask",
       "gravastar_qnm_root_clusters.csv; hilbert_mask_counts.csv; hilbert_threshold_comparison.png",sorted(if_evidence.epsilon.unique()),
       monotone and if_evidence.groupby("model").epsilon.nunique().eq(3).all() and gs_supported)
coverage=pd.DataFrame(audit_rows); coverage.to_csv(ARTIFACT_DIR/"corrections_coverage.csv",index=False); display(coverage)
if not coverage.passed.all(): raise AssertionError(coverage.loc[~coverage.passed].to_string(index=False))
print(f"{len(coverage)}/{len(coverage)} correction groups verified; clean echo periods:",
      int(scan_df.T_echo.notna().sum()),"of",len(scan_df),"scan points.")



# Rebuild every pair from the exported bank, including matrices with no BH leg.
comparison_readback_rows=[]
with h5py.File(h5_path,"r") as saved:
    bank=saved["controlled_comparison"]
    for label,wf in comparison_waveforms.items():
        data=bank[label]
        exact=bool(np.array_equal(data["time"][:],wf.t) and np.array_equal(data["waveform_real"][:],wf.psi))
        comparison_readback_rows.append(dict(check="exact_waveform_roundtrip",model=label,passed=exact,error=0. if exact else np.inf))
    for window,lo,hi in (("prompt",100.,190.),("late",190.,500.)):
        ts=bh.t[(bh.t>=lo)&(bh.t<=hi)]; dt=float(np.mean(np.diff(ts)))
        signals={}
        for label in names:
            g=bank[label]
            waveform=Waveform(label,g["time"][:],g["waveform_real"][:],CFG.extraction_x,{})
            signals[label]=windowed_signal(waveform,bh.t,lo,hi)[1]
        for weighted in (False,True):
            kind="detector" if weighted else "intrinsic"
            matrix=saved[f"{window}_{kind}_mismatch"][:]
            for i,a in enumerate(names):
                for j,b in enumerate(names):
                    overlap=rw_strain_detector_overlap(signals[a],signals[b],dt,CFG.detector_total_mass_solar) if weighted else optimized_overlap(signals[a],signals[b])
                    error=float(abs(matrix[i,j]-(1.-overlap)))
                    comparison_readback_rows.append(dict(check=f"{window}_{kind}_matrix_pair",model=f"{a}:{b}",passed=error<2e-12,error=error))
            symmetry=float(np.max(abs(matrix-matrix.T))); diagonal=float(np.max(abs(np.diag(matrix))))
            comparison_readback_rows.append(dict(check=f"{window}_{kind}_symmetry_diagonal",model="all",passed=symmetry<2e-12 and diagonal<2e-12,error=max(symmetry,diagonal)))
comparison_readback_df=pd.DataFrame(comparison_readback_rows)
comparison_readback_df.to_csv(ARTIFACT_DIR/"controlled_comparison_readback.csv",index=False)
if not comparison_readback_df.passed.all(): raise AssertionError(comparison_readback_df.loc[~comparison_readback_df.passed])
print(f"Controlled-bank export: {len(comparison_readback_df)} exact-record and all-pair checks passed")


"""Finite-time approach to the BH response, NOT convergence of the QNM spectrum.

Embed at notebook end. Uses its actual GS/WH solvers and corrected global OWA
reflectivity. The source and observer stay fixed in the common exterior.

Primary sources (the numerical thresholds below are ours, not paper values):
Pani et al., https://arxiv.org/abs/0909.0287 : gravastar spectra remain different
from black-hole spectra even at divergent surface redshift.
Cardoso, Franzin & Pani, https://arxiv.org/abs/1602.07309 : BH-like early ringdown
does not imply BH QNMs or an event horizon.
Yang, Xia & Miao, https://doi.org/10.1140/epjc/s10052-025-14446-w : the selected
Schwarzschild cut-and-paste axial model retains the throat delta junction.
Oshita, Wang & Afshordi, https://arxiv.org/html/1905.00464 : Eqs.14-16 and
Appendix A. The hierarchy changes reflection depth/phase, not the thermal
reflection magnitude at fixed dimensionless frequency.
"""


def run_finite_time_bh_limit_audit():
    if CFG.M!=1.: raise ValueError("This ADM-scaled limit audit requires M=1")
    # No amplitude or time fit is used. This is a fixed observation experiment,
    # not a comparison on a moving, first-echo-centred time interval.
    end=230.; source=55.; observer=95.; width=4.
    window=(100.,175.); after=(175.,225.)
    test_cfg=replace(CFG,pulse_center=source,extraction_x=observer,pulse_width=width,t_max=end)
    records={}; rows=[]; references={}
    for dx in (.20,.10):
        reference,*_=run_black_hole(cfg=test_cfg,dx=dx,t_max=end,x_left=-220.,
                                   extraction_xs=(observer,))
        references[dx]=reference
    for model,parameters in (("GS",(.49,.499,.4999)),("WH",(.01,.001,.0001))):
        for parameter in parameters:
            for dx in ((.20,.10) if parameter==parameters[-1] else (.20,)):
                if model=="GS":
                    wave,*_=run_gravastar(parameter,cfg=test_cfg,dx=dx,t_max=end,extraction_xs=(observer,))
                    interface=float(wave.metadata["shell_x"])
                    center=float(wave.metadata["center_x"])
                    extra=dict(compactness=parameter,epsilon=(1/parameter-2)/2,
                               nominal_center_return_M=source+observer-2*center)
                else:
                    r0=2*(1+parameter); interface=float(schwarzschild_rstar(r0))
                    local_cfg=replace(test_cfg,pulse_center=source-interface,extraction_x=observer-interface)
                    wave,*_=run_wormhole(r0,cfg=local_cfg,dx=dx,t_max=end,
                                        extraction_xs=(local_cfg.extraction_x,),x_left=-300.)
                    extra=dict(compactness=1/r0,epsilon=parameter,nominal_center_return_M=np.nan)
                reference=references[dx]
                error=relative_l2(reference.t,reference.psi,wave.t,wave.psi,*window)
                late=relative_l2(reference.t,reference.psi,wave.t,wave.psi,*after)
                if not np.isfinite([error,late]).all(): raise AssertionError("Nonfinite limit error")
                if not wave.metadata["stability_pass"]: raise AssertionError("Unstable limit evolution")
                rows.append(dict(model=model,parameter=parameter,dx_requested=dx,dx_actual=wave.metadata["dx"],
                    source_global_rstar=source,observer_global_rstar=observer,pulse_width_M=width,
                    analysis_start=window[0],analysis_end=window[1],raw_relative_L2=error,
                    after_window_start=after[0],after_window_end=after[1],after_relative_L2=late,
                    interface_global_rstar=interface,nominal_first_interface_return_M=source+observer-2*interface,
                    earliest_8sigma_interface_return_M=source-8*width+observer-2*interface,
                    **extra))
                records[(model,float(parameter),dx)]=wave
    result=pd.DataFrame(rows); checks=[]
    for model in ("GS","WH"):
        block=result[(result.model==model)&(result.dx_requested==.20)].sort_values("epsilon",ascending=False)
        errors=block.raw_relative_L2.to_numpy(); returns=block.nominal_first_interface_return_M.to_numpy()
        compact=float(block.iloc[-1].parameter)
        coarse=records[(model,compact,.20)]; fine=records[(model,compact,.10)]
        grid_error=relative_l2(fine.t,fine.psi,coarse.t,coarse.psi,*window)
        near_fine=float(result[(result.model==model)&(result.dx_requested==.10)].raw_relative_L2.iloc[0])
        # Resolution is measured separately. Monotonic approach may saturate at
        # that floor, but a resolved order-of-magnitude approach is mandatory.
        tests={"return_scale_moves_later":bool(np.all(np.diff(returns)>0)),
               "fixed_window_error_decreases":bool(np.all(np.diff(errors)<=2*grid_error+1e-7)),
               "resolved_BH_approach":bool(near_fine<.1*errors[0] and grid_error<.1*errors[0]),
               "near_limit_prompt_agreement":bool(near_fine<.01),
               "near_limit_grid_control":bool(grid_error<.005)}
        for name,passed in tests.items():
            checks.append(dict(model=model,check=name,passed=passed,far_error=errors[0],
                               near_fine_error=near_fine,coarse_fine_error=grid_error))
    result.to_csv(ARTIFACT_DIR/"finite_time_BH_approach.csv",index=False)
    checks=pd.DataFrame(checks); checks.to_csv(ARTIFACT_DIR/"finite_time_BH_approach_checks.csv",index=False)
    # Export the actual raw records, not only reduced errors.
    with h5py.File(ARTIFACT_DIR/"finite_time_BH_approach.h5","w") as saved:
        saved.attrs["interpretation"]="fixed-time waveform approach only; not full-spectrum or all-time convergence"
        saved.attrs["gaussian_tail"]="Gaussian has no compact support; 8sigma markers neglect amplitude <=exp(-32)"
        for dx,wave in references.items():
            group=saved.create_group(f"BH_dx{dx:g}")
            group.create_dataset("time",data=wave.t);group.create_dataset("waveform",data=wave.psi)
        for (model,parameter,dx),wave in records.items():
            group=saved.create_group(f"{model}_p{parameter:g}_dx{dx:g}")
            group.create_dataset("time",data=wave.t);group.create_dataset("waveform",data=wave.psi)
            group.attrs["metadata_json"]=json.dumps(wave.metadata,default=float)
    if not checks.passed.all(): raise AssertionError(checks.loc[~checks.passed].to_string(index=False))
    return result,checks


def run_qh_hierarchy_limit_identity():
    # The exact OWA Gamma ratios are independent of E_Pl/gamma. At fixed w,
    # increasing H rotates R_g by exp[2 i w log(H/H0)/kappa]; |R_g| is unchanged.
    # Thus the finite-time BH-like regime lengthens through delay, not absorption.
    omega=np.linspace(.08,.8,1201); kappa=.25; base=1e8
    r0=quantum_reflection(omega,EplM=base); records=[]
    for hierarchy in (1e8,1e16,1e40):
        response=quantum_reflection(omega,EplM=hierarchy)
        expected=2/kappa*np.log(hierarchy/base)
        phase=np.unwrap(np.angle(response*np.conj(r0)))
        measured=float(np.polyfit(omega,phase,1)[0])
        amplitude_error=float(np.max(abs(abs(response)-abs(r0))))
        phase_error=float(np.max(abs(response/r0-np.exp(1j*omega*expected))))
        records.append(dict(hierarchy=hierarchy,reference_hierarchy=base,
            measured_added_group_delay_M=measured,expected_added_group_delay_M=expected,
            effective_depth_change_M=-expected/2,maximum_absolute_amplitude_change=amplitude_error,
            phase_ratio_identity_error=phase_error,
            passed=amplitude_error<1e-12 and phase_error<1e-10 and abs(measured-expected)<1e-9))
    result=pd.DataFrame(records);result.to_csv(ARTIFACT_DIR/"qh_hierarchy_depth_not_absorption.csv",index=False)
    if not result.passed.all(): raise AssertionError(result)
    return result


finite_time_limit_df,finite_time_limit_checks=run_finite_time_bh_limit_audit()
qh_hierarchy_limit_df=run_qh_hierarchy_limit_identity()
display(finite_time_limit_df)
display(finite_time_limit_checks)
display(qh_hierarchy_limit_df)
print("The GS/WH checks establish a resolved finite-time approach for the stated experiment, not spectral convergence.")
print("The selected regular mini-boson-star family has no BH limit; no artificial BH-limit pass is assigned to it.")


"""Additional controlled waveform banks; append after the notebook export cell.

This is an actual finite experiment, not a claim of source-independent model
identification. All signals are freshly evolved; no cached waveform is used.
The QH input is measured as field AND spatial derivative in each source run.
"""

RUN_CONTROLLED_BANK_EXERCISES = True  # optional cost; enabled in the release
BANK_EXERCISE_NAMES = ("BH", "GS", "BS", "WH", "QH")
BANK_EXERCISE_WINDOWS = (("prompt", 100., 190.), ("late", 190., 500.))


def _bank_fresh_evolution(x, potential, cfg, *, left_bc="outgoing", interface=None,
                          monitor=False, duration=None):
    points = {"observer": cfg.extraction_x}
    if monitor:
        points["monitor"] = cfg.qh_surface_x
    tt, record, metadata = evolve_wave(
        x, potential, t_max=cfg.t_max if duration is None else duration,
        cfl=cfg.cfl, pulse_center=cfg.pulse_center, pulse_width=cfg.pulse_width,
        extraction_points=points, left_bc=left_bc, sample_every=cfg.sample_every,
        ell=cfg.ell, interface_jump=interface,
        derivative_points=("monitor",) if monitor else ())
    return Waveform("controlled_replay", tt, record["observer"], cfg.extraction_x,
                    metadata), record


def _bank_qh_from_record(record, time, hierarchy, cfg, common_t, nf):
    """Transform the freshly measured characteristic; no guessed incident wave."""
    step = float(np.mean(np.diff(common_t)))
    sampling = step*np.arange(int(np.floor(650./step))+1)
    if sampling[-1] > time[-1]:
        raise ValueError("QH source record ends before requested sampling")
    total = CubicSpline(time, record["monitor"], extrapolate=False)(sampling)
    deriv = CubicSpline(time, record["monitor_dx"], extrapolate=False)(sampling)
    omega = 2*np.pi*np.fft.rfftfreq(nf, step)
    field_fft = np.fft.rfft(total, nf)
    derivative_fft = np.fft.rfft(deriv, nf)
    incoming = np.zeros_like(field_fft)
    incoming[1:] = .5*(field_fft[1:]+derivative_fft[1:]/(1j*omega[1:]))
    rb = interp_scattering(omega, wgrid, Rbh, "R", cfg.ell)
    te = interp_scattering(omega, wgrid, Tdet, "T_E", cfg.ell)
    local = quantum_local_reflection(omega, cfg.qh_surface_x,
        kappa=1/(4*cfg.M), gamma=1., EplM=hierarchy)
    transfer = surface_K(omega, rb, te, local, cfg.qh_surface_x, cfg.extraction_x)
    transfer[omega > wgrid[-1]] = 0.
    correction = np.fft.irfft(np.conj(transfer)*incoming, nf)[:len(common_t)]
    data = dict(source_t=sampling, monitor_field=total, monitor_derivative=deriv,
        omega=omega, incoming_fft=incoming, transfer_K=transfer,
        isolated_correction=correction)
    return correction, data


def regenerate_controlled_bank(case, *, background):
    """Hold ADM mass, source/observer radii, normalization, and time origin fixed."""
    cfg = replace(CFG, pulse_width=float(case["width_M"]))
    if cfg.M != 1.:
        raise ValueError("The stellar coordinates require the common ADM M=1 scale")
    common_t = np.asarray(bh.t)
    target_radii = r_from_schwarzschild_rstar(
        [cfg.pulse_center, cfg.extraction_x], cfg.M)
    # A new long-domain source experiment supplies BOTH the BH waveform and
    # characteristic QH input. Do not change CFG or consult waveform caches.
    xx, vv, _, _ = black_hole_grid(cfg, x_left=-400., x_right=400.)
    bh_full, source_record = _bank_fresh_evolution(
        xx, vv, cfg, monitor=True, duration=650.3)
    base_signal = CubicSpline(bh_full.t, bh_full.psi, extrapolate=False)(common_t)
    base = Waveform("black_hole", common_t.copy(), base_signal,
                    cfg.extraction_x, dict(bh_full.metadata))
    nf = 1 << (int(4*len(common_t)-1).bit_length())
    correction, qdata = _bank_qh_from_record(source_record, bh_full.t,
        float(case["EplM"]), cfg, common_t, nf)
    doubled, _ = _bank_qh_from_record(source_record, bh_full.t,
        float(case["EplM"]), cfg, common_t, 2*nf)
    win = (common_t >= 190.) & (common_t <= 500.)
    padding_error = float(np.linalg.norm((correction-doubled)[win]) /
                          max(np.linalg.norm(doubled[win]), 1e-300))
    if padding_error >= .01:
        raise AssertionError((case["case"], "QH padding error", padding_error))
    qh_wave = Waveform("owa_boltzmann_quantum_horizon", common_t.copy(),
        base_signal+correction, cfg.extraction_x,
        {**bh_full.metadata, "EplM":float(case["EplM"]),
         "QH_drive":"fresh measured ingoing field/derivative characteristic",
         "QH_reflection":"published global law rephased to the computational plane",
         "QH_padding_relative_error":padding_error})

    gx, gv, _, gm = gravastar_grid(float(case["GS_M_over_a"]), cfg)
    gstar, _ = _bank_fresh_evolution(gx, gv, cfg, left_bc="regular_center")
    gstar.metadata.update(gm)
    wh_offset = float(schwarzschild_rstar(float(case["WH_r0_over_M"]), cfg.M))
    wcfg = replace(cfg, pulse_center=cfg.pulse_center-wh_offset,
                   extraction_x=cfg.extraction_x-wh_offset)
    wx, wv, _, wm = wormhole_grid(float(case["WH_r0_over_M"]), wcfg,
                                 x_left=-400., x_right=400.)
    worm, _ = _bank_fresh_evolution(wx, wv, wcfg,
        interface={"x":0., "Lambda":wm["throat_Lambda"]})
    worm.metadata.update(wm)
    bs_positions = boson_native_tortoise_at_radius(background, target_radii)
    bcfg = replace(cfg, pulse_center=float(bs_positions[0]),
                   extraction_x=float(bs_positions[1]))
    bx, bv, _ = boson_axial_grid(background, bcfg)
    star, _ = _bank_fresh_evolution(bx, bv, bcfg, left_bc="regular_center")
    star.metadata.update(sigma_c=background.sigma_c,
        raw_ADM_mass_mu_units=background.mass, ADM_mass_scaled=1.,
        EKG_residual=background.residual)
    bank = dict(BH=base, GS=gstar, BS=star, WH=worm, QH=qh_wave)
    configs = dict(BH=cfg, GS=cfg, BS=bcfg, WH=wcfg, QH=cfg)
    geometry = []
    for name in BANK_EXERCISE_NAMES:
        wf = bank[name]; own = configs[name]
        if name == "BS":
            rM = background.r/background.mass
            xM = cumulative_trapezoid(background.a/background.alpha,
                                      background.r, initial=0.)/background.mass
            actual = []
            for position in (own.pulse_center, own.extraction_x):
                if position > xM[-1]:
                    actual.append(float(r_from_schwarzschild_rstar(
                        schwarzschild_rstar(rM[-1])+position-xM[-1])))
                else:
                    idx = min(max(1, int(np.searchsorted(xM, position))), len(xM)-1)
                    actual.append(brentq(lambda rr:float(
                        boson_native_tortoise_at_radius(background, rr))-position,
                        rM[idx-1], rM[idx], xtol=1e-12))
            actual = np.asarray(actual)
        else:
            actual = r_from_schwarzschild_rstar(
                np.array([own.pulse_center, own.extraction_x])+
                (wh_offset if name == "WH" else 0.), cfg.M)
        initial_in = float(wf.metadata["F_in"])
        right = float(wf.metadata["x_right"])
        # Treat initial Gaussian tails beyond eight sigma as exponentially
        # negligible; physical centers are not artificial boundaries.
        boundary_margin = (2*right-own.pulse_center-8*own.pulse_width-
                           own.extraction_x-500.)
        if name in ("BH", "WH", "QH"):
            boundary_margin = min(boundary_margin, own.pulse_center-
                8*own.pulse_width+own.extraction_x-2*float(wf.metadata["x_left"])-500.)
        row = dict(case=case["case"], model=name, ADM_mass=1.,
            source_native=own.pulse_center, observer_native=own.extraction_x,
            source_radius=float(actual[0]), observer_radius=float(actual[1]),
            radius_error=float(max(abs(actual-target_radii))), width_M=own.pulse_width,
            F_in=initial_in, boundary_margin_M=boundary_margin,
            passed=bool(max(abs(actual-target_radii))<1e-9 and
                        abs(initial_in-1.)<1e-12 and boundary_margin>0. and
                        wf.metadata["stability_pass"] and np.isfinite(wf.psi).all()))
        wf.metadata.update(comparison_case=case["case"], comparison_protocol=row)
        geometry.append(row)
    return bank, qdata, pd.DataFrame(geometry)


def controlled_bank_matrices(bank, common_t):
    matrices = {}
    for window, lower, upper in BANK_EXERCISE_WINDOWS:
        signals = {label:windowed_signal(bank[label], common_t, lower, upper)[1]
                   for label in BANK_EXERCISE_NAMES}
        selected = common_t[(common_t >= lower) & (common_t <= upper)]
        step = float(np.mean(np.diff(selected)))
        for detector in (False, True):
            kind = "strain_detector" if detector else "intrinsic_Q"
            result = np.empty((5, 5))
            for i, first in enumerate(BANK_EXERCISE_NAMES):
                for j, second in enumerate(BANK_EXERCISE_NAMES):
                    a, b = signals[first], signals[second]
                    overlap = (rw_strain_detector_overlap(a,b,step,
                        CFG.detector_total_mass_solar) if detector else optimized_overlap(a,b))
                    result[i,j] = 1.-overlap
            if not (np.isfinite(result).all() and np.max(abs(result-result.T))<2e-12
                    and np.max(abs(np.diag(result)))<2e-12
                    and result.min()>-2e-12 and result.max()<1.+2e-12):
                raise AssertionError((window, kind, result))
            matrices[window+"_"+kind] = result
    return matrices


def run_controlled_bank_exercises(artifact_dir):
    cases = [dict(case="baseline_replay", width_M=CFG.pulse_width,
        GS_M_over_a=CFG.gravastar_compactness, WH_r0_over_M=CFG.wormhole_r0,
        BS_sigma_c=CFG.boson_sigma_c, EplM=1e8)]
    cases += [{**cases[0], "case":label, "width_M":width} for label,width in
              (("source_width_3",3.), ("source_width_5",5.))]
    cases.append(dict(case="second_parameter_configuration", width_M=CFG.pulse_width,
        GS_M_over_a=.485, WH_r0_over_M=2.005, BS_sigma_c=.22, EplM=1e10))
    path = artifact_dir/"controlled_bank_exercises.h5"
    rows=[]; geometries=[]; replay=[]; readback=[]; baseline_matrices=None
    original_matrices=controlled_bank_matrices(comparison_waveforms,bh.t)
    with h5py.File(path,"w") as saved:
        saved.attrs["scope"]="four finite controlled experiments; not a population or universal robustness claim"
        saved.attrs["models_json"]=json.dumps(BANK_EXERCISE_NAMES)
        saved.attrs["settings_json"]=json.dumps(cases)
        saved.attrs["intrinsic_metric"]="time/phase maximized, Tukey-windowed axial Q master-function overlap"
        saved.attrs["detector_metric"]="analytic aLIGO PSD; axial Q to strain by 1/(2*pi*f); positive band only"
        saved.attrs["initial_conditions"]="same unit left-going Gaussian, common ADM mass and source/observer areal radii; no fitted time shift before metrics"
        for case in cases:
            print("Additional controlled bank:",case["case"],flush=True)
            bg = bs_bg if case["BS_sigma_c"]==CFG.boson_sigma_c else solve_mini_boson_star(
                sigma_c=case["BS_sigma_c"])
            if bg.residual>=3e-7:
                raise AssertionError((case["case"],"EKG residual",bg.residual))
            bank,qdata,geometry=regenerate_controlled_bank(case,background=bg)
            if not geometry.passed.all(): raise AssertionError(geometry)
            matrices=controlled_bank_matrices(bank,bh.t)
            geometries.append(geometry)
            if case["case"]=="baseline_replay":
                baseline_matrices=matrices
                for label in BANK_EXERCISE_NAMES:
                    selection=(bh.t>=100.)&(bh.t<=500.)
                    old=CubicSpline(comparison_waveforms[label].t,
                        comparison_waveforms[label].psi,extrapolate=False)(bh.t[selection])
                    new=CubicSpline(bank[label].t,bank[label].psi,extrapolate=False)(bh.t[selection])
                    error=float(np.linalg.norm(new-old)/max(np.linalg.norm(old),1e-300))
                    replay.append(dict(check="fresh_baseline_waveform",label=label,error=error,
                        tolerance=1e-6,passed=error<1e-6))
                for key,matrix in matrices.items():
                    error=float(np.max(abs(matrix-original_matrices[key])))
                    replay.append(dict(check="fresh_baseline_matrix",label=key,error=error,
                        tolerance=5e-6,passed=error<5e-6))
            root=saved.create_group(case["case"])
            root.attrs["settings_json"]=json.dumps(case)
            root.attrs["geometry_json"]=geometry.to_json(orient="records")
            for label,wave in bank.items():
                item=root.create_group(label)
                item.create_dataset("time",data=wave.t,compression="gzip")
                item.create_dataset("Q_RW",data=wave.psi,compression="gzip")
                item.attrs["metadata_json"]=json.dumps(wave.metadata,default=float)
            source_group=root.create_group("QH_measured_input")
            for key,value in qdata.items():
                source_group.create_dataset(key,data=value,compression="gzip")
            for key,matrix in matrices.items():
                root.create_dataset(key,data=matrix)
                for i,first in enumerate(BANK_EXERCISE_NAMES):
                    for j,second in enumerate(BANK_EXERCISE_NAMES):
                        rows.append(dict(case=case["case"],matrix=key,first=first,
                            second=second,mismatch=matrix[i,j],
                            change_from_baseline_replay=matrix[i,j]-baseline_matrices[key][i,j],
                            **{k:v for k,v in case.items() if k!="case"}))
    # This readback recomputes every matrix pair from the exact serialized
    # records, not from the in-memory matrix or a BH-only subset of pairs.
    with h5py.File(path,"r") as saved:
        for case in cases:
            root=saved[case["case"]]
            bank={label:Waveform(label,root[label]["time"][:],
                root[label]["Q_RW"][:],CFG.extraction_x,{}) for label in BANK_EXERCISE_NAMES}
            recalculated=controlled_bank_matrices(bank,bh.t)
            for key,matrix in recalculated.items():
                difference=abs(matrix-root[key][:])
                for i,first in enumerate(BANK_EXERCISE_NAMES):
                    for j,second in enumerate(BANK_EXERCISE_NAMES):
                        readback.append(dict(case=case["case"],matrix=key,
                            pair=first+":"+second,error=difference[i,j],passed=difference[i,j]<2e-12))
            data=root["QH_measured_input"]
            nf=2*(len(data["omega"])-1)
            omega=data["omega"][:]
            source_fft=np.fft.rfft(data["monitor_field"][:],nf)
            derivative_fft=np.fft.rfft(data["monitor_derivative"][:],nf)
            fresh_incoming=np.zeros_like(source_fft)
            fresh_incoming[1:]=.5*(source_fft[1:]+derivative_fft[1:]/(1j*omega[1:]))
            split_error=float(np.max(abs(fresh_incoming-data["incoming_fft"][:])))
            readback.append(dict(case=case["case"],matrix="QH_measured_characteristic_split",
                pair="QH",error=split_error,passed=split_error<1e-12))
            reconstructed=np.fft.irfft(np.conj(data["transfer_K"][:])*
                fresh_incoming,nf)[:len(data["isolated_correction"])]
            error=float(np.max(abs(reconstructed-data["isolated_correction"][:])))
            readback.append(dict(case=case["case"],matrix="QH_characteristic_replay",
                pair="QH",error=error,passed=error<1e-14))
    frames=dict(matrices=pd.DataFrame(rows),geometry=pd.concat(geometries,ignore_index=True),
                baseline_replay=pd.DataFrame(replay),readback=pd.DataFrame(readback))
    for name,table in frames.items():
        table.to_csv(artifact_dir/("controlled_bank_"+name+".csv"),index=False)
        if "passed" in table and not table.passed.all(): raise AssertionError(table.loc[~table.passed])
    # Describe measured variability. Do not assert a rank or signature is
    # robust merely because every numerical/serialization check passed.
    unique=frames["matrices"].copy()
    unique=unique[unique["first"]<unique["second"]]
    summary=unique.groupby(["matrix","first","second"]).mismatch.agg(
        minimum="min",maximum="max").reset_index()
    summary["range"]=summary.maximum-summary.minimum
    summary.to_csv(artifact_dir/"controlled_bank_observed_variability.csv",index=False)
    frames["observed_variability"]=summary
    display(frames["baseline_replay"])
    display(frames["geometry"])
    display(summary)
    print(f"Saved {len(cases)} controlled 5-model banks, 16 full matrices; "
          f"{len(frames['readback'])} all-pair/input readback checks passed. "
          "Measured ranges describe these cases only.")
    return frames


if RUN_CONTROLLED_BANK_EXERCISES:
    controlled_bank_exercise_results=run_controlled_bank_exercises(ARTIFACT_DIR)


## 15. Interpretation and remaining physics

The notebook now distinguishes numerical artifacts, operational echo estimators, and model physics:

1. CFL, grid size, amplitude growth, and characteristic initial content are recorded per run.
2. Stability requires both a bounded amplitude ratio and acceptable middle/late segmented growth rates.
3. Boundary contamination is measured over the stated boundary-test interval, while extraction convergence is split into prompt and late regions.
4. Three Schwarzschild Leaver modes are computed; literature-independent time-domain clusters are formed before any mode identification.
5. The gravastar uses a fourth-order regular-center parity stencil and a shell-aligned interface; its QNM is independently reproduced by regular/outgoing frequency-domain matching and a ten-guess, order-14 clustered search with independent TD support (not a completeness proof).
6. The mini-BS I equilibrium is literature-normalized, its axial response is correctly scoped, and its photon-sphere condition is evaluated quantitatively.
7. The Schwarzschild thin-shell wormhole contains the positive axial throat delta coefficient as an exact jump. Its finite-band delay and parity-resolved QNM map use the complete barrier--throat--barrier transfer chain and are checked at $\Delta x=(0.20,0.10,0.05)M$ and three extraction radii.
8. A common physical reflection interface reports $|R_{\rm ECO}|$, phase, group delay, $K_{\rm ECO}$, direct/transfer waveform agreement, $A_{\rm echo}/A_{\rm prompt}$, and late mismatch; the nonfactorable boson-star case remains explicitly undefined.
9. The wormhole timing discrepancy is retained and explained as a competition among barrier/throat phases, bandwidth, overlapping parity families, estimator choice, and observation window rather than being tuned into a single period.
10. Quantum-boundary phase, low-frequency asymptotics, analytic causality/phase convention, a controlled full transfer-chain reconstruction, phase components, isolated correction, and hierarchy dependence are retained.
11. Intrinsic morphology, analytic-PSD detector weighting, BH-subtracted residuals, and master-function flux answer separate questions, with detector sensitivity deliberately downstream of the physical-response comparison.
12. Spatial and temporal fourth-order convergence are tested independently through $\Delta x=0.05M$.
13. Each base model receives a multi-diagnostic `CLEAN_ECHO`, `TRAPPED_RESONANCE`, or `NO_CLEAN_ECHO` classification; `T_echo` is absent unless the clean-echo criteria pass.

Open extensions remain: polar gravastar shell dynamics, coupled polar scalar–metric boson-star perturbations, rotating backgrounds, physical source excitation, calibrated strain at a distance, real detector PSD/data, and nonlinear merger initial data. These are not silently approximated by the present axial/free-response laboratory.

Scan rows report per-point diagnostics and may be INCONCLUSIVE: numerical convergence is not evidence for a physical echo. A first-return delay is not a repeated-return period. A short QH observation can support the former without resolving the latter.
INCONCLUSIVE is also an explicit outcome when the available evidence does not establish a resonance or a clean period.
The GS and WH finite-detector audits are specific to the stated linear axial problems, pulse, observer and time window. Passing a waveform test does not verify a universal echo period.

Release audit: wormhole poles are actual complex boundary-value zeros, with
independent published frequencies; gravastar Cauchy data are independently
checked against its analytic hypergeometric interior. The quantum coefficient
is global before rephasing, and changing its computational interface leaves
the physical signal invariant. Cross-model matrices use one controlled bank.
No result from a DS scalar wormhole or a massive-shell gravastar is represented
as a benchmark for the different axial models solved here.

Selected complex BS and QH modes are now computed as well. BS tail truncation is explicitly separated from numerical error; QH poles use a derived analytic branch of the specified effective boundary law. No absence-of-QNMs claim follows from the absence of a photon sphere.


The final cell also evolves fixed-time near-horizon GS/WH limits and repeats
the complete five-model controlled experiment for widths 3M and 5M and a
second parameter configuration. Each case has its own measured QH input.
All 16 extra matrices are replayed from the saved waveforms. Their observed
variability is reported, not assumed to vanish. The near-horizon tests concern
finite-time waveforms, not convergence of the ECO quasinormal spectra to BH
spectra. Increasing the QH Planck hierarchy delays reflection without reducing
its magnitude at fixed dimensionless frequency. The chosen mini-boson-star
family has no black-hole limit.
